# CNN+LSTM DDoS Detector — Live Testing & Evaluation

This notebook tests the trained CNN+LSTM model on **live simulated data** from the SDN monitoring API.

**Sections:**
1. Imports & Config
2. Load Model & Artifacts
3. Monitoring API *(paste your code here)*
4. Live Preprocessing Pipeline
5. Real-Time Prediction Loop
6. Evaluation & Metrics
7. Time-Series Visualization

## 1. Imports & Configuration

In [5]:
# ═══════════════════════════════════════════════════════════════════
# Section 1 — Imports & Configuration
# ═══════════════════════════════════════════════════════════════════
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('TkAgg')  # or 'Agg' for headless
import matplotlib.pyplot as plt
import seaborn as sns
import json, os, sys, time, pickle, re, warnings
from pathlib import Path
from collections import deque
from datetime import datetime, timezone

import tensorflow as tf
from tensorflow import keras
from sklearn.metrics import (
    classification_report, confusion_matrix,
    roc_auc_score, average_precision_score,
    f1_score, precision_score, recall_score,
    roc_curve, precision_recall_curve
)
from IPython.display import clear_output

warnings.filterwarnings('ignore')
%matplotlib notebook
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette('husl')

# ── Paths ──────────────────────────────────────────────────────────
ARTIFACT_DIR = 'outputs_detector'
MODEL_PATH   = os.path.join(ARTIFACT_DIR, 'model_final.keras')
SCALER_PATH  = os.path.join(ARTIFACT_DIR, 'scaler.pkl')
IMPUTER_PATH = os.path.join(ARTIFACT_DIR, 'imputer.pkl')
CONFIG_PATH  = os.path.join(ARTIFACT_DIR, 'config.json')
CALIB_PATH   = os.path.join(ARTIFACT_DIR, 'calibration.json')

# ── Live Testing Config ────────────────────────────────────────────
PREDICTION_INTERVAL = 2        # seconds between predictions
LIVE_HISTORY_MAX    = 5000     # max samples to keep in memory
PLOT_WINDOW         = 200      # samples to show in real-time plot

print('✅ Imports loaded')

2026-03-04 15:01:09.575518: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


✅ Imports loaded


## 2. Load Model, Scaler, Imputer & Config

In [2]:
# ═══════════════════════════════════════════════════════════════════
# Section 2 — Load Trained Artifacts
# ═══════════════════════════════════════════════════════════════════

# Config
with open(CONFIG_PATH) as f:
    config = json.load(f)

FEATURE_COLS    = config['feature_cols']       # ordered list of 135 features
WINDOW_LEN      = config['BEST_WINDOW_LEN']    # 5
BEST_THRESHOLD  = config['BEST_THRESHOLD']      # 0.847
N_FEATURES      = config['n_features']           # 135

# Calibration
with open(CALIB_PATH) as f:
    calib = json.load(f)
TEMPERATURE = calib['temperature']               # 0.957

# Model
model = keras.models.load_model(MODEL_PATH)
print(f'✅ Model loaded: {MODEL_PATH}')
model.summary()

# Scaler & Imputer
with open(SCALER_PATH, 'rb') as f:
    scaler = pickle.load(f)
with open(IMPUTER_PATH, 'rb') as f:
    imputer = pickle.load(f)

print(f'\n✅ Config loaded:')
print(f'  Features:  {N_FEATURES}')
print(f'  Window:    {WINDOW_LEN}')
print(f'  Threshold: {BEST_THRESHOLD:.4f}')
print(f'  Temperature: {TEMPERATURE:.4f}')
print(f'  Feature list (first 10): {FEATURE_COLS[:10]}')

I0000 00:00:1772525762.608586    4821 gpu_device.cc:2020] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 2216 MB memory:  -> device: 0, name: NVIDIA GeForce GTX 1650, pci bus id: 0000:01:00.0, compute capability: 7.5


✅ Model loaded: outputs_detector/model_final.keras


Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer (InputLayer)        │ (None, 5, 135)         │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv1d (Conv1D)                 │ (None, 5, 48)          │        19,488 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization             │ (None, 5, 48)          │           192 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ re_lu (ReLU)                    │ (None, 5, 48)          │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ spatial_dropout1d               │ (None, 5, 48)          │             0 │
│ (SpatialDropout1D)              │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv1d_1 (Conv1D)               │ (None, 5, 48)          │         6,960 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_1           │ (None, 5, 48)          │           192 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ re_lu_1 (ReLU)                  │ (None, 5, 48)          │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ spatial_dropout1d_1             │ (None, 5, 48)          │             0 │
│ (SpatialDropout1D)              │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm (LSTM)                     │ (None, 64)             │        28,928 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 32)             │         2,080 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 32)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 1)              │            33 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 173,237 (676.71 KB)

 Trainable params: 57,681 (225.32 KB)

 Non-trainable params: 192 (768.00 B)

 Optimizer params: 115,364 (450.64 KB)


✅ Config loaded:
  Features:  135
  Window:    5
  Threshold: 0.8472
  Temperature: 0.9568
  Feature list (first 10): ['onos1_flows', 'onos1_packets', 'onos1_bytes', 'onos1_flows_rate', 'onos1_packets_rate', 'onos1_bytes_rate', 'onos1_cpu', 'onos1_mem', 'onos1_pktin_total', 'onos1_pktin_total_rate']


## 3. Monitoring API

**Paste your SDN Controller-Level Monitoring API code in the cell below.**  
It should define: `start_monitoring()`, `stop_monitoring()`, `set_label()`, `show_status()`, `collected_samples`, `training_queue`, etc.

In [3]:
"""
Full SDN Controller-Level Monitoring (LSTM → DRL ready, updated v1.3.3)
CHANGES:
- RESET prev_* state on label flips (prevents delta/rate contamination across benign/attack boundaries)
- REDUCED smoothing for REST latency (deque maxlen reduced; can be 1 for no smoothing)
- Optional warm-up ignore (for evaluation only; still logs data)
"""

import requests, time, pandas as pd, numpy as np, subprocess, re, json, threading, hashlib, math, os, queue
from requests.auth import HTTPBasicAuth
from collections import defaultdict, deque
from datetime import datetime, timezone
from math import log2

# ===== Config (edit) =====
CONTROLLERS = [
    {"id": "onos1", "ip": "175.24.1.5", "port": 8181},
    {"id": "onos2", "ip": "175.24.1.6", "port": 8181},
    {"id": "onos3", "ip": "175.24.1.7", "port": 8181},
]
SWITCH_TO_CONTROLLER = {"s1":"175.24.1.7","s2":"175.24.1.6","s3":"175.24.1.6","s4":"175.24.1.5"}
AUTH = HTTPBasicAuth("onos", "rocks")
INTERVAL = 2
SWITCHES = ["s1","s2","s3","s4"]
CSV_PATH = "Online_test.csv"
PARQUET_PATH = "Online_test.parquet"
PARQUET_SYNC_EVERY = 100

SCHEMA_VERSION = "1.3.3"

# ---- Smoothing / evaluation knobs ----
REST_LAT_WINDOW = 2   # was 12; set to 1 to remove smoothing completely
WARMUP_SAMPLES = 0    # set e.g. 5 or 10 if you want to ignore first samples in evaluation

# ===== Globals =====
monitoring_active = False
monitoring_thread = None
prev_stats = {}            # deltas for packets/bytes/flows per controller
prev_metrics = {}          # previous PacketIn/Out/flowmiss counters per controller
prev_flows_set = set()     # for flow_arrival_rate
collected_samples = []
parquet_pending_counter = 0
training_queue = queue.Queue()
current_label = 0
flow_database = {}
attack_markers = []
monotonic_start = time.monotonic()

# reduced smoothing window here
rest_lat_hist = defaultdict(lambda: deque(maxlen=REST_LAT_WINDOW))

data_lock = threading.Lock()

# ===== Helpers =====
def now_utc_iso():
    return datetime.now(timezone.utc).strftime("%Y-%m-%dT%H:%M:%S.%fZ")

def query(ctrl, endpoint, timeout=3.0):
    url = f"http://{ctrl['ip']}:{ctrl['port']}/onos/v1{endpoint}"
    t0 = time.time()
    try:
        r = requests.get(url, auth=AUTH, timeout=timeout)
        lat_ms = (time.time() - t0) * 1000
        rest_lat_hist[(ctrl['id'], endpoint)].append(lat_ms)
        return (r.json() if r.status_code == 200 else None, lat_ms)
    except Exception:
        rest_lat_hist[(ctrl['id'], endpoint)].append(-1)
        return (None, -1)

def safe_rate(current, previous, interval):
    if current is None or previous is None: return 0.0
    if pd.isna(current) or pd.isna(previous): return 0.0
    try:
        curr_val = float(current)
        prev_val = float(previous)
    except (ValueError, TypeError):
        return 0.0
    delta = curr_val - prev_val
    if delta < 0: return 0.0
    safe_interval = max(1, interval)
    rate = delta / safe_interval
    if rate > 1e9: return 0.0
    return rate

def reset_prev_state(reason=""):
    """
    IMPORTANT: clears rate/delta history so next sample becomes a clean baseline.
    This reduces detection delay/glitches when label flips.
    """
    global prev_stats, prev_metrics, prev_flows_set
    prev_stats.clear()
    prev_metrics.clear()
    prev_flows_set.clear()
    # also clear REST latency history so it doesn't “remember” old regime
    rest_lat_hist.clear()

    attack_markers.append({
        "ts_utc": now_utc_iso(),
        "t_monotonic": time.monotonic() - monotonic_start,
        "event": "reset_prev_state",
        "reason": reason
    })
    print(f"[RESET] prev_* cleared. reason={reason}")

def get_docker_stats():
    try:
        p = subprocess.run(
            ['docker','stats','--no-stream','--format','{{.Name}}\t{{.CPUPerc}}\t{{.MemUsage}}'],
            capture_output=True, text=True, timeout=3
        )
        out = {}
        for line in p.stdout.strip().splitlines():
            parts = line.split('\t')
            if len(parts) >= 3:
                name = parts[0]
                try: cpu = float(parts[1].replace('%','').strip())
                except: cpu = 0.0
                mem = parts[2].split('/')[0].strip()
                out[name] = {"cpu": cpu, "mem": mem}
        return out
    except Exception:
        return {}

def get_hosts_mac_map():
    mac_map = {}
    for ctrl in CONTROLLERS:
        hosts_data, _ = query(ctrl, "/hosts")
        if hosts_data:
            for h in hosts_data.get("hosts", []):
                mac = (h.get("mac") or "").lower()
                ips = h.get("ipAddresses") or []
                if mac and ips:
                    mac_map[mac] = {"ip": ips[0]}
    return mac_map

def dump_switch_flows(sw):
    try:
        p = subprocess.run(['sudo','ovs-ofctl','-O','OpenFlow13','dump-flows',sw],
                           capture_output=True, text=True, timeout=5)
        return p.stdout
    except Exception:
        return ""

def generate_flow_id(switch, src_ip, dst_ip, src_port, dst_port, protocol):
    key = f"{switch}|{src_ip}|{dst_ip}|{src_port}|{dst_port}|{protocol}"
    return hashlib.md5(key.encode()).hexdigest()[:16]

def shannon_entropy(values):
    if not values: return 0.0
    total = sum(values)
    if total <= 0: return 0.0
    probs = [v/total for v in values if v > 0]
    return -sum(p*log2(p) for p in probs)

def parse_flows_enhanced(sw, text, mac_map):
    global flow_database
    flows = []
    if not text: return flows
    for line in text.strip().splitlines():
        if "n_packets=" not in line or "actions=" not in line: continue
        m_pkts = re.search(r'n_packets=(\d+)', line)
        if not m_pkts or int(m_pkts.group(1)) == 0: continue

        m_bytes = re.search(r'n_bytes=(\d+)', line)
        m_dur   = re.search(r'duration=(\d+(?:\.\d+)?)s', line)
        m_prio  = re.search(r'priority=(\d+)', line)
        m_cookie= re.search(r'cookie=(0x[0-9a-f]+)', line)

        m_srcmac= re.search(r'dl_src=([0-9a-f:]{17})', line, re.I)
        m_dstmac= re.search(r'dl_dst=([0-9a-f:]{17})', line, re.I)
        m_srcip = re.search(r'nw_src=([0-9.]+)', line)
        m_dstip = re.search(r'nw_dst=([0-9.]+)', line)

        m_proto = re.search(r'nw_proto=(\d+)', line)
        m_proto_token = re.search(r'\b(tcp|udp|icmp)\b', line, re.IGNORECASE)
        m_sport = re.search(r'tp_src=(\d+)', line)
        m_dport = re.search(r'tp_dst=(\d+)', line)

        m_tflags= re.search(r'tcp_flags=([^,\s]+)', line)
        m_itype = re.search(r'icmp_type=(\d+)', line)
        m_icode = re.search(r'icmp_code=(\d+)', line)
        m_inp   = re.search(r'in_port=(?:"([^"]+)|(\d+))', line)
        m_act   = re.search(r'actions=([^\s]+)', line)

        src_ip = m_srcip.group(1) if m_srcip else None
        dst_ip = m_dstip.group(1) if m_dstip else None
        if not src_ip and m_srcmac: src_ip = (mac_map.get(m_srcmac.group(1).lower()) or {}).get("ip")
        if not dst_ip and m_dstmac: dst_ip = (mac_map.get(m_dstmac.group(1).lower()) or {}).get("ip")
        if not (src_ip and dst_ip): continue

        proto_num = int(m_proto.group(1)) if m_proto else None
        if proto_num is not None:
            proto_map = {1: 'ICMP', 6: 'TCP', 17: 'UDP'}
            proto_name = proto_map.get(proto_num, f"other_{proto_num}")
        elif m_proto_token:
            token = m_proto_token.group(1).lower()
            token_map = {'tcp': ('TCP', 6), 'udp': ('UDP', 17), 'icmp': ('ICMP', 1)}
            proto_name, proto_num = token_map.get(token, (token.upper(), 0))
        else:
            proto_num = 0
            proto_name = 'OTHER'

        src_port  = int(m_sport.group(1)) if m_sport else 0
        dst_port  = int(m_dport.group(1)) if m_dport else 0

        fid = generate_flow_id(sw, src_ip, dst_ip, src_port, dst_port, proto_name)
        pkt = int(m_pkts.group(1))
        byt = int(m_bytes.group(1)) if m_bytes else 0
        dur = float(m_dur.group(1)) if m_dur else 0.0

        flow = {
            "flow_id": fid, "switch": sw,
            "cookie": m_cookie.group(1) if m_cookie else None,
            "priority": int(m_prio.group(1)) if m_prio else 0,
            "src_ip": src_ip, "dst_ip": dst_ip,
            "src_port": src_port, "dst_port": dst_port,
            "protocol": proto_name, "proto_num": proto_num,
            "packets": pkt, "bytes": byt, "duration": dur,
            "tcp_flags": m_tflags.group(1) if m_tflags else None,
            "icmp_type": int(m_itype.group(1)) if m_itype else None,
            "icmp_code": int(m_icode.group(1)) if m_icode else None,
            "in_port": (m_inp.group(1) if m_inp and m_inp.group(1) else
                        (int(m_inp.group(2)) if m_inp and m_inp.group(2) else None)),
            "actions": m_act.group(1) if m_act else None,
            "master_controller": SWITCH_TO_CONTROLLER.get(sw,"unknown"),
        }
        flows.append(flow)

        flow_database[fid] = {
            "switch": sw, "src_ip": src_ip, "dst_ip": dst_ip,
            "src_port": src_port, "dst_port": dst_port, "protocol": proto_name,
            "cookie": flow["cookie"], "priority": flow["priority"],
            "last_seen": now_utc_iso(), "total_packets": pkt, "total_bytes": byt
        }

    return flows

def get_onos_metrics_best_effort(ctrl):
    out = {"pktin_total": 0.0, "pktout_total": 0.0, "flowmiss_total": np.nan,
           "pktin_rate_direct": 0.0, "pktout_rate_direct": 0.0,
           "event_queue": np.nan, "atomix_nodes": np.nan, "gc_pause_ms": np.nan}

    mjson, _ = query(ctrl, "/metrics")
    if not mjson or not isinstance(mjson.get("metrics"), list):
        return out

    pktin_counters = []
    pktout_counters = []
    pktin_rates = []
    pktout_rates = []

    for m in mjson["metrics"]:
        name = (m.get("name") or "").lower()

        fval_count = None
        fval_mean_rate = None
        metric_obj = m.get('metric') if isinstance(m, dict) else None
        if isinstance(metric_obj, dict):
            meter = metric_obj.get('meter')
            if isinstance(meter, dict):
                fval_count = meter.get('counter')
                fval_mean_rate = meter.get('mean_rate')
            if fval_count is None and 'counter' in metric_obj and isinstance(metric_obj.get('counter'), dict):
                fval_count = metric_obj.get('counter').get('counter')

        if fval_count is None:
            if 'value' in m and m.get('value') is not None:
                try: fval_count = float(m.get('value'))
                except: fval_count = None
            elif 'measurements' in m and isinstance(m.get('measurements'), list) and m.get('measurements'):
                first = m.get('measurements')[0]
                if isinstance(first, dict):
                    fval_count = first.get('value') or first.get('count')
                else:
                    fval_count = first

        try:
            if fval_count is not None: fval_count = float(fval_count)
        except: fval_count = None
        try:
            if fval_mean_rate is not None: fval_mean_rate = float(fval_mean_rate)
        except: fval_mean_rate = None

        if re.search(r'of:[0-9a-f]+\.packet[_\.-]?in\.count', name):
            if fval_count is not None and fval_count > 0:
                pktin_counters.append(fval_count)

        if re.search(r'of:[0-9a-f]+\.packet[_\.-]?in\.rate', name):
            if fval_mean_rate is not None and fval_mean_rate > 0:
                pktin_rates.append(fval_mean_rate)

        if re.search(r'of:[0-9a-f]+\.packet[_\.-]?out\.count', name):
            if fval_count is not None and fval_count > 0:
                pktout_counters.append(fval_count)

        if re.search(r'of:[0-9a-f]+\.packet[_\.-]?out\.rate', name):
            if fval_mean_rate is not None and fval_mean_rate > 0:
                pktout_rates.append(fval_mean_rate)

        if re.search(r'flow[_\.-]?miss|miss[_\.-]?flow|flowmiss', name) and not re.search(r'rate', name):
            if fval_count is not None:
                out['flowmiss_total'] = fval_count

        if re.search(r'event[_\.-]?queue|queue[_\.-]?size', name):
            if fval_count is not None:
                out['event_queue'] = fval_count

        if 'atomix' in name and re.search(r'pending|backlog|queue', name):
            if fval_count is not None:
                out['atomix_nodes'] = fval_count

    if pktin_counters:
        out['pktin_total'] = sum(pktin_counters)
    if pktin_rates:
        out['pktin_rate_direct'] = sum(pktin_rates)
    if pktout_counters:
        out['pktout_total'] = sum(pktout_counters)
    if pktout_rates:
        out['pktout_rate_direct'] = sum(pktout_rates)

    fjson, _ = query(ctrl, "/flows")
    if fjson and isinstance(fjson.get("flows"), list):
        miss = sum(1 for f in fjson["flows"] if int(f.get("packets", 0)) == 0)
        if np.isnan(out.get("flowmiss_total", np.nan)):
            out["flowmiss_total"] = miss

    cjson, _ = query(ctrl, "/cluster")
    if cjson and isinstance(cjson.get("nodes"), list):
        out["atomix_nodes"] = sum(1 for n in cjson["nodes"] if n.get("status") == "READY")

    try:
        p = subprocess.run(['docker','logs','--tail','50', ctrl['id']],
                           capture_output=True, text=True, timeout=1.0)
        for line in (p.stderr or "").splitlines():
            if 'GC' in line and 'pause' in line.lower():
                m_gc = re.search(r'(\d+\.?\d*)\s*ms', line)
                if m_gc: out["gc_pause_ms"] = float(m_gc.group(1)); break
    except Exception:
        pass

    return out

def get_onos_flows_with_protocol(ctrl):
    proto_counts = {"tcp": 0, "udp": 0, "icmp": 0, "other": 0, "syn_flows": 0}
    fjson, _ = query(ctrl, "/flows")
    if not fjson or not isinstance(fjson.get("flows"), list):
        return proto_counts

    for flow in fjson["flows"]:
        packets = flow.get("packets", 0)
        if packets == 0:
            continue

        criteria = flow.get("selector", {}).get("criteria", [])
        proto_found = False
        is_tcp = False

        for c in criteria:
            if c.get("type") == "IP_PROTO":
                proto_num = c.get("protocol")
                if proto_num == 6:
                    proto_counts["tcp"] += 1
                    proto_found = True
                    is_tcp = True
                elif proto_num == 17:
                    proto_counts["udp"] += 1
                    proto_found = True
                elif proto_num == 1:
                    proto_counts["icmp"] += 1
                    proto_found = True
                break

        if is_tcp and packets <= 5:
            proto_counts["syn_flows"] += 1

        if not proto_found:
            for c in criteria:
                if c.get("type") == "ETH_TYPE" and c.get("ethType") in ["0x800", "0x0800"]:
                    proto_counts["other"] += 1
                    break

    return proto_counts

def compute_features(all_switch_flows, per_ctrl_stats, per_switch_stats, docker_stats, per_ctrl_metrics, per_ctrl_protos):
    global prev_stats, prev_flows_set, current_label, prev_metrics

    five_seen = set()
    dedup = []
    for f in all_switch_flows:
        key5 = (f["src_ip"], f["dst_ip"], f["src_port"], f["dst_port"], f["protocol"])
        if key5 in five_seen: continue
        five_seen.add(key5)
        dedup.append(f)
    flows = dedup

    utc_now = datetime.now(timezone.utc)
    features = {
        "schema": SCHEMA_VERSION,
        "timestamp_utc": utc_now.isoformat(),
        "timestamp_unix": utc_now.timestamp(),
        "t_monotonic": time.monotonic() - monotonic_start,
        "sample_id": len(collected_samples) + 1,
        "label": current_label
    }

    for ctrl in CONTROLLERS:
        cid = ctrl['id']
        agg = per_ctrl_stats.get(cid, {"flows":0,"packets":0,"bytes":0})
        features[f"{cid}_flows"]   = agg["flows"]
        features[f"{cid}_packets"] = agg["packets"]
        features[f"{cid}_bytes"]   = agg["bytes"]

        for k in ("flows","packets","bytes"):
            kk = f"{cid}_{k}"
            prev = prev_stats.get(kk)
            features[f"{kk}_rate"] = safe_rate(agg[k], prev, INTERVAL)
            prev_stats[kk] = agg[k]

        features[f"{cid}_cpu"] = (docker_stats.get(cid) or {}).get("cpu", 0.0)
        features[f"{cid}_mem"] = (docker_stats.get(cid) or {}).get("mem", "0B")

        m = per_ctrl_metrics.get(cid, {})
        for key in ("pktin_total","pktout_total","flowmiss_total"):
            v = m.get(key, np.nan)
            features[f"{cid}_{key}"] = v if not pd.isna(v) else 0.0
            prev_v = prev_metrics.get(f"{cid}_{key}")
            features[f"{cid}_{key}_rate"] = safe_rate(v, prev_v, INTERVAL)
            prev_metrics[f"{cid}_{key}"] = v if not pd.isna(v) else 0.0

        features[f"{cid}_event_queue"]  = m.get("event_queue", np.nan)
        features[f"{cid}_atomix_nodes"] = m.get("atomix_nodes", np.nan)
        features[f"{cid}_gc_pause_ms"]  = m.get("gc_pause_ms", np.nan)

        for ep in ("/flows","/hosts","/metrics"):
            hist = rest_lat_hist.get((cid, ep), [])
            if hist:
                clean = [x for x in hist if x >= 0]
                features[f"{cid}_rest{ep.replace('/','_')}_avg_ms"] = float(np.mean(clean)) if clean else np.nan
                features[f"{cid}_rest{ep.replace('/','_')}_max_ms"] = float(np.max(clean)) if clean else np.nan
            else:
                features[f"{cid}_rest{ep.replace('/','_')}_avg_ms"] = np.nan
                features[f"{cid}_rest{ep.replace('/','_')}_max_ms"] = np.nan

        proto = per_ctrl_protos.get(cid, {"tcp": 0, "udp": 0, "icmp": 0, "other": 0, "syn_flows": 0})
        features[f"{cid}_tcp_flows"] = proto["tcp"]
        features[f"{cid}_udp_flows"] = proto["udp"]
        features[f"{cid}_icmp_flows"] = proto["icmp"]
        features[f"{cid}_other_flows"] = proto["other"]
        features[f"{cid}_syn_flows"] = proto["syn_flows"]

    for sw in SWITCHES:
        s = per_switch_stats.get(sw, {"flows":0,"packets":0,"bytes":0})
        features[f"{sw}_flows"] = s["flows"]
        features[f"{sw}_packets"] = s["packets"]
        features[f"{sw}_bytes"] = s["bytes"]
        features[f"{sw}_master"] = SWITCH_TO_CONTROLLER.get(sw,"unknown")

    if flows:
        df = pd.DataFrame(flows)
        features["total_flows"]   = len(df)
        features["total_packets"] = int(df["packets"].sum())
        features["total_bytes"]   = int(df["bytes"].sum())
        features["avg_pkt_size"]  = features["total_bytes"]/max(1,features["total_packets"])

        counts = df["protocol"].value_counts()
        features["tcp_flows"] = int(counts.get("TCP",0))
        features["udp_flows"] = int(counts.get("UDP",0))
        features["icmp_flows"]= int(counts.get("ICMP",0))
        total = max(1,features["total_flows"])
        features["tcp_ratio"]  = features["tcp_flows"]/total
        features["udp_ratio"]  = features["udp_flows"]/total
        features["icmp_ratio"] = features["icmp_flows"]/total

        cluster_tcp = sum(per_ctrl_protos.get(c["id"], {}).get("tcp", 0) for c in CONTROLLERS)
        cluster_udp = sum(per_ctrl_protos.get(c["id"], {}).get("udp", 0) for c in CONTROLLERS)
        cluster_icmp = sum(per_ctrl_protos.get(c["id"], {}).get("icmp", 0) for c in CONTROLLERS)
        cluster_other = sum(per_ctrl_protos.get(c["id"], {}).get("other", 0) for c in CONTROLLERS)
        cluster_syn = sum(per_ctrl_protos.get(c["id"], {}).get("syn_flows", 0) for c in CONTROLLERS)
        cluster_total = max(1, cluster_tcp + cluster_udp + cluster_icmp + cluster_other)
        features["cluster_tcp_flows"] = cluster_tcp
        features["cluster_udp_flows"] = cluster_udp
        features["cluster_icmp_flows"] = cluster_icmp
        features["cluster_other_flows"] = cluster_other
        features["cluster_syn_flows"] = cluster_syn
        features["cluster_tcp_ratio"] = cluster_tcp / cluster_total
        features["cluster_udp_ratio"] = cluster_udp / cluster_total
        features["cluster_icmp_ratio"] = cluster_icmp / cluster_total
        features["cluster_syn_ratio"] = cluster_syn / max(1, cluster_tcp) if cluster_tcp > 0 else 0.0

        features["unique_srcs"] = df["src_ip"].nunique()
        features["unique_dsts"] = df["dst_ip"].nunique()
        features["src_dst_ratio"] = features["unique_srcs"]/max(1,features["unique_dsts"])
        features["src_entropy"] = shannon_entropy(df.groupby("src_ip")["packets"].sum().tolist())
        features["dst_entropy"] = shannon_entropy(df.groupby("dst_ip")["packets"].sum().tolist())

        features["unique_dst_ports"] = df["dst_port"].nunique()
        features["unique_src_ports"] = df["src_port"].nunique()
        features["port_diversity"] = (features["unique_src_ports"] + features["unique_dst_ports"])/2

        top_dst = df.groupby("dst_ip")["packets"].sum().idxmax()
        top_dst_pkts = int(df[df["dst_ip"] == top_dst]["packets"].sum())
        features["top_dst"] = top_dst
        features["top_dst_packets"] = top_dst_pkts
        features["top_dst_share"] = top_dst_pkts/max(1,features["total_packets"])

        curr = set(df["flow_id"])
        new_flows = len(curr - prev_flows_set)
        features["flow_arrival_rate"] = new_flows/max(1,INTERVAL)
        prev_flows_set.clear(); prev_flows_set.update(curr)

        features["avg_flow_duration"] = float(df["duration"].mean())
        features["max_flow_duration"] = float(df["duration"].max())
        features["min_flow_duration"] = float(df["duration"].min())
        features["avg_pkts_per_flow"] = features["total_packets"]/max(1,features["total_flows"])
        features["max_pkts_per_flow"] = int(df["packets"].max())
        stdv = df["packets"].std()
        features["std_pkts_per_flow"] = float(stdv if not pd.isna(stdv) else 0.0)

    else:
        features.update({
            "total_flows":0,"total_packets":0,"total_bytes":0,"avg_pkt_size":0.0,
            "tcp_flows":0,"udp_flows":0,"icmp_flows":0,"tcp_ratio":0.0,"udp_ratio":0.0,"icmp_ratio":0.0,
            "cluster_tcp_flows":0,"cluster_udp_flows":0,"cluster_icmp_flows":0,"cluster_other_flows":0,
            "cluster_syn_flows":0,"cluster_tcp_ratio":0.0,"cluster_udp_ratio":0.0,"cluster_icmp_ratio":0.0,"cluster_syn_ratio":0.0,
            "unique_srcs":0,"unique_dsts":0,"src_dst_ratio":0.0,"src_entropy":0.0,"dst_entropy":0.0,
            "unique_dst_ports":0,"unique_src_ports":0,"port_diversity":0.0,
            "top_dst":"none","top_dst_packets":0,"top_dst_share":0.0,
            "flow_arrival_rate":0.0,
            "avg_flow_duration":0.0,"max_flow_duration":0.0,"min_flow_duration":0.0,
            "avg_pkts_per_flow":0.0,"max_pkts_per_flow":0,"std_pkts_per_flow":0.0
        })

    ctrl_loads = [per_ctrl_stats.get(c["id"],{}).get("packets",0) for c in CONTROLLERS]
    sw_loads   = [per_switch_stats.get(sw,{}).get("packets",0) for sw in SWITCHES]
    features["ctrl_load_std"]  = float(np.std(ctrl_loads)) if ctrl_loads else 0.0
    features["ctrl_load_mean"] = float(np.mean(ctrl_loads)) if ctrl_loads else 0.0
    features["ctrl_load_max"]  = int(max(ctrl_loads)) if ctrl_loads else 0
    features["ctrl_load_min"]  = int(min(ctrl_loads)) if ctrl_loads else 0
    features["switch_load_std"]= float(np.std(sw_loads)) if sw_loads else 0.0
    features["switch_load_mean"]=float(np.mean(sw_loads)) if sw_loads else 0.0

    features["cluster_size"] = len(CONTROLLERS)
    features["active_controllers"] = sum(1 for c in CONTROLLERS if per_ctrl_stats.get(c["id"]))

    features["tracked_flows"] = len(flow_database)
    features["attack_in_progress"] = 1 if current_label==1 else 0
    features["cfg_interval_s"] = INTERVAL
    features["cfg_switches"] = ",".join(SWITCHES)
    features["cfg_controllers"] = ",".join(c["id"] for c in CONTROLLERS)

    # helpful for evaluation (warmup tracking)
    features["warmup"] = 1 if features["sample_id"] <= WARMUP_SAMPLES else 0

    return features

def monitoring_loop():
    global monitoring_active, collected_samples, current_label, parquet_pending_counter
    last_label_mtime = 0
    label_file = '/tmp/label_state'

    while monitoring_active:
        t_start_loop = time.time()
        old_label = current_label

        try:
            # ---- label sync ----
            try:
                if os.path.exists(label_file):
                    mtime = os.path.getmtime(label_file)
                    if mtime > last_label_mtime:
                        with open(label_file, 'r') as f:
                            lines = f.readlines()
                            if lines:
                                last_line = lines[-1].strip()
                                parts = last_line.split()
                                if len(parts) >= 1:
                                    try:
                                        new_label = int(parts[0])
                                        if new_label != current_label:
                                            current_label = new_label
                                            print(f"[Label Sync] Changed to {current_label}")
                                    except ValueError:
                                        pass
                        last_label_mtime = mtime
            except Exception as e:
                print(f"[Label Sync Error] {e}")

            # ---- RESET on label flip ----
            if current_label != old_label:
                reset_prev_state(reason=f"label_flip {old_label}->{current_label}")

            mac_map = get_hosts_mac_map()
            docker = get_docker_stats()

            per_switch_stats = {}
            per_ctrl_stats = {}
            all_switch_flows = []

            for sw in SWITCHES:
                text = dump_switch_flows(sw)
                flows = parse_flows_enhanced(sw, text, mac_map)
                all_switch_flows.extend(flows)

                s_flows = len(flows)
                s_packets = sum(f.get("packets", 0) for f in flows)
                s_bytes = sum(f.get("bytes", 0) for f in flows)
                per_switch_stats[sw] = {"flows": s_flows, "packets": s_packets, "bytes": s_bytes}

                ctrl_ip = SWITCH_TO_CONTROLLER.get(sw)
                ctrl_id = next((c["id"] for c in CONTROLLERS if c["ip"] == ctrl_ip), None)
                if ctrl_id:
                    agg = per_ctrl_stats.setdefault(ctrl_id, {"flows": 0, "packets": 0, "bytes": 0})
                    agg["flows"] += s_flows
                    agg["packets"] += s_packets
                    agg["bytes"] += s_bytes

            per_ctrl_metrics = {}
            per_ctrl_protos = {}
            for ctrl in CONTROLLERS:
                try:
                    per_ctrl_metrics[ctrl["id"]] = get_onos_metrics_best_effort(ctrl)
                    per_ctrl_protos[ctrl["id"]] = get_onos_flows_with_protocol(ctrl)
                except Exception as e:
                    print(f"[monitor] Error querying {ctrl['id']}: {e}")
                    per_ctrl_metrics[ctrl["id"]] = {"pktin_total": 0.0, "pktout_total": 0.0}
                    per_ctrl_protos[ctrl["id"]] = {"tcp": 0, "udp": 0, "icmp": 0, "other": 0, "syn_flows": 0}

            features = compute_features(all_switch_flows, per_ctrl_stats, per_switch_stats, docker, per_ctrl_metrics, per_ctrl_protos)

            with data_lock:
                collected_samples.append(features)
                parquet_pending_counter += 1
                if len(collected_samples) > 10000:
                    collected_samples.pop(0)

            training_queue.put(features)

            try:
                df = pd.DataFrame([features])
                header = not os.path.exists(CSV_PATH)
                df.to_csv(CSV_PATH, mode='a', header=header, index=False)
            except Exception as e:
                print("[monitor] CSV save failed:", e)

            if parquet_pending_counter >= PARQUET_SYNC_EVERY:
                try:
                    import pyarrow
                    with data_lock:
                        dfp = pd.DataFrame(collected_samples)
                        parquet_pending_counter = 0
                    dfp.to_parquet(PARQUET_PATH, index=False)
                except Exception:
                    pass

        except Exception as e:
            print("[monitor] loop error:", e)

        elapsed = time.time() - t_start_loop
        sleep_time = max(0, INTERVAL - elapsed)
        if monitoring_active and sleep_time > 0:
            time.sleep(sleep_time)

def start_monitoring(label=0):
    global monitoring_active, monitoring_thread, current_label
    if monitoring_active:
        print("Monitoring already running")
        return
    current_label = int(label)
    monitoring_active = True

    # reset state at start to avoid “first run” glitches
    reset_prev_state(reason="start_monitoring")

    monitoring_thread = threading.Thread(target=monitoring_loop, daemon=True)
    monitoring_thread.start()
    print(f"✓ Monitoring started (label={current_label}). Sampling every {INTERVAL}s")

def stop_monitoring():
    global monitoring_active, monitoring_thread, parquet_pending_counter
    if not monitoring_active:
        print("Monitoring is not running")
        return
    monitoring_active = False
    if monitoring_thread is not None:
        monitoring_thread.join(timeout=10)
    with data_lock:
        df_all = pd.DataFrame(collected_samples)

    try:
        header = not os.path.exists(CSV_PATH)
        df_all.to_csv(CSV_PATH, mode='a', header=header, index=False)
    except Exception:
        pass

    try:
        import pyarrow
        df_all.to_parquet(PARQUET_PATH, index=False)
        parquet_pending_counter = 0
    except Exception:
        pass
    print(f"✓ Monitoring stopped. {len(collected_samples)} samples saved to: {CSV_PATH}")

def set_label(label):
    global current_label
    old = current_label
    current_label = int(label)
    print(f"✓ Label set to {current_label}")
    if current_label != old:
        reset_prev_state(reason=f"manual_label {old}->{current_label}")

def show_status(verbose=True, top_n=10):
    print("=" * 60)
    print(f"MONITORING STATUS (schema v{SCHEMA_VERSION})")
    print("=" * 60)
    print(f"Active: {monitoring_active}")
    with data_lock:
        print(f"Collected samples (in memory): {len(collected_samples)}")
        last = collected_samples[-1] if collected_samples else None
    print(f"Training queue size: {training_queue.qsize()}")

    if not last:
        print("No samples recorded yet.")
        return

    print("\n" + "-" * 60)
    print("LAST SAMPLE")
    print("-" * 60)
    print(f"Timestamp: {last.get('timestamp_utc')}")
    print(f"Sample ID: {last.get('sample_id')} | Label: {last.get('label')} | Warmup: {last.get('warmup')}")
    print(f"Total flows: {last.get('total_flows', 0)} | Total packets: {last.get('total_packets', 0)}")
    print("=" * 60)

def list_active_flows(limit=20):
    items = sorted(flow_database.items(), key=lambda kv: kv[1].get("total_packets", 0), reverse=True)
    for fid, info in items[:limit]:
        print(fid, info.get("switch"), info.get("src_ip"), "->", info.get("dst_ip"), "pkts=", info.get("total_packets",0))

def get_flow_info(flow_id):
    return flow_database.get(flow_id)

print("=" * 70)
print("✓ SDN Controller Monitoring API v1.3.3 - READY")
print("=" * 70)
print("Functions: start_monitoring(label=0), stop_monitoring(), set_label(label)")
print("           show_status(verbose=True), list_active_flows(limit=20)")
print("           get_flow_info(flow_id)")
print("=" * 70)
print(f"REST_LAT_WINDOW={REST_LAT_WINDOW} | WARMUP_SAMPLES={WARMUP_SAMPLES}")
print("Key change: resets prev_* on label flips to reduce detection delay glitches")
print("=" * 70)


✓ SDN Controller Monitoring API v1.3.3 - READY
Functions: start_monitoring(label=0), stop_monitoring(), set_label(label)
           show_status(verbose=True), list_active_flows(limit=20)
           get_flow_info(flow_id)
REST_LAT_WINDOW=2 | WARMUP_SAMPLES=0
Key change: resets prev_* on label flips to reduce detection delay glitches


## 4. Live Preprocessing Pipeline

This replicates the **exact** preprocessing from the training notebook:
- Feature mapping (`cluster_syn_flows` → `syn_flows`, etc.)
- Memory string conversion (GiB → bytes)
- Median imputation → StandardScaler
- Sliding window creation

In [3]:
# ═══════════════════════════════════════════════════════════════════
# Section 4 — Live Preprocessing Pipeline
# ═══════════════════════════════════════════════════════════════════

def parse_memory_string(val):
    """Convert memory strings like '1.638GiB' to numeric bytes."""
    if isinstance(val, (int, float)):
        return float(val)
    if not isinstance(val, str):
        return np.nan
    val = val.strip()
    multipliers = {
        'gib': 1024**3, 'gb': 1e9,
        'mib': 1024**2, 'mb': 1e6,
        'kib': 1024,    'kb': 1e3,
        'b': 1
    }
    for suffix, mult in multipliers.items():
        if val.lower().endswith(suffix):
            try:
                return float(val[:len(val)-len(suffix)]) * mult
            except ValueError:
                return np.nan
    try:
        return float(val)
    except ValueError:
        return np.nan


def apply_feature_mapping(df):
    """
    Replicate EXACTLY the same feature engineering from training:
    - Keep cluster_syn_flows values as syn_flows
    - Keep cluster_syn_ratio values as syn_ratio
    - Calculate well_known_port_flows proxy
    - Calculate five_unique
    """
    # ── cluster_syn_flows → syn_flows (keep cluster version) ──
    if 'cluster_syn_flows' in df.columns and 'syn_flows' in df.columns:
        df.drop(columns=['syn_flows'], inplace=True)
        df.rename(columns={'cluster_syn_flows': 'syn_flows'}, inplace=True)
    elif 'cluster_syn_flows' in df.columns:
        df.rename(columns={'cluster_syn_flows': 'syn_flows'}, inplace=True)

    # ── cluster_syn_ratio → syn_ratio (keep cluster version) ──
    if 'cluster_syn_ratio' in df.columns and 'syn_ratio' in df.columns:
        df.drop(columns=['syn_ratio'], inplace=True)
        df.rename(columns={'cluster_syn_ratio': 'syn_ratio'}, inplace=True)
    elif 'cluster_syn_ratio' in df.columns:
        df.rename(columns={'cluster_syn_ratio': 'syn_ratio'}, inplace=True)

    # ── well_known_port_flows (proxy) ──
    if 'unique_dst_ports' in df.columns:
        udp = df['unique_dst_ports'].fillna(0).clip(lower=0)
        if 'well_known_port_flows' not in df.columns or df['well_known_port_flows'].isna().all():
            df['well_known_port_flows'] = np.where(
                udp > 0,
                np.minimum(udp, 1024) / np.maximum(udp, 1),
                0.0
            )

    # ── five_unique ──
    if 'unique_srcs' in df.columns and 'unique_dsts' in df.columns:
        if 'five_unique' not in df.columns or df['five_unique'].isna().all():
            df['five_unique'] = df['unique_srcs'].fillna(0) + df['unique_dsts'].fillna(0)

    return df


def preprocess_live_sample(raw_df):
    """
    Full preprocessing pipeline for live data.
    Input:  raw DataFrame from monitoring API (any number of rows)
    Output: preprocessed feature array ready for windowing
    """
    df = raw_df.copy()

    # 1. Feature mapping
    df = apply_feature_mapping(df)

    # 2. Convert memory strings to bytes
    mem_cols = [c for c in df.columns if '_mem' in c]
    for col in mem_cols:
        df[col] = df[col].apply(parse_memory_string)

    # 3. Replace inf with NaN
    df.replace([np.inf, -np.inf], np.nan, inplace=True)

    # 4. Ensure all required features exist
    for col in FEATURE_COLS:
        if col not in df.columns:
            df[col] = np.nan

    # 5. Select features in EXACT training order
    X = df[FEATURE_COLS].values.astype(np.float64)

    # 6. Impute (using training-fitted imputer)
    X = imputer.transform(X)

    # 7. Scale (using training-fitted scaler)
    X = scaler.transform(X)

    return X


def create_windows(X, window_len):
    """
    Create sliding windows from preprocessed feature array.
    Returns windows of shape (n_windows, window_len, n_features)
    """
    if len(X) < window_len:
        return np.empty((0, window_len, X.shape[1]))
    windows = []
    for i in range(len(X) - window_len + 1):
        windows.append(X[i:i + window_len])
    return np.array(windows)


def calibrate_probability(raw_prob, temperature):
    """Apply temperature scaling to raw sigmoid output."""
    eps = 1e-7
    logits = np.log(np.clip(raw_prob, eps, 1 - eps) /
                    np.clip(1 - raw_prob, eps, 1 - eps))
    return 1.0 / (1.0 + np.exp(-logits / temperature))


print('✅ Preprocessing pipeline defined.')
print(f'   Features expected: {N_FEATURES}')
print(f'   Window length:     {WINDOW_LEN}')

✅ Preprocessing pipeline defined.
   Features expected: 135
   Window length:     5


## 5. Start Monitoring & Real-Time Prediction

In [9]:
# ═══════════════════════════════════════════════════════════════════
# Section 5a — Start Monitoring (benign traffic first)
# ═══════════════════════════════════════════════════════════════════

start_monitoring(label=0)
print(f'Monitoring started. Waiting for initial {WINDOW_LEN} samples...')
time.sleep(WINDOW_LEN * 2 + 2)  # wait for enough samples
show_status()

[RESET] prev_* cleared. reason=start_monitoring
✓ Monitoring started (label=0). Sampling every 2s
Monitoring started. Waiting for initial 5 samples...
MONITORING STATUS (schema v1.3.3)
Active: True
Collected samples (in memory): 573
Training queue size: 573

------------------------------------------------------------
LAST SAMPLE
------------------------------------------------------------
Timestamp: 2026-02-27T08:20:03.909697+00:00
Sample ID: 573 | Label: 0 | Warmup: 0
Total flows: 4 | Total packets: 2061


In [24]:
# ═══════════════════════════════════════════════════════════════════
# Section 5b — Real-Time Prediction Loop
# ═══════════════════════════════════════════════════════════════════
# This cell runs continuously. Press the Stop button to end.

# Storage for predictions
prediction_history = []

# Rolling buffer for windowing (keeps last WINDOW_LEN preprocessed rows)
feature_buffer = deque(maxlen=LIVE_HISTORY_MAX)

# Set up live plot
fig, axes = plt.subplots(3, 1, figsize=(14, 10), sharex=True)
fig.suptitle('CNN+LSTM DDoS Detector — Live Predictions', fontsize=14, fontweight='bold')

print('🔴 LIVE PREDICTION LOOP STARTED')
print('   Press Stop (■) to end the loop.\n')

sample_counter = 0
last_processed = 0

try:
    while True:
        # Wait for new samples
        time.sleep(PREDICTION_INTERVAL)

        with data_lock:
            current_samples = list(collected_samples)

        n_available = len(current_samples)
        if n_available <= last_processed:
            continue

        # Process only NEW samples
        new_samples = current_samples[last_processed:]
        last_processed = n_available

        # Convert to DataFrame and preprocess
        df_new = pd.DataFrame(new_samples)
        X_new = preprocess_live_sample(df_new)

        # Add to rolling buffer
        for row in X_new:
            feature_buffer.append(row)

        # Need at least WINDOW_LEN samples in buffer
        if len(feature_buffer) < WINDOW_LEN:
            print(f'  Buffering... {len(feature_buffer)}/{WINDOW_LEN}')
            continue

        # Create window from the LAST WINDOW_LEN samples
        buffer_arr = np.array(list(feature_buffer))
        window = buffer_arr[-WINDOW_LEN:].reshape(1, WINDOW_LEN, N_FEATURES)

        # Predict
        raw_prob = model.predict(window, verbose=0).ravel()[0]
        cal_prob = calibrate_probability(raw_prob, TEMPERATURE)
        binary   = int(cal_prob >= BEST_THRESHOLD)
        confidence = abs(cal_prob - 0.5) * 2

        # Get ground truth label from latest sample
        true_label = new_samples[-1].get('label', -1)
        ts_utc     = new_samples[-1].get('timestamp_utc', '')
        sample_id  = new_samples[-1].get('sample_id', sample_counter)

        # Severity score
        last_row = new_samples[-1]
        syn_r = float(last_row.get('syn_ratio', 0) or 0)
        syn_f = float(last_row.get('syn_flows', 0) or 0)
        far   = float(last_row.get('flow_arrival_rate', 0) or 0)
        udst  = float(last_row.get('unique_dsts', 1) or 1)

        prediction_history.append({
            'sample_id':    sample_id,
            'timestamp':    ts_utc,
            'true_label':   true_label,
            'raw_prob':     raw_prob,
            'cal_prob':     cal_prob,
            'prediction':   binary,
            'confidence':   confidence,
            'syn_ratio':    syn_r,
            'syn_flows':    syn_f,
            'flow_arrival_rate': far,
            'unique_dsts':  udst,
            'total_flows':  last_row.get('total_flows', 0),
        })

        sample_counter += 1

        # Print status
        status = '🔴 ATTACK' if binary == 1 else '🟢 BENIGN'
        match  = '✅' if binary == true_label else '❌'
        print(f'[{sample_id:>5}] {status}  prob={cal_prob:.4f}  '
              f'conf={confidence:.3f}  true={true_label}  {match}')

        # ── Update live plot every 5 predictions ────────────────────
        if sample_counter % 5 == 0 and len(prediction_history) > 1:
            df_hist = pd.DataFrame(prediction_history[-PLOT_WINDOW:])

            for ax in axes:
                ax.clear()

            # Plot 1: Probability vs Threshold
            axes[0].plot(df_hist.index, df_hist['cal_prob'], 'b-', linewidth=1, label='P(attack)')
            axes[0].axhline(y=BEST_THRESHOLD, color='r', linestyle='--', alpha=0.7, label=f'Threshold={BEST_THRESHOLD:.3f}')
            axes[0].fill_between(df_hist.index, 0, 1,
                                 where=df_hist['true_label'] == 1,
                                 alpha=0.15, color='red', label='True attack')
            axes[0].set_ylabel('Attack Probability')
            axes[0].set_ylim(-0.05, 1.05)
            axes[0].legend(loc='upper right', fontsize=8)
            axes[0].set_title('Live Attack Probability', fontsize=11)

            # Plot 2: Binary Prediction vs True Label
            axes[1].step(df_hist.index, df_hist['prediction'], 'r-', linewidth=1.5, where='post', label='Predicted')
            axes[1].step(df_hist.index, df_hist['true_label'], 'g--', linewidth=1.5, where='post', label='True label')
            axes[1].set_ylabel('Label')
            axes[1].set_yticks([0, 1])
            axes[1].set_yticklabels(['Benign', 'Attack'])
            axes[1].legend(loc='upper right', fontsize=8)
            axes[1].set_title('Prediction vs Ground Truth', fontsize=11)

            # Plot 3: Total Flows (traffic volume)
            axes[2].plot(df_hist.index, df_hist['total_flows'], 'purple', linewidth=1)
            axes[2].fill_between(df_hist.index, 0, df_hist['total_flows'], alpha=0.3, color='purple')
            axes[2].set_ylabel('Total Flows')
            axes[2].set_xlabel('Sample')
            axes[2].set_title('Network Traffic Volume', fontsize=11)

            fig.tight_layout()
            fig.canvas.draw()
            fig.canvas.flush_events()

except KeyboardInterrupt:
    print(f'\n\n🛑 Prediction loop stopped after {sample_counter} predictions.')

<IPython.core.display.Javascript object>

🔴 LIVE PREDICTION LOOP STARTED
   Press Stop (■) to end the loop.

[   99] 🟢 BENIGN  prob=0.0704  conf=0.859  true=0  ✅
[  101] 🟢 BENIGN  prob=0.0204  conf=0.959  true=0  ✅
[  102] 🟢 BENIGN  prob=0.0416  conf=0.917  true=0  ✅
[  103] 🟢 BENIGN  prob=0.0103  conf=0.979  true=0  ✅
[  104] 🟢 BENIGN  prob=0.0075  conf=0.985  true=0  ✅
[  105] 🟢 BENIGN  prob=0.0084  conf=0.983  true=0  ✅
[  106] 🟢 BENIGN  prob=0.0019  conf=0.996  true=0  ✅
[  107] 🟢 BENIGN  prob=0.0031  conf=0.994  true=0  ✅
[  108] 🟢 BENIGN  prob=0.0045  conf=0.991  true=0  ✅
[  109] 🔴 ATTACK  prob=0.9996  conf=0.999  true=1  ✅
[  110] 🔴 ATTACK  prob=0.9696  conf=0.939  true=1  ✅
[  111] 🔴 ATTACK  prob=0.9965  conf=0.993  true=1  ✅
[  112] 🔴 ATTACK  prob=0.9992  conf=0.998  true=1  ✅
[  113] 🔴 ATTACK  prob=0.9995  conf=0.999  true=1  ✅
[  114] 🔴 ATTACK  prob=0.9991  conf=0.998  true=1  ✅
[  115] 🔴 ATTACK  prob=0.9992  conf=0.998  true=1  ✅
[  116] 🔴 ATTACK  prob=0.9991  conf=0.998  true=1  ✅
[  117] 🔴 ATTACK  prob=0.9981  c

## 5c. Label Control

Run the cell below to **switch labels** during the experiment.  
- `set_label(0)` → mark as **benign** traffic
- `set_label(1)` → mark as **attack** traffic

In [ ]:
# ── Switch label here when you start/stop attacks ──
set_label(1)   # Change to 0 for benign, 1 for attack

In [10]:
# ── Stop monitoring when done ──
stop_monitoring()
print(f'Total predictions collected: {len(prediction_history)}')

✓ Monitoring stopped. 613 samples saved to: Online_test_1.csv
Total predictions collected: 578


## 6. Post-Experiment Evaluation

In [11]:
# ═══════════════════════════════════════════════════════════════════
# Section 6 — Comprehensive Evaluation
# ═══════════════════════════════════════════════════════════════════

df_results = pd.DataFrame(prediction_history)

# Filter out samples with unknown labels
df_eval = df_results[df_results['true_label'].isin([0, 1])].copy()
print(f'Evaluating {len(df_eval)} samples (excluded {len(df_results) - len(df_eval)} with unknown labels)\n')

y_true = df_eval['true_label'].values
y_pred = df_eval['prediction'].values
y_prob = df_eval['cal_prob'].values

# ── Classification Report ──
print('═' * 60)
print('CLASSIFICATION REPORT')
print('═' * 60)
print(classification_report(y_true, y_pred, target_names=['Benign', 'Attack']))

# ── Key Metrics ──
roc_auc = roc_auc_score(y_true, y_prob) if len(np.unique(y_true)) > 1 else float('nan')
pr_auc  = average_precision_score(y_true, y_prob) if len(np.unique(y_true)) > 1 else float('nan')
f1      = f1_score(y_true, y_pred)
prec    = precision_score(y_true, y_pred, zero_division=0)
rec     = recall_score(y_true, y_pred, zero_division=0)

cm = confusion_matrix(y_true, y_pred)
tn, fp, fn, tp = cm.ravel() if cm.size == 4 else (0, 0, 0, 0)
fpr = fp / max(1, fp + tn)
fnr = fn / max(1, fn + tp)

print(f'\n── Live Test Metrics ──')
print(f'  ROC-AUC     : {roc_auc:.4f}')
print(f'  PR-AUC      : {pr_auc:.4f}')
print(f'  F1          : {f1:.4f}')
print(f'  Precision   : {prec:.4f}')
print(f'  Recall      : {rec:.4f}')
print(f'  FPR         : {fpr:.4f}')
print(f'  FNR         : {fnr:.4f}')
print(f'  TP={tp}  FP={fp}  TN={tn}  FN={fn}')

# ── Detection Latency ──
# How many samples after an attack starts before the model detects it?
attack_starts = []
in_attack = False
for i, row in df_eval.iterrows():
    if row['true_label'] == 1 and not in_attack:
        attack_starts.append(i)
        in_attack = True
    elif row['true_label'] == 0:
        in_attack = False

latencies = []
for start_idx in attack_starts:
    subset = df_eval.loc[start_idx:]
    for j, (idx, row) in enumerate(subset.iterrows()):
        if row['true_label'] != 1:
            break
        if row['prediction'] == 1:
            latencies.append(j)
            break

if latencies:
    print(f'\n── Detection Latency ──')
    print(f'  Mean: {np.mean(latencies):.1f} samples ({np.mean(latencies) * PREDICTION_INTERVAL:.1f}s)')
    print(f'  Max:  {np.max(latencies)} samples ({np.max(latencies) * PREDICTION_INTERVAL:.1f}s)')
    print(f'  Min:  {np.min(latencies)} samples ({np.min(latencies) * PREDICTION_INTERVAL:.1f}s)')
else:
    print('\n  ℹ️  No attack episodes detected for latency measurement.')

Evaluating 578 samples (excluded 0 with unknown labels)

════════════════════════════════════════════════════════════
CLASSIFICATION REPORT
════════════════════════════════════════════════════════════
              precision    recall  f1-score   support

      Benign       0.98      0.95      0.97       335
      Attack       0.94      0.97      0.95       243

    accuracy                           0.96       578
   macro avg       0.96      0.96      0.96       578
weighted avg       0.96      0.96      0.96       578


── Live Test Metrics ──
  ROC-AUC     : 0.9924
  PR-AUC      : 0.9893
  F1          : 0.9535
  Precision   : 0.9365
  Recall      : 0.9712
  FPR         : 0.0478
  FNR         : 0.0288
  TP=236  FP=16  TN=319  FN=7

── Detection Latency ──
  Mean: 0.0 samples (0.0s)
  Max:  0 samples (0.0s)
  Min:  0 samples (0.0s)


In [12]:
# ═══════════════════════════════════════════════════════════════════
# Section 6b — Confusion Matrix Plot
# ═══════════════════════════════════════════════════════════════════

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Confusion Matrix
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=['Benign', 'Attack'],
            yticklabels=['Benign', 'Attack'], ax=axes[0])
axes[0].set_xlabel('Predicted')
axes[0].set_ylabel('True')
axes[0].set_title(f'Confusion Matrix (Live Test)\nF1={f1:.4f}')

# ROC Curve
if len(np.unique(y_true)) > 1:
    fpr_curve, tpr_curve, _ = roc_curve(y_true, y_prob)
    axes[1].plot(fpr_curve, tpr_curve, 'b-', linewidth=2)
    axes[1].plot([0, 1], [0, 1], 'r--', alpha=0.5)
    axes[1].set_xlabel('False Positive Rate')
    axes[1].set_ylabel('True Positive Rate')
    axes[1].set_title(f'ROC Curve (AUC={roc_auc:.4f})')
    axes[1].grid(True, alpha=0.3)

# PR Curve
if len(np.unique(y_true)) > 1:
    prec_curve, rec_curve, _ = precision_recall_curve(y_true, y_prob)
    axes[2].plot(rec_curve, prec_curve, 'g-', linewidth=2)
    axes[2].set_xlabel('Recall')
    axes[2].set_ylabel('Precision')
    axes[2].set_title(f'PR Curve (AUC={pr_auc:.4f})')
    axes[2].grid(True, alpha=0.3)

fig.tight_layout()
fig.savefig(os.path.join(ARTIFACT_DIR, 'live_test_metrics.png'), dpi=150, bbox_inches='tight')
plt.show()
print(f'✅ Saved: {ARTIFACT_DIR}/live_test_metrics.png')

<IPython.core.display.Javascript object>

✅ Saved: outputs_detector/live_test_metrics.png


## 7. Time-Series Visualization

In [13]:
# ═══════════════════════════════════════════════════════════════════
# Section 7 — Full Time-Series Plot
# ═══════════════════════════════════════════════════════════════════

fig, axes = plt.subplots(4, 1, figsize=(16, 14), sharex=True)
fig.suptitle('CNN+LSTM Live Detection — Full Session', fontsize=14, fontweight='bold')

x = range(len(df_eval))

# 1. Attack Probability
axes[0].plot(x, df_eval['cal_prob'], 'b-', linewidth=0.8, alpha=0.9)
axes[0].axhline(y=BEST_THRESHOLD, color='r', linestyle='--', alpha=0.7,
                label=f'Threshold={BEST_THRESHOLD:.3f}')
axes[0].fill_between(x, 0, 1,
                     where=df_eval['true_label'].values == 1,
                     alpha=0.15, color='red', label='True attack window')
axes[0].set_ylabel('P(attack)')
axes[0].set_ylim(-0.05, 1.05)
axes[0].legend(fontsize=8)
axes[0].set_title('Calibrated Attack Probability')

# 2. Binary prediction vs truth
correct = (df_eval['prediction'].values == df_eval['true_label'].values).astype(int)
colors = ['green' if c else 'red' for c in correct]
axes[1].bar(x, df_eval['prediction'], color=colors, alpha=0.6, width=1.0)
axes[1].step(x, df_eval['true_label'], 'k-', linewidth=1.5, where='mid', label='True')
axes[1].set_ylabel('Label')
axes[1].set_yticks([0, 1])
axes[1].set_yticklabels(['Benign', 'Attack'])
axes[1].legend(fontsize=8)
axes[1].set_title('Predictions (green=correct, red=wrong)')

# 3. Confidence
axes[2].plot(x, df_eval['confidence'], 'orange', linewidth=0.8)
axes[2].fill_between(x, 0, df_eval['confidence'], alpha=0.3, color='orange')
axes[2].set_ylabel('Confidence')
axes[2].set_ylim(-0.05, 1.05)
axes[2].set_title('Prediction Confidence')

# 4. Traffic features
axes[3].plot(x, df_eval['total_flows'], 'purple', linewidth=0.8, label='Total flows')
ax3b = axes[3].twinx()
ax3b.plot(x, df_eval['syn_ratio'], 'red', linewidth=0.8, alpha=0.6, label='SYN ratio')
ax3b.set_ylabel('SYN Ratio', color='red')
axes[3].set_ylabel('Total Flows', color='purple')
axes[3].set_xlabel('Sample Index')
axes[3].set_title('Network Features')
axes[3].legend(loc='upper left', fontsize=8)
ax3b.legend(loc='upper right', fontsize=8)

fig.tight_layout()
fig.savefig(os.path.join(ARTIFACT_DIR, 'live_test_timeseries.png'), dpi=150, bbox_inches='tight')
plt.show()
print(f'✅ Saved: {ARTIFACT_DIR}/live_test_timeseries.png')

<IPython.core.display.Javascript object>

✅ Saved: outputs_detector/live_test_timeseries.png


In [14]:
# ═══════════════════════════════════════════════════════════════════
# Section 7b — Prediction Distribution
# ═══════════════════════════════════════════════════════════════════

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Histogram by class
benign_probs = df_eval[df_eval['true_label'] == 0]['cal_prob']
attack_probs = df_eval[df_eval['true_label'] == 1]['cal_prob']

axes[0].hist(benign_probs, bins=50, alpha=0.6, color='green', label='Benign', density=True)
axes[0].hist(attack_probs, bins=50, alpha=0.6, color='red', label='Attack', density=True)
axes[0].axvline(x=BEST_THRESHOLD, color='black', linestyle='--', label=f'Threshold={BEST_THRESHOLD:.3f}')
axes[0].set_xlabel('Calibrated Probability')
axes[0].set_ylabel('Density')
axes[0].set_title('Prediction Distribution by True Class')
axes[0].legend()

# Confidence by correctness
correct_conf = df_eval[df_eval['prediction'] == df_eval['true_label']]['confidence']
wrong_conf   = df_eval[df_eval['prediction'] != df_eval['true_label']]['confidence']

axes[1].hist(correct_conf, bins=50, alpha=0.6, color='green', label=f'Correct (n={len(correct_conf)})', density=True)
if len(wrong_conf) > 0:
    axes[1].hist(wrong_conf, bins=50, alpha=0.6, color='red', label=f'Wrong (n={len(wrong_conf)})', density=True)
axes[1].set_xlabel('Confidence')
axes[1].set_ylabel('Density')
axes[1].set_title('Confidence Distribution by Correctness')
axes[1].legend()

fig.tight_layout()
fig.savefig(os.path.join(ARTIFACT_DIR, 'live_test_distributions.png'), dpi=150, bbox_inches='tight')
plt.show()
print(f'✅ Saved: {ARTIFACT_DIR}/live_test_distributions.png')

<IPython.core.display.Javascript object>

✅ Saved: outputs_detector/live_test_distributions.png


In [15]:
# ═══════════════════════════════════════════════════════════════════
# Section 7c — Save Results CSV
# ═══════════════════════════════════════════════════════════════════

results_path = os.path.join(ARTIFACT_DIR, 'live_test_results.csv')
df_results.to_csv(results_path, index=False)

# Save summary metrics
live_metrics = {
    'total_samples': len(df_eval),
    'ROC_AUC': float(roc_auc),
    'PR_AUC': float(pr_auc),
    'F1': float(f1),
    'Precision': float(prec),
    'Recall': float(rec),
    'FPR': float(fpr),
    'FNR': float(fnr),
    'TP': int(tp), 'FP': int(fp), 'TN': int(tn), 'FN': int(fn),
    'threshold': BEST_THRESHOLD,
    'temperature': TEMPERATURE,
    'window_len': WINDOW_LEN,
    'timestamp': datetime.now().isoformat(),
}
with open(os.path.join(ARTIFACT_DIR, 'live_test_metrics.json'), 'w') as f:
    json.dump(live_metrics, f, indent=2)

print(f'✅ Results saved: {results_path}')
print(f'✅ Metrics saved: {ARTIFACT_DIR}/live_test_metrics.json')
print(f'\n🏁 Live testing complete!')

✅ Results saved: outputs_detector/live_test_results.csv
✅ Metrics saved: outputs_detector/live_test_metrics.json

🏁 Live testing complete!


In [5]:
# ═══════════════════════════════════════════════════════════════════
# Section 5a — Start Monitoring (benign traffic first)
# ═══════════════════════════════════════════════════════════════════

start_monitoring(label=0)
print(f'Monitoring started. Waiting for initial {WINDOW_LEN} samples...')
time.sleep(WINDOW_LEN * 2 + 2)  # wait for enough samples
show_status()

[RESET] prev_* cleared. reason=start_monitoring
✓ Monitoring started (label=0). Sampling every 2s
Monitoring started. Waiting for initial 5 samples...
MONITORING STATUS (schema v1.3.3)
Active: True
Collected samples (in memory): 5
Training queue size: 5

------------------------------------------------------------
LAST SAMPLE
------------------------------------------------------------
Timestamp: 2026-02-28T07:39:25.644721+00:00
Sample ID: 5 | Label: 0 | Warmup: 0
Total flows: 5 | Total packets: 1051


[Label Sync] Changed to 1
[RESET] prev_* cleared. reason=label_flip 0->1
[Label Sync] Changed to 0
[RESET] prev_* cleared. reason=label_flip 1->0


In [14]:
# ════════════════════════════════════════════════════════════════════
# BASELINE MITIGATION — LSTM-Driven, Paste AFTER monitoring cell.
# Assumes from notebook: collected_samples, data_lock,
#   model, scaler, imputer, FEATURE_COLS, WINDOW_LEN,
#   TEMPERATURE, BEST_THRESHOLD, N_FEATURES,
#   preprocess_live_sample, calibrate_probability
# ════════════════════════════════════════════════════════════════════
import subprocess, threading, requests, json, re, os, time
import numpy as np
import pandas as pd
from collections import Counter, defaultdict, deque
from datetime import datetime, timezone
from math import log2
from requests.auth import HTTPBasicAuth

# ─── ONOS Cluster Config ────────────────────────────────────────────
ONOS_NODES = [
    {'id': 'onos1', 'ip': '175.24.1.5', 'port': 8181},
    {'id': 'onos2', 'ip': '175.24.1.6', 'port': 8181},
    {'id': 'onos3', 'ip': '175.24.1.7', 'port': 8181},
]
ONOS_AUTH = ('onos', 'rocks')
CONTROLLER_IPS = {'175.24.1.5', '175.24.1.6', '175.24.1.7'}
SWITCHES = ['s1', 's2', 's3', 's4']
SWITCH_DPIDS = [
    'of:0000000000000001', 'of:0000000000000002',
    'of:0000000000000003', 'of:0000000000000004',
]

# ─── Detection Thresholds ──────────────────────────────────────────
P_RATE   = 0.55     # prob >= this → RATE_LIMIT
P_DROP   = 0.85     # prob >= this → DROP
P_CLEAR  = 0.35     # prob < this for M steps → clear
S_MIN    = 0.20     # min severity for DROP
C_MIN    = 0.40     # min confidence for any mitigation

# ─── Hysteresis ─────────────────────────────────────────────────────
N_ATTACK_CONFIRM = 2
M_CLEAR_CONFIRM  = 5

# ─── Progressive Mitigation ────────────────────────────────────────
# First attacker → RATE_LIMIT probe at 5 Mbps
# Second attacker → baseline_decide action, locked for whole attack
RATE_LIMIT_BPS_PROBE     = 5_000_000    # 5 Mbps probe for first attacker
RATE_LIMIT_BPS_CONFIRMED = 100_000      # 100 Kbps confirmed
RATE_LIMIT_DURATION      = 600          # long enough to cover attack
COOLDOWN_SECONDS         = 15

# ─── Duration ───────────────────────────────────────────────────────
BASELINE_DURATION = 600
SAVE_DIR = 'outputs_baseline'

# ─── Runtime State ──────────────────────────────────────────────────
mitigation_lock = threading.Lock()
active_mitigations = {}
cooldown_tracker = {}
hysteresis_above = 0
hysteresis_below = 0
attack_state = False
step_log = []
mitigation_log = []
discovery_log = []
PROTECTED_PORTS = set()
attackers_seen = set()          # global uniqueness tracker

# Per-attack tracking (reset when attack_state clears)
first_attacker = None           # {'ip', 'victim', 'port', 'key', 'step', 'elapsed_s'}
second_attacker = None          # {'ip', 'victim', 'key', 'step', 'elapsed_s', 'action'}
attack_confirmed_step = None
attack_confirmed_elapsed = None

# Controller CPU baseline for overhead calculation
controller_baseline_sum = defaultdict(float)
controller_baseline_count = defaultdict(int)


# ────────────────────────────────────────────────────────────────────
#  SYSTEM METRICS (for paper logging only)
# ────────────────────────────────────────────────────────────────────
def get_docker_cpu_mem():
    stats = {}
    try:
        out = subprocess.run(
            ['docker', 'stats', '--no-stream', '--format',
             '{{.Name}}\t{{.CPUPerc}}\t{{.MemUsage}}'],
            capture_output=True, text=True, timeout=5)
        if out.returncode == 0:
            for line in out.stdout.strip().splitlines():
                parts = line.split('\t')
                if len(parts) >= 3:
                    name = parts[0].strip()
                    cpu_str = parts[1].strip().replace('%', '')
                    mem_str = parts[2].strip().split('/')[0].strip()
                    try:
                        cpu = float(cpu_str)
                    except ValueError:
                        cpu = 0.0
                    stats[name] = {'cpu_pct': cpu, 'mem': mem_str}
    except Exception:
        pass
    return stats


def get_ovs_switch_stats():
    sw_stats = {}
    for sw in SWITCHES:
        try:
            out = subprocess.run(
                ['sudo', 'ovs-ofctl', '-O', 'OpenFlow13', 'dump-aggregate', sw],
                capture_output=True, text=True, timeout=3)
            if out.returncode == 0:
                m_pkt = re.search(r'packet_count=(\d+)', out.stdout)
                m_byt = re.search(r'byte_count=(\d+)', out.stdout)
                sw_stats[sw] = {
                    'packets': int(m_pkt.group(1)) if m_pkt else 0,
                    'bytes': int(m_byt.group(1)) if m_byt else 0,
                }
        except Exception:
            sw_stats[sw] = {'packets': 0, 'bytes': 0}
    return sw_stats


# ────────────────────────────────────────────────────────────────────
#  ONOS HELPERS
# ────────────────────────────────────────────────────────────────────
def get_master_controller(device_id):
    for node in ONOS_NODES:
        try:
            url = f"http://{node['ip']}:{node['port']}/onos/v1/mastership/{device_id}/master"
            r = requests.get(url, auth=ONOS_AUTH, timeout=2)
            if r.ok:
                master_id = r.json().get('nodeId', '')
                for n2 in ONOS_NODES:
                    if master_id and (master_id in n2['ip'] or master_id == n2['id']):
                        return n2
        except Exception:
            continue
    return ONOS_NODES[0]


def get_all_flows_from_onos():
    flows = []
    for node in ONOS_NODES:
        try:
            url = f"http://{node['ip']}:{node['port']}/onos/v1/flows"
            r = requests.get(url, auth=ONOS_AUTH, timeout=3)
            if r.ok:
                for flow in r.json().get('flows', []):
                    if flow.get('packets', 0) == 0:
                        continue
                    src_ip, dst_ip, proto = None, None, None
                    for c in flow.get('selector', {}).get('criteria', []):
                        if c.get('type') == 'IPV4_SRC':
                            src_ip = c.get('ip', '').split('/')[0]
                        elif c.get('type') == 'IPV4_DST':
                            dst_ip = c.get('ip', '').split('/')[0]
                        elif c.get('type') == 'IP_PROTO':
                            proto = c.get('protocol')
                    if src_ip and dst_ip:
                        flows.append({
                            'src_ip': src_ip, 'dst_ip': dst_ip, 'proto': proto,
                            'packets': flow.get('packets', 0),
                            'bytes': flow.get('bytes', 0),
                            'deviceId': flow.get('deviceId'),
                        })
                return flows
        except Exception:
            continue
    return flows


def discover_protected_ports():
    global PROTECTED_PORTS
    protected = set()
    for node in ONOS_NODES:
        try:
            url = f"http://{node['ip']}:{node['port']}/onos/v1/links"
            r = requests.get(url, auth=ONOS_AUTH, timeout=2)
            if r.ok:
                for link in r.json().get('links', []):
                    for ep in [link.get('src', {}), link.get('dst', {})]:
                        dev = ep.get('device', '')
                        port = ep.get('port', '')
                        try:
                            sw_idx = int(dev.split(':')[-1], 16)
                            protected.add(f's{sw_idx}-eth{port}')
                        except Exception:
                            continue
                break
        except Exception:
            continue
    PROTECTED_PORTS.update(protected)
    print(f'🔍 Protected ports: {sorted(PROTECTED_PORTS)}')


# ────────────────────────────────────────────────────────────────────
#  VICTIM / ATTACKER DISCOVERY
# ────────────────────────────────────────────────────────────────────
def _shannon_entropy(values):
    if not values or sum(values) == 0:
        return 0.0
    total = sum(values)
    probs = [v / total for v in values if v > 0]
    return -sum(p * log2(p) for p in probs)


def discover_victims_and_attackers():
    """Returns dict with victim info, top attacker, and ranked attackers list."""
    flows = get_all_flows_from_onos()
    empty_a = {'src_ip': None, 'packets': 0, 'bytes': 0,
               'flow_count': 0, 'share': 0.0, 'proto': 'TCP'}
    empty_v = {'victim_ip': None, 'victim_packets': 0, 'victim_bytes': 0,
               'victim_flow_count': 0, 'victim_share': 0.0,
               'suspiciousness': 0.0, 'attacker': empty_a.copy(),
               'attackers': []}

    if not flows:
        return empty_v

    dst_stats = defaultdict(lambda: {
        'packets': 0, 'bytes': 0, 'flows': 0, 'syn_flows': 0,
        'src_ips': [], 'devices': Counter(),
    })
    total_packets = sum(f['packets'] for f in flows)

    for f in flows:
        dst = f['dst_ip']
        if not dst:
            continue
        dst_stats[dst]['packets'] += f['packets']
        dst_stats[dst]['bytes'] += f['bytes']
        dst_stats[dst]['flows'] += 1
        dst_stats[dst]['src_ips'].append(f['packets'])
        dst_stats[dst]['devices'][f.get('deviceId')] += 1
        if f.get('proto') in [6, 'TCP'] and f['packets'] <= 5:
            dst_stats[dst]['syn_flows'] += 1

    candidates = []
    for dst_ip, stats in dst_stats.items():
        share = stats['packets'] / max(1, total_packets)
        flow_norm = stats['flows'] / max(1, len(flows))
        syn_share = stats['syn_flows'] / max(1, stats['flows'])
        ent = 1 - (_shannon_entropy(stats['src_ips'])
                   / max(1, log2(len(stats['src_ips']) + 1)))
        susp = 0.4 * share + 0.2 * flow_norm + 0.2 * syn_share + 0.2 * ent
        candidates.append({
            'victim_ip': dst_ip, 'victim_packets': stats['packets'],
            'victim_bytes': stats['bytes'], 'victim_flow_count': stats['flows'],
            'victim_share': share, 'suspiciousness': susp,
        })

    candidates.sort(key=lambda x: x['suspiciousness'], reverse=True)
    top = candidates[0] if candidates else empty_v

    # Find ALL attackers targeting this victim (ranked by packets)
    vip = top['victim_ip']
    all_attackers = []
    if vip:
        victim_flows = [f for f in flows if f['dst_ip'] == vip]
        src_stats = defaultdict(lambda: {'packets': 0, 'bytes': 0, 'flows': 0, 'proto': Counter()})
        for f in victim_flows:
            s = f['src_ip']
            if not s:
                continue
            src_stats[s]['packets'] += f['packets']
            src_stats[s]['bytes'] += f['bytes']
            src_stats[s]['flows'] += 1
            src_stats[s]['proto'][f.get('proto', 'TCP')] += 1
        vtotal = sum(st['packets'] for st in src_stats.values())
        for src_ip, st in src_stats.items():
            all_attackers.append({
                'src_ip': src_ip, 'packets': st['packets'], 'bytes': st['bytes'],
                'flow_count': st['flows'],
                'share': st['packets'] / max(1, vtotal),
                'proto': st['proto'].most_common(1)[0][0] if st['proto'] else 'TCP',
            })
        all_attackers.sort(key=lambda x: x['packets'], reverse=True)
        top['attacker'] = all_attackers[0] if all_attackers else empty_a.copy()
    else:
        top['attacker'] = empty_a.copy()
    top['attackers'] = all_attackers

    return top


# ────────────────────────────────────────────────────────────────────
#  MITIGATION FUNCTIONS
# ────────────────────────────────────────────────────────────────────
def get_host_location(target_ip):
    for node in ONOS_NODES:
        try:
            url = f"http://{node['ip']}:{node['port']}/onos/v1/hosts"
            r = requests.get(url, auth=ONOS_AUTH, timeout=1)
            if r.ok:
                for host in r.json().get('hosts', []):
                    if target_ip in host.get('ipAddresses', []):
                        locs = host.get('locations', [])
                        if locs:
                            return locs[0].get('elementId'), locs[0].get('port')
        except Exception:
            continue
    return None, None


def find_port_for_ip(target_ip):
    device_id, port_num = get_host_location(target_ip)
    if device_id and port_num:
        try:
            sw_idx = int(device_id.split(':')[-1], 16)
            return f's{sw_idx}-eth{port_num}'
        except Exception:
            pass
    fallback = {
        '10.10.0.1': 's1-eth1', '10.10.0.2': 's1-eth2',
        '10.10.0.3': 's1-eth3', '10.10.0.4': 's1-eth4',
        '10.10.0.101': 's3-eth1', '10.10.0.102': 's3-eth2',
    }
    return fallback.get(target_ip)


def install_drop_flow(attacker_ip, victim_ip, timeout=0):
    if not attacker_ip or not victim_ip:
        return False
    if attacker_ip in CONTROLLER_IPS or victim_ip in CONTROLLER_IPS:
        return False
    ok = 0
    with mitigation_lock:
        for dev in SWITCH_DPIDS:
            master = get_master_controller(dev)
            flow = {
                'priority': 65000,
                'isPermanent': (timeout == 0),
                'deviceId': dev,
                'treatment': {'instructions': []},
                'selector': {'criteria': [
                    {'type': 'ETH_TYPE', 'ethType': '0x0800'},
                    {'type': 'IPV4_SRC', 'ip': f'{attacker_ip}/32'},
                    {'type': 'IPV4_DST', 'ip': f'{victim_ip}/32'},
                ]},
            }
            if timeout > 0:
                flow['timeout'] = timeout
                flow['isPermanent'] = False
            try:
                url = f"http://{master['ip']}:{master['port']}/onos/v1/flows"
                requests.post(url, json={'flows': [flow]}, auth=ONOS_AUTH, timeout=2)
                ok += 1
            except Exception:
                continue
    ttl_str = f'{timeout}s' if timeout > 0 else 'PERM'
    print(f'🛡️  [DROP {ttl_str}] {attacker_ip} → {victim_ip} on {ok}/4 switches')
    return ok > 0


def clear_rate_limit(port):
    if not port:
        return
    try:
        subprocess.run(f'sudo ovs-vsctl -- clear Port {port} qos',
                       shell=True, timeout=3, capture_output=True)
        subprocess.run('sudo ovs-vsctl --all destroy QoS',
                       shell=True, timeout=3, capture_output=True)
        subprocess.run('sudo ovs-vsctl --all destroy Queue',
                       shell=True, timeout=3, capture_output=True)
    except Exception:
        pass


def apply_rate_limit(port, bps, duration=RATE_LIMIT_DURATION):
    if not port or port in PROTECTED_PORTS:
        return False
    try:
        check = subprocess.run(
            f'sudo ovs-vsctl --if-exists get Port {port} name',
            shell=True, capture_output=True, timeout=3)
        if check.returncode != 0:
            return False
        cmd = (f'sudo ovs-vsctl -- set Port {port} qos=@newqos '
               f'-- --id=@newqos create QoS type=linux-htb '
               f'other-config:max-rate={bps} queues:1=@q1 '
               f'-- --id=@q1 create Queue other-config:max-rate={bps}')
        subprocess.run(cmd, shell=True, timeout=3)
        bps_h = f'{bps/1e6:.1f}Mbps' if bps >= 1e6 else f'{bps/1e3:.0f}Kbps'
        print(f'🚦 [RATE_LIMIT] {port} @ {bps_h}')
        return True
    except Exception:
        return False


def remove_drop_flows(attacker_ip, victim_ip):
    for sw in SWITCHES:
        match = f'ip,nw_src={attacker_ip},nw_dst={victim_ip}'
        try:
            subprocess.run(
                ['sudo', 'ovs-ofctl', '-O', 'OpenFlow13', 'del-flows', sw, match],
                timeout=3, capture_output=True)
        except Exception:
            continue


def clear_all_mitigations():
    global active_mitigations
    for key, info in list(active_mitigations.items()):
        parts = key.split('->')
        if len(parts) == 2:
            a, v = parts
            if info.get('type') == 'DROP':
                remove_drop_flows(a, v)
            elif info.get('type') in ('RATE_LIMIT', 'RATE_PROBE'):
                clear_rate_limit(info.get('port'))
    active_mitigations.clear()
    for sw in SWITCHES:
        try:
            subprocess.run(['sudo', 'ovs-ofctl', '-O', 'OpenFlow13',
                            'del-flows', sw, 'priority=65000'],
                           capture_output=True, timeout=3)
        except Exception:
            pass
    try:
        subprocess.run('sudo ovs-vsctl --all destroy QoS', shell=True, timeout=3, capture_output=True)
        subprocess.run('sudo ovs-vsctl --all destroy Queue', shell=True, timeout=3, capture_output=True)
    except Exception:
        pass
    print('🧹 All mitigations cleared')


def reset_attack_tracking():
    """Reset per-attack attacker tracking when attack_state clears."""
    global first_attacker, second_attacker
    global attack_confirmed_step, attack_confirmed_elapsed
    first_attacker = None
    second_attacker = None
    attack_confirmed_step = None
    attack_confirmed_elapsed = None


# ────────────────────────────────────────────────────────────────────
#  BASELINE POLICY — LSTM P drives all decisions
# ────────────────────────────────────────────────────────────────────
def baseline_decide(prob, severity, confidence):
    """
    LSTM probability + confidence + severity gating.
    Returns (action: 0=ALLOW, 1=RATE_LIMIT, 2=DROP, reason: str)
    """
    global hysteresis_above, hysteresis_below, attack_state
    global attack_confirmed_step, attack_confirmed_elapsed

    # Low confidence → don't act
    if confidence < C_MIN:
        hysteresis_below += 1
        hysteresis_above = 0
        if hysteresis_below >= M_CLEAR_CONFIRM and attack_state:
            attack_state = False
            print(f'⬇️  Attack CLEARED (low confidence for {M_CLEAR_CONFIRM} steps)')
        return 0, 'low_confidence'

    if prob < P_CLEAR:
        hysteresis_below += 1
        hysteresis_above = 0
        if hysteresis_below >= M_CLEAR_CONFIRM and attack_state:
            attack_state = False
            print(f'⬇️  Attack CLEARED (P < {P_CLEAR} for {M_CLEAR_CONFIRM} steps)')
        return 0, 'below_clear'

    if prob < P_RATE:
        hysteresis_below = 0
        hysteresis_above = 0
        if attack_state:
            return 1, 'attack_persists_low'
        return 0, 'below_rate'

    # prob >= P_RATE
    hysteresis_below = 0
    hysteresis_above += 1
    if not attack_state:
        if hysteresis_above >= N_ATTACK_CONFIRM:
            attack_state = True
            print(f'⬆️  Attack CONFIRMED after {N_ATTACK_CONFIRM} steps (P={prob:.4f})')
        else:
            return 0, f'confirming_{hysteresis_above}/{N_ATTACK_CONFIRM}'

    if prob >= P_DROP and severity >= S_MIN:
        return 2, 'high_prob_high_severity'
    else:
        return 1, 'moderate_attack'


def execute_baseline_action(action, prob, step_idx, elapsed_s):
    """
    LSTM-driven progressive mitigation:
      1) First attacker discovered → RATE_LIMIT probe (5 Mbps)
      2) Second attacker discovered → baseline_decide action, LOCKED for
         the whole attack (until attack_state clears)
    """
    global first_attacker, second_attacker

    result = {'action': 'ALLOW', 'applied': False,
              'victim': None, 'attacker': None, 'port': None,
              'reason': '', 'bps': 0, 'is_new_attacker': False,
              'first_detect_latency_s': None, 'second_detect_latency_s': None}

    if action == 0:
        result['applied'] = True
        result['reason'] = 'allow'
        return result

    # ── Second attacker already locked? Keep its mitigation ──────────
    if second_attacker is not None:
        sa = second_attacker
        sa_key = sa['key']
        # Re-apply if somehow missing
        if sa_key not in active_mitigations:
            if sa['action'] == 'DROP':
                install_drop_flow(sa['ip'], sa['victim'], timeout=0)
                active_mitigations[sa_key] = {'type': 'DROP', 'ts': time.time()}
        result.update({
            'action': sa['action'], 'applied': True,
            'attacker': sa['ip'], 'victim': sa['victim'],
            'reason': f'locked_{sa["action"].lower()}_whole_attack',
            'second_detect_latency_s': sa.get('latency_s'),
        })
        return result

    # ── Discover targets ────────────────────────────────────────────
    disc = discover_victims_and_attackers()
    victim_ip = disc['victim_ip']
    if not victim_ip:
        result['reason'] = 'no_victim'
        return result

    all_attackers = disc.get('attackers', [])
    top_attacker = disc['attacker']
    attacker_ip = top_attacker['src_ip']
    if not attacker_ip:
        result['reason'] = 'no_attacker'
        return result

    result['victim'] = victim_ip
    result['attacker'] = attacker_ip

    # Global tracking
    is_new = attacker_ip not in attackers_seen
    result['is_new_attacker'] = is_new
    if is_new:
        attackers_seen.add(attacker_ip)

    # ── First attacker: RATE_LIMIT probe ────────────────────────────
    if first_attacker is None:
        port = find_port_for_ip(attacker_ip)
        key = f'{attacker_ip}->{victim_ip}'
        latency = elapsed_s - attack_confirmed_elapsed if attack_confirmed_elapsed else None
        applied = apply_rate_limit(port, bps=RATE_LIMIT_BPS_PROBE, duration=RATE_LIMIT_DURATION)
        first_attacker = {
            'ip': attacker_ip, 'victim': victim_ip, 'port': port,
            'key': key, 'step': step_idx, 'elapsed_s': elapsed_s,
            'latency_s': latency,
        }
        if applied:
            active_mitigations[key] = {'type': 'RATE_PROBE', 'port': port, 'ts': time.time()}
        discovery_log.append({
            'step': step_idx, 'timestamp': datetime.now(timezone.utc).isoformat(),
            'event': 'FIRST_ATTACKER', 'attacker_ip': attacker_ip,
            'victim_ip': victim_ip, 'packets': top_attacker['packets'],
            'share': top_attacker['share'], 'proto': top_attacker['proto'],
            'detection_latency_s': latency,
        })
        mitigation_log.append({
            'step': step_idx, 'timestamp': datetime.now(timezone.utc).isoformat(),
            'type': 'RATE_PROBE_5MBPS', 'attacker': attacker_ip,
            'victim': victim_ip, 'port': port, 'bps': RATE_LIMIT_BPS_PROBE,
            'detection_latency_s': latency,
        })
        print(f'    🆕 1st attacker: {attacker_ip} → {victim_ip} '
              f'(pkts={top_attacker["packets"]}, share={top_attacker["share"]:.2f})')
        result.update({
            'action': 'RATE_LIMIT', 'applied': applied, 'port': port,
            'bps': RATE_LIMIT_BPS_PROBE,
            'reason': 'first_attacker_probe',
            'first_detect_latency_s': latency,
        })
        return result

    # ── Look for second attacker (different from first) ─────────────
    second_ip = None
    second_info = None
    for cand in all_attackers:
        if cand['src_ip'] and cand['src_ip'] != first_attacker['ip']:
            second_ip = cand['src_ip']
            second_info = cand
            break

    if second_ip:
        key = f'{second_ip}->{victim_ip}'
        latency = elapsed_s - attack_confirmed_elapsed if attack_confirmed_elapsed else None
        is_new2 = second_ip not in attackers_seen
        if is_new2:
            attackers_seen.add(second_ip)

        # Remove first attacker's probe rate limit
        first_key = first_attacker['key']
        if first_key in active_mitigations:
            fa_info = active_mitigations[first_key]
            if fa_info.get('type') == 'RATE_PROBE':
                clear_rate_limit(fa_info.get('port'))
            active_mitigations.pop(first_key, None)
            print(f'    🔄 Removed probe on first attacker {first_attacker["ip"]}')
            mitigation_log.append({
                'step': step_idx, 'timestamp': datetime.now(timezone.utc).isoformat(),
                'type': 'FIRST_PROBE_REMOVED', 'attacker': first_attacker['ip'],
                'victim': first_attacker['victim'],
            })

        # Apply baseline_decide action to second attacker, LOCKED
        ACTION_NAMES = {0: 'ALLOW', 1: 'RATE_LIMIT', 2: 'DROP'}
        lock_action = ACTION_NAMES.get(action, 'DROP')  # use what baseline_decide chose

        applied = False
        if lock_action == 'DROP':
            applied = install_drop_flow(second_ip, victim_ip, timeout=0)
            if applied:
                active_mitigations[key] = {'type': 'DROP', 'ts': time.time()}
        elif lock_action == 'RATE_LIMIT':
            port2 = find_port_for_ip(second_ip)
            applied = apply_rate_limit(port2, bps=RATE_LIMIT_BPS_CONFIRMED, duration=RATE_LIMIT_DURATION)
            if applied:
                active_mitigations[key] = {'type': 'RATE_LIMIT', 'port': port2, 'ts': time.time()}

        second_attacker = {
            'ip': second_ip, 'victim': victim_ip, 'key': key,
            'step': step_idx, 'elapsed_s': elapsed_s,
            'latency_s': latency, 'action': lock_action,
        }
        discovery_log.append({
            'step': step_idx, 'timestamp': datetime.now(timezone.utc).isoformat(),
            'event': 'SECOND_ATTACKER', 'attacker_ip': second_ip,
            'victim_ip': victim_ip, 'packets': second_info['packets'],
            'share': second_info['share'], 'proto': second_info['proto'],
            'detection_latency_s': latency, 'locked_action': lock_action,
        })
        mitigation_log.append({
            'step': step_idx, 'timestamp': datetime.now(timezone.utc).isoformat(),
            'type': f'SECOND_ATTACKER_LOCKED_{lock_action}',
            'attacker': second_ip, 'victim': victim_ip,
            'detection_latency_s': latency,
        })
        print(f'    🆕 2nd attacker: {second_ip} → {victim_ip} '
              f'(pkts={second_info["packets"]}, share={second_info["share"]:.2f}) '
              f'→ LOCKED {lock_action}')
        result.update({
            'action': lock_action, 'applied': applied,
            'attacker': second_ip, 'victim': victim_ip,
            'reason': f'second_attacker_locked_{lock_action.lower()}',
            'second_detect_latency_s': latency,
        })
        return result

    # ── Still only first attacker, keep probe active ────────────────
    fa_key = first_attacker['key']
    if fa_key not in active_mitigations:
        port = first_attacker['port'] or find_port_for_ip(first_attacker['ip'])
        apply_rate_limit(port, bps=RATE_LIMIT_BPS_PROBE, duration=RATE_LIMIT_DURATION)
        active_mitigations[fa_key] = {'type': 'RATE_PROBE', 'port': port, 'ts': time.time()}
    result.update({
        'action': 'RATE_LIMIT', 'applied': True,
        'attacker': first_attacker['ip'], 'victim': first_attacker['victim'],
        'port': first_attacker['port'], 'bps': RATE_LIMIT_BPS_PROBE,
        'reason': 'first_probe_active_searching_second',
    })
    return result


def get_ground_truth():
    try:
        with open('/tmp/label_state', 'r') as f:
            lines = f.readlines()
            if lines:
                parts = lines[-1].strip().split()
                label = int(parts[0])
                if len(parts) > 1 and parts[1].isdigit() and len(parts) > 2:
                    name = parts[2]
                elif len(parts) > 1 and not parts[1].isdigit():
                    name = parts[1]
                else:
                    name = parts[2] if len(parts) > 2 else 'UNKNOWN'
                return label, name
    except Exception:
        pass
    return 0, 'UNKNOWN'


# ════════════════════════════════════════════════════════════════════
#  MAIN LOOP
# ════════════════════════════════════════════════════════════════════
os.makedirs(SAVE_DIR, exist_ok=True)
discover_protected_ports()
clear_all_mitigations()

print('╔════════════════════════════════════════════════════════════╗')
print('║   BASELINE — LSTM-Driven + Progressive Mitigation        ║')
print(f'║   P_RATE={P_RATE}  P_DROP={P_DROP}  P_CLEAR={P_CLEAR}              ║')
print(f'║   1st attacker → 5Mbps probe                            ║')
print(f'║   2nd attacker → baseline action locked for attack      ║')
print(f'║   Duration: {BASELINE_DURATION}s                                   ║')
print('╚════════════════════════════════════════════════════════════╝')

feature_buffer = deque(maxlen=5000)
last_processed = 0
step = 0
t_start = time.time()
prev_attack_state = False
reset_attack_tracking()

try:
    while time.time() - t_start < BASELINE_DURATION:
        time.sleep(2)

        with data_lock:
            current_samples = list(collected_samples)

        n_available = len(current_samples)
        if n_available <= last_processed:
            continue

        new_samples = current_samples[last_processed:]
        last_processed = n_available

        # Preprocess (same pipeline as notebook cell 11)
        df_new = pd.DataFrame(new_samples)
        X_new = preprocess_live_sample(df_new)
        for row in X_new:
            feature_buffer.append(row)

        if len(feature_buffer) < WINDOW_LEN:
            print(f'  Warming up ({len(feature_buffer)}/{WINDOW_LEN})')
            step += 1
            continue

        buffer_arr = np.array(list(feature_buffer))
        window = buffer_arr[-WINDOW_LEN:].reshape(1, WINDOW_LEN, N_FEATURES)

        # ── LSTM Prediction ─────────────────────────────────────────
        raw_prob = model.predict(window, verbose=0).ravel()[0]
        cal_prob = float(calibrate_probability(raw_prob, TEMPERATURE))
        binary = int(cal_prob >= BEST_THRESHOLD)
        confidence = abs(cal_prob - 0.5) * 2
        severity = min(1.0, cal_prob * 1.2)
        elapsed_s = round(time.time() - t_start, 1)

        # Ground truth (for logging/evaluation only — NOT for decisions)
        gt_label, gt_name = get_ground_truth()

        # System metrics (for paper logging only)
        docker_stats = get_docker_cpu_mem()
        ovs_stats = get_ovs_switch_stats()
        total_cpu = sum(v.get('cpu_pct', 0) for v in docker_stats.values())
        onos_cpu = {k: v.get('cpu_pct', 0) for k, v in docker_stats.items()
                    if k.startswith('onos')}
        total_sw_packets = sum(v['packets'] for v in ovs_stats.values())
        total_sw_bytes = sum(v['bytes'] for v in ovs_stats.values())
        # Controller overhead
        onos_overhead = {}
        for cid in ['onos1', 'onos2', 'onos3']:
            cur = float(onos_cpu.get(cid, 0.0))
            if gt_label == 0:
                controller_baseline_sum[cid] += cur
                controller_baseline_count[cid] += 1
            n0 = controller_baseline_count[cid]
            base = (controller_baseline_sum[cid] / n0) if n0 > 0 else cur
            onos_overhead[cid] = cur - base
        ctrl_cpu_avg = float(np.mean([onos_cpu.get(c, 0) for c in ['onos1', 'onos2', 'onos3']]))
        ctrl_overhead_avg = float(np.mean([onos_overhead[c] for c in ['onos1', 'onos2', 'onos3']]))

        # ── LSTM-driven baseline decision ───────────────────────────
        action, reason = baseline_decide(cal_prob, severity, confidence)

        # Record when attack is first confirmed (for latency calc)
        if attack_state and attack_confirmed_step is None:
            attack_confirmed_step = step
            attack_confirmed_elapsed = elapsed_s

        # When attack_state clears → clean up everything
        if prev_attack_state and not attack_state:
            print(f'  🧹 Attack ended at step {step} → clearing all mitigations')
            clear_all_mitigations()
            reset_attack_tracking()
        prev_attack_state = attack_state

        # ── Execute mitigation ──────────────────────────────────────
        result = execute_baseline_action(action, cal_prob, step, elapsed_s)

        # Pretty print
        ACTION_ICONS = {'ALLOW': '✅ ALLOW', 'RATE_LIMIT': '🚦 RATE', 'DROP': '🛡️ DROP'}
        match_icon = '✅' if (binary == gt_label) else '❌'
        atk_str = f' [{result["attacker"]}→{result["victim"]}]' if result.get('attacker') else ''
        action_disp = ACTION_ICONS.get(result['action'], result['action'])
        reason_disp = result.get('reason') or reason

        print(f'[{step:>4}] P={cal_prob:.4f} → {action_disp:10s} '
              f'| gt={gt_label}({gt_name}) {match_icon} '
              f'| {reason_disp}{atk_str}')

        # ── Full log entry for paper ────────────────────────────────
        step_log.append({
            'step': step,
            'timestamp': datetime.now(timezone.utc).isoformat(),
            'elapsed_s': elapsed_s,
            'prob': cal_prob, 'raw_prob': float(raw_prob),
            'severity': severity, 'confidence': confidence,
            'prediction': binary,
            'action': result['action'], 'reason': reason_disp,
            'applied': result['applied'], 'attack_state': attack_state,
            'first_attacker_ip': first_attacker['ip'] if first_attacker else None,
            'first_attacker_step': first_attacker['step'] if first_attacker else None,
            'first_detect_latency_s': first_attacker.get('latency_s') if first_attacker else None,
            'second_attacker_ip': second_attacker['ip'] if second_attacker else None,
            'second_attacker_step': second_attacker['step'] if second_attacker else None,
            'second_detect_latency_s': second_attacker.get('latency_s') if second_attacker else None,
            'second_locked_action': second_attacker['action'] if second_attacker else None,
            'victim': result.get('victim'), 'attacker': result.get('attacker'),
            'is_new_attacker': result.get('is_new_attacker', False),
            'port': result.get('port'), 'bps': result.get('bps', 0),
            'gt_label': gt_label, 'gt_name': gt_name,
            'correct': (binary == gt_label),
            'total_cpu_pct': round(total_cpu, 1),
            'controllers_cpu_avg_pct': ctrl_cpu_avg,
            'controllers_overhead_avg_pct': ctrl_overhead_avg,
            'onos1_cpu': onos_cpu.get('onos1', 0),
            'onos2_cpu': onos_cpu.get('onos2', 0),
            'onos3_cpu': onos_cpu.get('onos3', 0),
            'onos1_mem': docker_stats.get('onos1', {}).get('mem', ''),
            'onos2_mem': docker_stats.get('onos2', {}).get('mem', ''),
            'onos3_mem': docker_stats.get('onos3', {}).get('mem', ''),
            'total_sw_packets': total_sw_packets,
            'total_sw_bytes': total_sw_bytes,
            's1_packets': ovs_stats.get('s1', {}).get('packets', 0),
            's2_packets': ovs_stats.get('s2', {}).get('packets', 0),
            's3_packets': ovs_stats.get('s3', {}).get('packets', 0),
            's4_packets': ovs_stats.get('s4', {}).get('packets', 0),
            'active_mitigations_count': len(active_mitigations),
            'unique_attackers_seen': len(attackers_seen),
        })

        step += 1

except KeyboardInterrupt:
    print(f'\n🛑 Interrupted after {step} steps')

# ── Cleanup ──
elapsed = time.time() - t_start
print(f'\n⏱️  Done: {step} steps in {elapsed:.0f}s')
clear_all_mitigations()

# ── Save all logs ──
if step_log:
    df_log = pd.DataFrame(step_log)
    df_log.to_csv(os.path.join(SAVE_DIR, 'step_log.csv'), index=False)
    print(f'💾 step_log.csv ({len(step_log)} rows)')

    total = len(df_log)
    print('\n' + '═' * 60)
    print('  BASELINE SUMMARY')
    print('═' * 60)
    for a in ['ALLOW', 'RATE_LIMIT', 'DROP']:
        n = len(df_log[df_log['action'] == a])
        print(f'  {a:15s}: {n:>4} ({100*n/total:5.1f}%)')

    gt_atk = df_log[df_log['gt_label'] == 1]
    gt_ben = df_log[df_log['gt_label'] == 0]
    tp = len(gt_atk[gt_atk['action'] != 'ALLOW'])
    fn = len(gt_atk[gt_atk['action'] == 'ALLOW'])
    fp = len(gt_ben[gt_ben['action'] != 'ALLOW'])
    tn = len(gt_ben[gt_ben['action'] == 'ALLOW'])
    print(f'  TP={tp}  FN={fn}  FP={fp}  TN={tn}')
    if tp + fp > 0:
        print(f'  Precision: {tp/(tp+fp):.3f}')
    if tp + fn > 0:
        print(f'  Recall:    {tp/(tp+fn):.3f}')
    if tp + fp > 0 and tp + fn > 0:
        prec = tp / (tp + fp)
        rec = tp / (tp + fn)
        if prec + rec > 0:
            print(f'  F1:        {2*prec*rec/(prec+rec):.3f}')
    print(f'  Unique attackers: {len(attackers_seen)}: {attackers_seen}')
    if first_attacker:
        print(f'  1st attacker detection latency: {first_attacker.get("latency_s")}s')
    if second_attacker:
        print(f'  2nd attacker detection latency: {second_attacker.get("latency_s")}s')
        print(f'  2nd attacker locked action: {second_attacker.get("action")}')
    if len(gt_atk) > 0:
        print(f'  Avg CPU (attack):  {gt_atk["total_cpu_pct"].mean():.1f}%')
    if len(gt_ben) > 0:
        print(f'  Avg CPU (benign):  {gt_ben["total_cpu_pct"].mean():.1f}%')
    print('═' * 60)

if discovery_log:
    df_disc = pd.DataFrame(discovery_log)
    df_disc.to_csv(os.path.join(SAVE_DIR, 'discovery_log.csv'), index=False)
    print(f'💾 discovery_log.csv ({len(discovery_log)} events)')

if mitigation_log:
    df_mit = pd.DataFrame(mitigation_log)
    df_mit.to_csv(os.path.join(SAVE_DIR, 'mitigation_log.csv'), index=False)
    print(f'💾 mitigation_log.csv ({len(mitigation_log)} events)')


🔍 Protected ports: ['s1-eth1', 's1-eth2', 's1-eth3', 's2-eth1', 's3-eth1', 's4-eth1']
🧹 All mitigations cleared
╔════════════════════════════════════════════════════════════╗
║   BASELINE — LSTM-Driven + Progressive Mitigation        ║
║   P_RATE=0.55  P_DROP=0.85  P_CLEAR=0.35              ║
║   1st attacker → 5Mbps probe                            ║
║   2nd attacker → baseline action locked for attack      ║
║   Duration: 600s                                   ║
╚════════════════════════════════════════════════════════════╝
[   0] P=0.0044 → ✅ ALLOW    | gt=0(BENIGN) ✅ | allow
[   1] P=0.0019 → ✅ ALLOW    | gt=0(BENIGN) ✅ | allow
[   2] P=0.0041 → ✅ ALLOW    | gt=0(BENIGN) ✅ | allow
[   3] P=0.0027 → ✅ ALLOW    | gt=0(BENIGN) ✅ | allow

🛑 Interrupted after 4 steps

⏱️  Done: 4 steps in 19s
🧹 All mitigations cleared
💾 step_log.csv (4 rows)

════════════════════════════════════════════════════════════
  BASELINE SUMMARY
════════════════════════════════════════════════════════════
  ALL

### 3. LIVE AGENT EVALUATION: Progressive DRL Mitigation
This cell evaluates the pre-trained PPO Agent using **Progressive Mitigation Hysteresis**, acting identically to the static baseline framework but guided by deep reinforcement learning. 

When the LSTM Detector registers elevated attack probabilities, a **5Mbps PROBE rate-limit** is issued to the primary attacker. If the attack persists, the agent dynamically locks onto secondary attackers utilizing continuous state assessments (19-dimensional vectors) to issue appropriate DROP or RATE_LIMIT actions via ONOS REST, minimizing collateral damage and ensuring stability under heavy DDoS load.


In [6]:
# ════════════════════════════════════════════════════════════════════
# DRL (PPO) MITIGATION — Speed-Triggered, Paste AFTER monitoring cell.
# Assumes from notebook: collected_samples, data_lock,
#   model, scaler, imputer, FEATURE_COLS, WINDOW_LEN,
#   TEMPERATURE, BEST_THRESHOLD, N_FEATURES,
#   preprocess_live_sample, calibrate_probability
# ════════════════════════════════════════════════════════════════════
import subprocess, threading, requests, json, re, os, time, pickle
import numpy as np
import pandas as pd
from collections import Counter, defaultdict, deque
from datetime import datetime, timezone
from math import log2
from requests.auth import HTTPBasicAuth
from stable_baselines3 import PPO

# ─── ONOS Cluster Config ────────────────────────────────────────────
ONOS_NODES = [
    {'id': 'onos1', 'ip': '175.24.1.5', 'port': 8181},
    {'id': 'onos2', 'ip': '175.24.1.6', 'port': 8181},
    {'id': 'onos3', 'ip': '175.24.1.7', 'port': 8181},
]
ONOS_AUTH = ('onos', 'rocks')
CONTROLLER_IPS = {'175.24.1.5', '175.24.1.6', '175.24.1.7'}
SWITCHES = ['s1', 's2', 's3', 's4']
SWITCH_DPIDS = [
    'of:0000000000000001', 'of:0000000000000002',
    'of:0000000000000003', 'of:0000000000000004',
]

# ─── DRL Model & Recipe ────────────────────────────────────────────
RECIPE_PATH = 'outputs_ppo_speed/deployment_recipe.json'
with open(RECIPE_PATH) as f:
    recipe = json.load(f)

PPO_MODEL_PATH = recipe['model_path']
OBS_FEATURE_NAMES = recipe['obs_feature_names']
OBS_DIM = recipe['obs_dim']
N_DELTA = recipe['n_delta_features']
ACTION_MAP = {int(k): v for k, v in recipe['action_map'].items()}
STEP_SECONDS = recipe['step_seconds']
FEATURE_COLS_135 = recipe['feature_cols_135']

ppo_agent = PPO.load(PPO_MODEL_PATH)
print(f'✅ PPO agent loaded: {PPO_MODEL_PATH}')
print(f'   Obs dim: {OBS_DIM}, Actions: {ACTION_MAP}')

# Feature index map for building observations
feat_idx = {c: i for i, c in enumerate(FEATURE_COLS_135)}

# ─── Duration & Logging ────────────────────────────────────────────
DRL_DURATION = 600
SAVE_DIR = 'outputs_drl_speed'
os.makedirs(SAVE_DIR, exist_ok=True)

# ─── Runtime State ──────────────────────────────────────────────────
mitigation_lock = threading.Lock()
active_mitigations = {}
cooldown_tracker = {}
attack_state = False
step_log = []
mitigation_log = []
discovery_log = []
PROTECTED_PORTS = set()
attackers_seen = set()

first_attacker = None
second_attacker = None
attack_confirmed_step = None
attack_confirmed_elapsed = None

controller_baseline_sum = defaultdict(float)
controller_baseline_count = defaultdict(int)

# Previous observation for delta computation
prev_obs_base = None


# ────────────────────────────────────────────────────────────────────
#  SYSTEM METRICS
# ────────────────────────────────────────────────────────────────────
def get_docker_cpu_mem():
    stats = {}
    try:
        out = subprocess.run(
            ['docker', 'stats', '--no-stream', '--format',
             '{{.Name}}\t{{.CPUPerc}}\t{{.MemUsage}}'],
            capture_output=True, text=True, timeout=5)
        if out.returncode == 0:
            for line in out.stdout.strip().splitlines():
                parts = line.split('\t')
                if len(parts) >= 3:
                    name = parts[0].strip()
                    cpu_str = parts[1].strip().replace('%', '')
                    mem_str = parts[2].strip().split('/')[0].strip()
                    try:
                        cpu = float(cpu_str)
                    except ValueError:
                        cpu = 0.0
                    stats[name] = {'cpu_pct': cpu, 'mem': mem_str}
    except Exception:
        pass
    return stats


def get_ovs_switch_stats():
    sw_stats = {}
    for sw in SWITCHES:
        try:
            out = subprocess.run(
                ['sudo', 'ovs-ofctl', '-O', 'OpenFlow13', 'dump-aggregate', sw],
                capture_output=True, text=True, timeout=3)
            if out.returncode == 0:
                m_pkt = re.search(r'packet_count=(\d+)', out.stdout)
                m_byt = re.search(r'byte_count=(\d+)', out.stdout)
                sw_stats[sw] = {
                    'packets': int(m_pkt.group(1)) if m_pkt else 0,
                    'bytes': int(m_byt.group(1)) if m_byt else 0,
                }
        except Exception:
            sw_stats[sw] = {'packets': 0, 'bytes': 0}
    return sw_stats


# ────────────────────────────────────────────────────────────────────
#  ONOS HELPERS (same as baseline)
# ────────────────────────────────────────────────────────────────────
def get_master_controller(device_id):
    for node in ONOS_NODES:
        try:
            url = f"http://{node['ip']}:{node['port']}/onos/v1/mastership/{device_id}/master"
            r = requests.get(url, auth=ONOS_AUTH, timeout=2)
            if r.ok:
                master_id = r.json().get('nodeId', '')
                for n2 in ONOS_NODES:
                    if master_id and (master_id in n2['ip'] or master_id == n2['id']):
                        return n2
        except Exception:
            continue
    return ONOS_NODES[0]


def get_all_flows_from_onos():
    flows = []
    for node in ONOS_NODES:
        try:
            url = f"http://{node['ip']}:{node['port']}/onos/v1/flows"
            r = requests.get(url, auth=ONOS_AUTH, timeout=3)
            if r.ok:
                for flow in r.json().get('flows', []):
                    if flow.get('packets', 0) == 0:
                        continue
                    src_ip, dst_ip, proto = None, None, None
                    for c in flow.get('selector', {}).get('criteria', []):
                        if c.get('type') == 'IPV4_SRC':
                            src_ip = c.get('ip', '').split('/')[0]
                        elif c.get('type') == 'IPV4_DST':
                            dst_ip = c.get('ip', '').split('/')[0]
                        elif c.get('type') == 'IP_PROTO':
                            proto = c.get('protocol')
                    if src_ip and dst_ip:
                        flows.append({
                            'src_ip': src_ip, 'dst_ip': dst_ip, 'proto': proto,
                            'packets': flow.get('packets', 0),
                            'bytes': flow.get('bytes', 0),
                            'deviceId': flow.get('deviceId'),
                        })
                return flows
        except Exception:
            continue
    return flows


def discover_protected_ports():
    global PROTECTED_PORTS
    protected = set()
    for node in ONOS_NODES:
        try:
            url = f"http://{node['ip']}:{node['port']}/onos/v1/links"
            r = requests.get(url, auth=ONOS_AUTH, timeout=2)
            if r.ok:
                for link in r.json().get('links', []):
                    for ep in [link.get('src', {}), link.get('dst', {})]:
                        dev = ep.get('device', '')
                        port = ep.get('port', '')
                        try:
                            sw_idx = int(dev.split(':')[-1], 16)
                            protected.add(f's{sw_idx}-eth{port}')
                        except Exception:
                            continue
                break
        except Exception:
            continue
    PROTECTED_PORTS.update(protected)
    print(f'🔍 Protected ports: {sorted(PROTECTED_PORTS)}')


# ────────────────────────────────────────────────────────────────────
#  VICTIM / ATTACKER DISCOVERY (same as baseline)
# ────────────────────────────────────────────────────────────────────
def _shannon_entropy(values):
    if not values or sum(values) == 0:
        return 0.0
    total = sum(values)
    probs = [v / total for v in values if v > 0]
    return -sum(p * log2(p) for p in probs)


def discover_victims_and_attackers():
    flows = get_all_flows_from_onos()
    empty_a = {'src_ip': None, 'packets': 0, 'bytes': 0,
               'flow_count': 0, 'share': 0.0, 'proto': 'TCP'}
    empty_v = {'victim_ip': None, 'victim_packets': 0, 'victim_bytes': 0,
               'victim_flow_count': 0, 'victim_share': 0.0,
               'suspiciousness': 0.0, 'attacker': empty_a.copy(),
               'attackers': []}
    if not flows:
        return empty_v

    dst_stats = defaultdict(lambda: {
        'packets': 0, 'bytes': 0, 'flows': 0, 'syn_flows': 0,
        'src_ips': [], 'devices': Counter(),
    })
    total_packets = sum(f['packets'] for f in flows)

    for f in flows:
        dst = f['dst_ip']
        if not dst:
            continue
        dst_stats[dst]['packets'] += f['packets']
        dst_stats[dst]['bytes'] += f['bytes']
        dst_stats[dst]['flows'] += 1
        dst_stats[dst]['src_ips'].append(f['packets'])
        dst_stats[dst]['devices'][f.get('deviceId')] += 1
        if f.get('proto') in [6, 'TCP'] and f['packets'] <= 5:
            dst_stats[dst]['syn_flows'] += 1

    candidates = []
    for dst_ip, stats in dst_stats.items():
        share = stats['packets'] / max(1, total_packets)
        flow_norm = stats['flows'] / max(1, len(flows))
        syn_share = stats['syn_flows'] / max(1, stats['flows'])
        ent = 1 - (_shannon_entropy(stats['src_ips'])
                   / max(1, log2(len(stats['src_ips']) + 1)))
        susp = 0.4 * share + 0.2 * flow_norm + 0.2 * syn_share + 0.2 * ent
        candidates.append({
            'victim_ip': dst_ip, 'victim_packets': stats['packets'],
            'victim_bytes': stats['bytes'], 'victim_flow_count': stats['flows'],
            'victim_share': share, 'suspiciousness': susp,
        })

    candidates.sort(key=lambda x: x['suspiciousness'], reverse=True)
    top = candidates[0] if candidates else empty_v

    vip = top['victim_ip']
    all_attackers = []
    if vip:
        victim_flows = [f for f in flows if f['dst_ip'] == vip]
        src_stats = defaultdict(lambda: {'packets': 0, 'bytes': 0, 'flows': 0, 'proto': Counter()})
        for f in victim_flows:
            s = f['src_ip']
            if not s:
                continue
            src_stats[s]['packets'] += f['packets']
            src_stats[s]['bytes'] += f['bytes']
            src_stats[s]['flows'] += 1
            src_stats[s]['proto'][f.get('proto', 'TCP')] += 1
        vtotal = sum(st['packets'] for st in src_stats.values())
        for src_ip, st in src_stats.items():
            all_attackers.append({
                'src_ip': src_ip, 'packets': st['packets'], 'bytes': st['bytes'],
                'flow_count': st['flows'],
                'share': st['packets'] / max(1, vtotal),
                'proto': st['proto'].most_common(1)[0][0] if st['proto'] else 'TCP',
            })
        all_attackers.sort(key=lambda x: x['packets'], reverse=True)
        top['attacker'] = all_attackers[0] if all_attackers else empty_a.copy()
    else:
        top['attacker'] = empty_a.copy()
    top['attackers'] = all_attackers
    return top


# ────────────────────────────────────────────────────────────────────
#  MITIGATION FUNCTIONS (same as baseline)
# ────────────────────────────────────────────────────────────────────
def get_host_location(target_ip):
    for node in ONOS_NODES:
        try:
            url = f"http://{node['ip']}:{node['port']}/onos/v1/hosts"
            r = requests.get(url, auth=ONOS_AUTH, timeout=1)
            if r.ok:
                for host in r.json().get('hosts', []):
                    if target_ip in host.get('ipAddresses', []):
                        locs = host.get('locations', [])
                        if locs:
                            return locs[0].get('elementId'), locs[0].get('port')
        except Exception:
            continue
    return None, None


def find_port_for_ip(target_ip):
    device_id, port_num = get_host_location(target_ip)
    if device_id and port_num:
        try:
            sw_idx = int(device_id.split(':')[-1], 16)
            return f's{sw_idx}-eth{port_num}'
        except Exception:
            pass
    fallback = {
        '10.10.0.1': 's1-eth1', '10.10.0.2': 's1-eth2',
        '10.10.0.3': 's1-eth3', '10.10.0.4': 's1-eth4',
        '10.10.0.101': 's3-eth1', '10.10.0.102': 's3-eth2',
    }
    return fallback.get(target_ip)


def install_drop_flow(attacker_ip, victim_ip, timeout=0):
    if not attacker_ip or not victim_ip:
        return False
    if attacker_ip in CONTROLLER_IPS or victim_ip in CONTROLLER_IPS:
        return False
    ok = 0
    with mitigation_lock:
        for dev in SWITCH_DPIDS:
            master = get_master_controller(dev)
            flow = {
                'priority': 65000,
                'isPermanent': (timeout == 0),
                'deviceId': dev,
                'treatment': {'instructions': []},
                'selector': {'criteria': [
                    {'type': 'ETH_TYPE', 'ethType': '0x0800'},
                    {'type': 'IPV4_SRC', 'ip': f'{attacker_ip}/32'},
                    {'type': 'IPV4_DST', 'ip': f'{victim_ip}/32'},
                ]},
            }
            if timeout > 0:
                flow['timeout'] = timeout
                flow['isPermanent'] = False
            try:
                url = f"http://{master['ip']}:{master['port']}/onos/v1/flows"
                requests.post(url, json={'flows': [flow]}, auth=ONOS_AUTH, timeout=2)
                ok += 1
            except Exception:
                continue
    ttl_str = f'{timeout}s' if timeout > 0 else 'PERM'
    print(f'🛡️  [DROP {ttl_str}] {attacker_ip} → {victim_ip} on {ok}/4 switches')
    return ok > 0


def clear_rate_limit(port):
    if not port:
        return
    try:
        subprocess.run(f'sudo ovs-vsctl -- clear Port {port} qos',
                       shell=True, timeout=3, capture_output=True)
        subprocess.run('sudo ovs-vsctl --all destroy QoS',
                       shell=True, timeout=3, capture_output=True)
        subprocess.run('sudo ovs-vsctl --all destroy Queue',
                       shell=True, timeout=3, capture_output=True)
    except Exception:
        pass


def apply_rate_limit(port, bps, duration=600):
    if not port or port in PROTECTED_PORTS:
        return False
    try:
        check = subprocess.run(
            f'sudo ovs-vsctl --if-exists get Port {port} name',
            shell=True, capture_output=True, timeout=3)
        if check.returncode != 0:
            return False
        cmd = (f'sudo ovs-vsctl -- set Port {port} qos=@newqos '
               f'-- --id=@newqos create QoS type=linux-htb '
               f'other-config:max-rate={bps} queues:1=@q1 '
               f'-- --id=@q1 create Queue other-config:max-rate={bps}')
        subprocess.run(cmd, shell=True, timeout=3)
        bps_h = f'{bps/1e6:.1f}Mbps' if bps >= 1e6 else f'{bps/1e3:.0f}Kbps'
        print(f'🚦 [RATE_LIMIT] {port} @ {bps_h}')
        return True
    except Exception:
        return False


def remove_drop_flows(attacker_ip, victim_ip):
    for sw in SWITCHES:
        match = f'ip,nw_src={attacker_ip},nw_dst={victim_ip}'
        try:
            subprocess.run(
                ['sudo', 'ovs-ofctl', '-O', 'OpenFlow13', 'del-flows', sw, match],
                timeout=3, capture_output=True)
        except Exception:
            continue


def clear_all_mitigations():
    global active_mitigations
    for key, info in list(active_mitigations.items()):
        parts = key.split('->')
        if len(parts) == 2:
            a, v = parts
            if info.get('type') == 'DROP':
                remove_drop_flows(a, v)
            elif info.get('type') in ('RATE_LIMIT', 'RATE_PROBE'):
                clear_rate_limit(info.get('port'))
    active_mitigations.clear()
    for sw in SWITCHES:
        try:
            subprocess.run(['sudo', 'ovs-ofctl', '-O', 'OpenFlow13',
                            'del-flows', sw, 'priority=65000'],
                           capture_output=True, timeout=3)
        except Exception:
            pass
    try:
        subprocess.run('sudo ovs-vsctl --all destroy QoS', shell=True, timeout=3, capture_output=True)
        subprocess.run('sudo ovs-vsctl --all destroy Queue', shell=True, timeout=3, capture_output=True)
    except Exception:
        pass
    print('🧹 All mitigations cleared')


def reset_attack_tracking():
    global first_attacker, second_attacker
    global attack_confirmed_step, attack_confirmed_elapsed
    first_attacker = None
    second_attacker = None
    attack_confirmed_step = None
    attack_confirmed_elapsed = None


# ────────────────────────────────────────────────────────────────────
#  DRL OBSERVATION BUILDER
# ────────────────────────────────────────────────────────────────────
def build_ppo_obs(cal_prob, severity, confidence, live_features_scaled):
    """
    Build the 32-dim observation vector matching training.
    29 base features + 3 deltas = 32.
    """
    global prev_obs_base

    row = []
    for f in OBS_FEATURE_NAMES:
        if f == 'attack_probability':
            row.append(float(cal_prob))
        elif f == 'severity_score':
            row.append(float(severity))
        elif f == 'confidence':
            row.append(float(confidence))
        elif f in feat_idx:
            row.append(float(live_features_scaled[feat_idx[f]]))
        else:
            row.append(0.0)

    base = np.array(row, dtype=np.float32)
    base = np.nan_to_num(base, nan=0.0, posinf=5.0, neginf=-5.0)
    base = np.clip(base, -10.0, 10.0)

    # Temporal deltas
    if prev_obs_base is not None:
        delta_prob    = base[0] - prev_obs_base[0]
        delta_packets = base[4] - prev_obs_base[4]
        delta_flows   = base[3] - prev_obs_base[3]
    else:
        delta_prob = delta_packets = delta_flows = 0.0

    deltas = np.clip(np.array([delta_prob, delta_packets, delta_flows], dtype=np.float32), -10.0, 10.0)
    prev_obs_base = base.copy()

    obs = np.concatenate([base, deltas])
    return obs.astype(np.float32)


# ────────────────────────────────────────────────────────────────────
#  DRL MITIGATION EXECUTOR
# ────────────────────────────────────────────────────────────────────
def execute_drl_action(drl_action, step_idx, elapsed_s):
    """
    DRL action: 0=ALLOW, 1=INVESTIGATE, 2=EMERGENCY
    INVESTIGATE → discover + targeted DROP on top attacker
    EMERGENCY   → discover + DROP on ALL top-2 attackers
    """
    global first_attacker, second_attacker, attack_state

    action_name = ACTION_MAP.get(drl_action, 'ALLOW')
    result = {'action': action_name, 'applied': False,
              'victim': None, 'attacker': None, 'port': None,
              'reason': '', 'bps': 0, 'is_new_attacker': False,
              'first_detect_latency_s': None, 'second_detect_latency_s': None}

    if drl_action == 0:  # ALLOW
        result['applied'] = True
        result['reason'] = 'drl_allow'
        return result

    # ── Already have locked second attacker? Keep it ────────────
    if second_attacker is not None:
        sa = second_attacker
        sa_key = sa['key']
        if sa_key not in active_mitigations:
            install_drop_flow(sa['ip'], sa['victim'], timeout=0)
            active_mitigations[sa_key] = {'type': 'DROP', 'ts': time.time()}
        result.update({
            'action': 'DROP', 'applied': True,
            'attacker': sa['ip'], 'victim': sa['victim'],
            'reason': f'locked_drop_active',
            'second_detect_latency_s': sa.get('latency_s'),
        })
        return result

    # ── Discover targets ────────────────────────────────────────
    disc = discover_victims_and_attackers()
    victim_ip = disc['victim_ip']
    if not victim_ip:
        result['reason'] = 'no_victim'
        return result

    all_attackers = disc.get('attackers', [])
    top_attacker = disc['attacker']
    attacker_ip = top_attacker['src_ip']
    if not attacker_ip:
        result['reason'] = 'no_attacker'
        return result

    result['victim'] = victim_ip
    result['attacker'] = attacker_ip

    is_new = attacker_ip not in attackers_seen
    result['is_new_attacker'] = is_new
    if is_new:
        attackers_seen.add(attacker_ip)

    if not attack_state:
        attack_state = True
        print(f'⬆️  DRL triggered — attack state ON')

    # ── First attacker: DROP immediately (no probe delay!) ──────
    if first_attacker is None:
        key = f'{attacker_ip}->{victim_ip}'
        latency = elapsed_s - attack_confirmed_elapsed if attack_confirmed_elapsed else elapsed_s

        if drl_action == 2:  # EMERGENCY → DROP
            applied = install_drop_flow(attacker_ip, victim_ip, timeout=0)
            mit_type = 'DRL_EMERGENCY_DROP'
            if applied:
                active_mitigations[key] = {'type': 'DROP', 'ts': time.time()}
        else:  # INVESTIGATE → rate-limit probe first
            port = find_port_for_ip(attacker_ip)
            applied = apply_rate_limit(port, bps=5_000_000)
            mit_type = 'DRL_INVESTIGATE_RATE_LIMIT'
            if applied:
                active_mitigations[key] = {'type': 'RATE_PROBE', 'port': port, 'ts': time.time()}

        first_attacker = {
            'ip': attacker_ip, 'victim': victim_ip,
            'port': find_port_for_ip(attacker_ip),
            'key': key, 'step': step_idx, 'elapsed_s': elapsed_s,
            'latency_s': latency,
        }
        discovery_log.append({
            'step': step_idx, 'timestamp': datetime.now(timezone.utc).isoformat(),
            'event': 'FIRST_ATTACKER', 'attacker_ip': attacker_ip,
            'victim_ip': victim_ip, 'packets': top_attacker['packets'],
            'share': top_attacker['share'], 'proto': top_attacker['proto'],
            'detection_latency_s': latency, 'drl_action': action_name,
        })
        mitigation_log.append({
            'step': step_idx, 'timestamp': datetime.now(timezone.utc).isoformat(),
            'type': mit_type, 'attacker': attacker_ip,
            'victim': victim_ip, 'port': first_attacker['port'],
            'detection_latency_s': latency,
        })
        print(f'    🆕 1st attacker: {attacker_ip} → {victim_ip} '
              f'(pkts={top_attacker["packets"]}, share={top_attacker["share"]:.2f}) '
              f'[{action_name}]')
        result.update({
            'action': action_name, 'applied': applied,
            'port': first_attacker['port'],
            'reason': f'first_attacker_{action_name.lower()}',
            'first_detect_latency_s': latency,
        })
        return result

    # ── Look for second attacker ────────────────────────────────
    second_ip = None
    second_info = None
    for cand in all_attackers:
        if cand['src_ip'] and cand['src_ip'] != first_attacker['ip']:
            second_ip = cand['src_ip']
            second_info = cand
            break

    if second_ip:
        key = f'{second_ip}->{victim_ip}'
        latency = elapsed_s - attack_confirmed_elapsed if attack_confirmed_elapsed else elapsed_s
        if second_ip not in attackers_seen:
            attackers_seen.add(second_ip)

        # Remove first probe if it was rate-limit
        first_key = first_attacker['key']
        if first_key in active_mitigations:
            fa_info = active_mitigations[first_key]
            if fa_info.get('type') == 'RATE_PROBE':
                clear_rate_limit(fa_info.get('port'))
            active_mitigations.pop(first_key, None)
            print(f'    🔄 Removed probe on first attacker {first_attacker["ip"]}')
            mitigation_log.append({
                'step': step_idx, 'timestamp': datetime.now(timezone.utc).isoformat(),
                'type': 'FIRST_PROBE_REMOVED', 'attacker': first_attacker['ip'],
                'victim': first_attacker['victim'],
            })

        # DROP second attacker (locked)
        applied = install_drop_flow(second_ip, victim_ip, timeout=0)
        if applied:
            active_mitigations[key] = {'type': 'DROP', 'ts': time.time()}

        second_attacker = {
            'ip': second_ip, 'victim': victim_ip, 'key': key,
            'step': step_idx, 'elapsed_s': elapsed_s,
            'latency_s': latency, 'action': 'DROP',
        }
        discovery_log.append({
            'step': step_idx, 'timestamp': datetime.now(timezone.utc).isoformat(),
            'event': 'SECOND_ATTACKER', 'attacker_ip': second_ip,
            'victim_ip': victim_ip, 'packets': second_info['packets'],
            'share': second_info['share'], 'detection_latency_s': latency,
        })
        mitigation_log.append({
            'step': step_idx, 'timestamp': datetime.now(timezone.utc).isoformat(),
            'type': 'SECOND_ATTACKER_DROP_LOCKED', 'attacker': second_ip,
            'victim': victim_ip, 'detection_latency_s': latency,
        })
        print(f'    🆕 2nd attacker: {second_ip} → {victim_ip} '
              f'(pkts={second_info["packets"]}, share={second_info["share"]:.2f}) '
              f'→ LOCKED DROP')
        result.update({
            'action': 'DROP', 'applied': applied,
            'attacker': second_ip, 'victim': victim_ip,
            'reason': 'second_attacker_locked_drop',
            'second_detect_latency_s': latency,
        })
        return result

    # Still only first — keep active
    result.update({
        'action': action_name, 'applied': True,
        'attacker': first_attacker['ip'], 'victim': first_attacker['victim'],
        'reason': 'first_active_searching_second',
    })
    return result


def get_ground_truth():
    try:
        with open('/tmp/label_state', 'r') as f:
            lines = f.readlines()
            if lines:
                parts = lines[-1].strip().split()
                label = int(parts[0])
                if len(parts) > 1 and parts[1].isdigit() and len(parts) > 2:
                    name = parts[2]
                elif len(parts) > 1 and not parts[1].isdigit():
                    name = parts[1]
                else:
                    name = parts[2] if len(parts) > 2 else 'UNKNOWN'
                return label, name
    except Exception:
        pass
    return 0, 'UNKNOWN'


# ════════════════════════════════════════════════════════════════════
#  MAIN LOOP
# ════════════════════════════════════════════════════════════════════
discover_protected_ports()
clear_all_mitigations()

print('╔════════════════════════════════════════════════════════════╗')
print('║   DRL (PPO) — Speed-Triggered Mitigation                 ║')
print(f'║   Model: {PPO_MODEL_PATH:<47s} ║')
print(f'║   Actions: ALLOW / INVESTIGATE / EMERGENCY               ║')
print(f'║   Duration: {DRL_DURATION}s                                   ║')
print('╚════════════════════════════════════════════════════════════╝')

feature_buffer = deque(maxlen=5000)
last_processed = 0
step = 0
t_start = time.time()
prev_attack_state = False
reset_attack_tracking()

try:
    while time.time() - t_start < DRL_DURATION:
        time.sleep(2)

        with data_lock:
            current_samples = list(collected_samples)

        n_available = len(current_samples)
        if n_available <= last_processed:
            continue

        new_samples = current_samples[last_processed:]
        last_processed = n_available

        df_new = pd.DataFrame(new_samples)
        X_new = preprocess_live_sample(df_new)
        for row in X_new:
            feature_buffer.append(row)

        if len(feature_buffer) < WINDOW_LEN:
            print(f'  Warming up ({len(feature_buffer)}/{WINDOW_LEN})')
            step += 1
            continue

        buffer_arr = np.array(list(feature_buffer))
        window = buffer_arr[-WINDOW_LEN:].reshape(1, WINDOW_LEN, N_FEATURES)

        # ── LSTM Prediction ─────────────────────────────────────────
        raw_prob = model.predict(window, verbose=0).ravel()[0]
        cal_prob = float(calibrate_probability(raw_prob, TEMPERATURE))
        binary = int(cal_prob >= BEST_THRESHOLD)
        confidence = abs(cal_prob - 0.5) * 2
        severity = min(1.0, cal_prob * 1.2)
        elapsed_s = round(time.time() - t_start, 1)

        gt_label, gt_name = get_ground_truth()

        # System metrics
        docker_stats = get_docker_cpu_mem()
        ovs_stats = get_ovs_switch_stats()
        total_cpu = sum(v.get('cpu_pct', 0) for v in docker_stats.values())
        onos_cpu = {k: v.get('cpu_pct', 0) for k, v in docker_stats.items()
                    if k.startswith('onos')}
        total_sw_packets = sum(v['packets'] for v in ovs_stats.values())
        total_sw_bytes = sum(v['bytes'] for v in ovs_stats.values())
        onos_overhead = {}
        for cid in ['onos1', 'onos2', 'onos3']:
            cur = float(onos_cpu.get(cid, 0.0))
            if gt_label == 0:
                controller_baseline_sum[cid] += cur
                controller_baseline_count[cid] += 1
            n0 = controller_baseline_count[cid]
            base = (controller_baseline_sum[cid] / n0) if n0 > 0 else cur
            onos_overhead[cid] = cur - base
        ctrl_cpu_avg = float(np.mean([onos_cpu.get(c, 0) for c in ['onos1', 'onos2', 'onos3']]))
        ctrl_overhead_avg = float(np.mean([onos_overhead[c] for c in ['onos1', 'onos2', 'onos3']]))

        # ── Build PPO observation & get action ──────────────────────
        live_scaled = scaler.transform(imputer.transform(
            buffer_arr[-1:].reshape(1, -1))).ravel()
        ppo_obs = build_ppo_obs(cal_prob, severity, confidence, live_scaled)
        drl_action, _ = ppo_agent.predict(ppo_obs, deterministic=True)
        drl_action = int(drl_action)
        action_name = ACTION_MAP.get(drl_action, 'ALLOW')

        # Record attack confirmation for latency calc
        if drl_action > 0 and attack_confirmed_step is None:
            attack_confirmed_step = step
            attack_confirmed_elapsed = elapsed_s

        # ── Execute DRL-driven mitigation ───────────────────────────
        result = execute_drl_action(drl_action, step, elapsed_s)

        # When attack clears (DRL says ALLOW for 5+ consecutive steps)
        if drl_action == 0 and attack_state:
            if not hasattr(execute_drl_action, '_clear_count'):
                execute_drl_action._clear_count = 0
            execute_drl_action._clear_count += 1
            if execute_drl_action._clear_count >= 5:
                attack_state = False
                execute_drl_action._clear_count = 0
                print(f'  ⬇️  DRL: Attack CLEARED (5 consecutive ALLOWs)')
        else:
            execute_drl_action._clear_count = 0

        if prev_attack_state and not attack_state:
            print(f'  🧹 Attack ended at step {step} → clearing all mitigations')
            clear_all_mitigations()
            reset_attack_tracking()
        prev_attack_state = attack_state

        # Pretty print
        ACTION_ICONS = {'ALLOW': '✅ ALLOW', 'INVESTIGATE': '🔍 INVEST', 'EMERGENCY': '🚨 EMERG', 'DROP': '🛡️ DROP'}
        match_icon = '✅' if (binary == gt_label) else '❌'
        atk_str = f' [{result["attacker"]}→{result["victim"]}]' if result.get('attacker') else ''
        action_disp = ACTION_ICONS.get(result['action'], result['action'])
        reason_disp = result.get('reason', action_name)

        print(f'[{step:>4}] P={cal_prob:.4f} → {action_disp:12s} '
              f'| gt={gt_label}({gt_name}) {match_icon} '
              f'| {reason_disp}{atk_str}')

        # ── Full log entry ──────────────────────────────────────────
        step_log.append({
            'step': step,
            'timestamp': datetime.now(timezone.utc).isoformat(),
            'elapsed_s': elapsed_s,
            'prob': cal_prob, 'raw_prob': float(raw_prob),
            'severity': severity, 'confidence': confidence,
            'prediction': binary,
            'drl_action': drl_action, 'drl_action_name': action_name,
            'action': result['action'], 'reason': reason_disp,
            'applied': result['applied'], 'attack_state': attack_state,
            'first_attacker_ip': first_attacker['ip'] if first_attacker else None,
            'first_attacker_step': first_attacker['step'] if first_attacker else None,
            'first_detect_latency_s': first_attacker.get('latency_s') if first_attacker else None,
            'second_attacker_ip': second_attacker['ip'] if second_attacker else None,
            'second_attacker_step': second_attacker['step'] if second_attacker else None,
            'second_detect_latency_s': second_attacker.get('latency_s') if second_attacker else None,
            'second_locked_action': second_attacker['action'] if second_attacker else None,
            'victim': result.get('victim'), 'attacker': result.get('attacker'),
            'is_new_attacker': result.get('is_new_attacker', False),
            'port': result.get('port'), 'bps': result.get('bps', 0),
            'gt_label': gt_label, 'gt_name': gt_name,
            'correct': (binary == gt_label),
            'total_cpu_pct': round(total_cpu, 1),
            'controllers_cpu_avg_pct': ctrl_cpu_avg,
            'controllers_overhead_avg_pct': ctrl_overhead_avg,
            'onos1_cpu': onos_cpu.get('onos1', 0),
            'onos2_cpu': onos_cpu.get('onos2', 0),
            'onos3_cpu': onos_cpu.get('onos3', 0),
            'onos1_mem': docker_stats.get('onos1', {}).get('mem', ''),
            'onos2_mem': docker_stats.get('onos2', {}).get('mem', ''),
            'onos3_mem': docker_stats.get('onos3', {}).get('mem', ''),
            'total_sw_packets': total_sw_packets,
            'total_sw_bytes': total_sw_bytes,
            's1_packets': ovs_stats.get('s1', {}).get('packets', 0),
            's2_packets': ovs_stats.get('s2', {}).get('packets', 0),
            's3_packets': ovs_stats.get('s3', {}).get('packets', 0),
            's4_packets': ovs_stats.get('s4', {}).get('packets', 0),
            'active_mitigations_count': len(active_mitigations),
            'unique_attackers_seen': len(attackers_seen),
        })

        step += 1

except KeyboardInterrupt:
    print(f'\n🛑 Interrupted after {step} steps')

# ── Cleanup ──
elapsed = time.time() - t_start
print(f'\n⏱️  Done: {step} steps in {elapsed:.0f}s')
clear_all_mitigations()

# ── Save all logs ──
if step_log:
    df_log = pd.DataFrame(step_log)
    df_log.to_csv(os.path.join(SAVE_DIR, 'step_log.csv'), index=False)
    print(f'💾 step_log.csv ({len(step_log)} rows)')

    total = len(df_log)
    print('\n' + '═' * 60)
    print('  DRL (PPO) SUMMARY')
    print('═' * 60)
    for a in ['ALLOW', 'INVESTIGATE', 'EMERGENCY', 'DROP']:
        n = len(df_log[df_log['action'] == a])
        if n > 0:
            print(f'  {a:15s}: {n:>4} ({100*n/total:5.1f}%)')

    gt_atk = df_log[df_log['gt_label'] == 1]
    gt_ben = df_log[df_log['gt_label'] == 0]
    tp = len(gt_atk[gt_atk['action'] != 'ALLOW'])
    fn = len(gt_atk[gt_atk['action'] == 'ALLOW'])
    fp = len(gt_ben[gt_ben['action'] != 'ALLOW'])
    tn = len(gt_ben[gt_ben['action'] == 'ALLOW'])
    print(f'  TP={tp}  FN={fn}  FP={fp}  TN={tn}')
    if tp + fp > 0:
        print(f'  Precision: {tp/(tp+fp):.3f}')
    if tp + fn > 0:
        print(f'  Recall:    {tp/(tp+fn):.3f}')
    if tp + fp > 0 and tp + fn > 0:
        prec = tp / (tp + fp)
        rec = tp / (tp + fn)
        if prec + rec > 0:
            print(f'  F1:        {2*prec*rec/(prec+rec):.3f}')
    print(f'  Unique attackers: {len(attackers_seen)}: {attackers_seen}')
    if first_attacker:
        print(f'  1st attacker detection latency: {first_attacker.get("latency_s")}s')
    if second_attacker:
        print(f'  2nd attacker detection latency: {second_attacker.get("latency_s")}s')
    if len(gt_atk) > 0:
        print(f'  Avg CPU (attack):  {gt_atk["total_cpu_pct"].mean():.1f}%')
    if len(gt_ben) > 0:
        print(f'  Avg CPU (benign):  {gt_ben["total_cpu_pct"].mean():.1f}%')
    print('═' * 60)

if discovery_log:
    df_disc = pd.DataFrame(discovery_log)
    df_disc.to_csv(os.path.join(SAVE_DIR, 'discovery_log.csv'), index=False)
    print(f'💾 discovery_log.csv ({len(discovery_log)} events)')

if mitigation_log:
    df_mit = pd.DataFrame(mitigation_log)
    df_mit.to_csv(os.path.join(SAVE_DIR, 'mitigation_log.csv'), index=False)
    print(f'💾 mitigation_log.csv ({len(mitigation_log)} events)')


Gym has been unmaintained since 2022 and does not support NumPy 2.0 amongst other critical functionality.
Please upgrade to Gymnasium, the maintained drop-in replacement of Gym, or contact the authors of your software and request that they upgrade.
Users of this version of Gym should be able to simply replace 'import gym' with 'import gymnasium as gym' in the vast majority of cases.
See the migration guide at https://gymnasium.farama.org/introduction/migration_guide/ for additional information.


✅ PPO agent loaded: ./outputs_ppo_speed/ppo_speed_mitigation_final.zip
   Obs dim: 32, Actions: {0: 'ALLOW', 1: 'INVESTIGATE', 2: 'EMERGENCY'}
🔍 Protected ports: ['s1-eth1', 's1-eth2', 's1-eth3', 's2-eth1', 's3-eth1', 's4-eth1']
🧹 All mitigations cleared
╔════════════════════════════════════════════════════════════╗
║   DRL (PPO) — Speed-Triggered Mitigation                 ║
║   Model: ./outputs_ppo_speed/ppo_speed_mitigation_final.zip ║
║   Actions: ALLOW / INVESTIGATE / EMERGENCY               ║
║   Duration: 600s                                   ║
╚════════════════════════════════════════════════════════════╝


2026-02-28 12:41:24.241239: I external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:473] Loaded cuDNN version 91002


[   0] P=0.0047 → ✅ ALLOW      | gt=0(INIT) ✅ | drl_allow
[   1] P=0.0114 → ✅ ALLOW      | gt=0(INIT) ✅ | drl_allow
[   2] P=0.0045 → ✅ ALLOW      | gt=0(INIT) ✅ | drl_allow
[   3] P=0.0052 → ✅ ALLOW      | gt=0(INIT) ✅ | drl_allow
[   4] P=0.0037 → ✅ ALLOW      | gt=0(INIT) ✅ | drl_allow
[   5] P=0.0033 → ✅ ALLOW      | gt=0(INIT) ✅ | drl_allow
⬆️  DRL triggered — attack state ON
🛡️  [DROP PERM] 10.10.0.3 → 10.10.0.200 on 4/4 switches
    🆕 1st attacker: 10.10.0.3 → 10.10.0.200 (pkts=1901, share=0.77) [EMERGENCY]
[   6] P=0.0055 → 🚨 EMERG      | gt=0(INIT) ✅ | first_attacker_emergency [10.10.0.3→10.10.0.200]
[   7] P=0.0081 → ✅ ALLOW      | gt=0(INIT) ✅ | drl_allow
[   8] P=0.0002 → ✅ ALLOW      | gt=0(INIT) ✅ | drl_allow
[   9] P=0.0001 → ✅ ALLOW      | gt=0(INIT) ✅ | drl_allow
[  10] P=0.0002 → ✅ ALLOW      | gt=0(INIT) ✅ | drl_allow
  ⬇️  DRL: Attack CLEARED (5 consecutive ALLOWs)
  🧹 Attack ended at step 11 → clearing all mitigations
🧹 All mitigations cleared
[  11] P=0.0001 → ✅ A

### 3. LIVE AGENT EVALUATION: Online Continual Learning (PPO)
Unlike the Baseline LSTM which applies static heuristics, this cell deploys the offline-trained **Proximal Policy Optimization (PPO)** agent into a custom `Gymnasium` environment (`LiveDDoSEnv`). 

It actively interacts with the live SDN controller to issue OpenFlow actions. Crucially, the model **learns continuously** (`ppo_model.learn()`) by observing the physical effects of its actions. It receives a **hybrid reward** based on drops in network latency, switch packet rates, and detector probabilities, enforcing strict penalties for dropping benign traffic (Ground Truth SLA violations).


In [ ]:
    # ════════════════════════════════════════════════════════════════════
    # PPO AGENT — ONLINE CONTINUAL LEARNING, Paste AFTER monitoring cell.
    # Assumes from notebook: collected_samples, data_lock,
    #   model, scaler, imputer, FEATURE_COLS, WINDOW_LEN,
    #   TEMPERATURE, BEST_THRESHOLD, N_FEATURES,
    #   preprocess_live_sample, calibrate_probability
    # ════════════════════════════════════════════════════════════════════
    import subprocess, threading, requests, json, re, os, time, pickle
    import numpy as np
    import pandas as pd
    from collections import Counter, defaultdict, deque
    from datetime import datetime, timezone
    from math import log2
    import gymnasium as gym
    from gymnasium import spaces
    from stable_baselines3 import PPO

    # ─── Load PPO Artifacts ─────────────────────────────────────────────
    PPO_MODEL_PATH = 'outputs_ppo_eval/best_model.zip'
    OBS_SCALER_PATH = 'outputs_ppo_eval/obs_scaler_v2.pkl'

    print("loading PPO models for ONLINE CONTINUAL LEARNING...")
    ppo_model = PPO.load(PPO_MODEL_PATH)
    with open(OBS_SCALER_PATH, 'rb') as f:
        obs_scaler = pickle.load(f)

    PPO_OBS_FEATURES = [
        'attack_probability', 'confidence', 'severity_score',  # LSTM stats
        'cluster_syn_ratio', 'tcp_ratio', 'udp_ratio',
        'flow_arrival_rate', 'port_diversity', 'five_unique',
        'avg_pkt_size', 'top_dst_share', 'src_dst_ratio',
        'cluster_tcp_ratio', 'switch_load_std', 'ctrl_load_std',
        'src_entropy', 'dst_entropy', 'total_packets', 'total_flows'
    ]

    # ─── Config & Discovery Hooks ────────────────────────────────────────
    ONOS_NODES = [{'id': 'onos1', 'ip': '175.24.1.5', 'port': 8181},
                {'id': 'onos2', 'ip': '175.24.1.6', 'port': 8181},
                {'id': 'onos3', 'ip': '175.24.1.7', 'port': 8181}]
    ONOS_AUTH = ('onos', 'rocks')
    CONTROLLER_IPS = {'175.24.1.5', '175.24.1.6', '175.24.1.7'}
    SWITCHES = ['s1', 's2', 's3', 's4']
    SWITCH_DPIDS = ['of:0'+str(i) for i in range(1, 5)]

    RATE_LIMIT_BPS       = 100_000   # 100kbps limit for attacking flow 
    MITIGATION_DURATION  = 60
    SAVE_DIR = 'outputs_ppo_online'

    mitigation_lock = threading.Lock()
    active_mitigations = {}
    step_log = []
    PROTECTED_PORTS = set()
    attackers_seen = set()          

    ACTION_MAP = {0: 'ALLOW', 1: 'RATE_LIMIT', 2: 'DROP'}


    # ─── System Metrics ──────────────────────────────────────────────────
    def get_ovs_switch_stats():
        sw_stats = {}
        total_packets = 0
        total_bytes = 0
        for sw in SWITCHES:
            try:
                out = subprocess.run(
                    ['sudo', 'ovs-ofctl', '-O', 'OpenFlow13', 'dump-aggregate', sw],
                    capture_output=True, text=True, timeout=3)
                if out.returncode == 0:
                    m_pkt = re.search(r'packet_count=(\d+)', out.stdout)
                    m_byt = re.search(r'byte_count=(\d+)', out.stdout)
                    pkts = int(m_pkt.group(1)) if m_pkt else 0
                    byts = int(m_byt.group(1)) if m_byt else 0
                    total_packets += pkts
                    total_bytes += byts
                    sw_stats[sw] = {'packets': pkts, 'bytes': byts}
            except Exception: pass
        return total_packets, total_bytes


    # ─── Discovery Helpers ──────────────────────────────────────────────
    def get_master_controller(device_id):
        for node in ONOS_NODES:
            try:
                url = f"http://{node['ip']}:{node['port']}/onos/v1/mastership/{device_id}/master"
                r = requests.get(url, auth=ONOS_AUTH, timeout=2)
                if r.ok:
                    master_id = r.json().get('nodeId', '')
                    for n2 in ONOS_NODES:
                        if master_id and (master_id in n2['ip'] or master_id == n2['id']): return n2
            except Exception: pass
        return ONOS_NODES[0]

    def get_all_flows_from_onos():
        flows = []
        for node in ONOS_NODES:
            try:
                url = f"http://{node['ip']}:{node['port']}/onos/v1/flows"
                r = requests.get(url, auth=ONOS_AUTH, timeout=3)
                if r.ok:
                    for flow in r.json().get('flows', []):
                        if flow.get('packets', 0) == 0: continue
                        src_ip, dst_ip, proto = None, None, None
                        for c in flow.get('selector', {}).get('criteria', []):
                            if c.get('type') == 'IPV4_SRC': src_ip = c.get('ip', '').split('/')[0]
                            elif c.get('type') == 'IPV4_DST': dst_ip = c.get('ip', '').split('/')[0]
                            elif c.get('type') == 'IP_PROTO': proto = c.get('protocol')
                        if src_ip and dst_ip:
                            flows.append({'src_ip': src_ip, 'dst_ip': dst_ip, 'proto': proto,
                                        'packets': flow.get('packets', 0), 'bytes': flow.get('bytes', 0),
                                        'deviceId': flow.get('deviceId')})
                    return flows
            except Exception: pass
        return flows

    def _shannon_entropy(values):
        if not values or sum(values) == 0: return 0.0
        total = sum(values)
        probs = [v / total for v in values if v > 0]
        return -sum(p * log2(p) for p in probs)

    def discover_victims_and_attackers():
        flows = get_all_flows_from_onos()
        empty_a = {'src_ip': None, 'packets': 0, 'bytes': 0}
        empty_v = {'victim_ip': None, 'victim_packets': 0, 'attacker': empty_a.copy(), 'attackers': [], 'suspiciousness': 0.0}
        if not flows: return empty_v

        dst_stats = defaultdict(lambda: {'packets': 0, 'flows': 0, 'syn_flows': 0, 'src_ips': []})
        total_packets = sum(f['packets'] for f in flows)

        for f in flows:
            dst = f['dst_ip']
            if not dst: continue
            dst_stats[dst]['packets'] += f['packets']
            dst_stats[dst]['flows'] += 1
            dst_stats[dst]['src_ips'].append(f['packets'])
            if f.get('proto') in [6, 'TCP'] and f['packets'] <= 5: dst_stats[dst]['syn_flows'] += 1

        cands = []
        for dst_ip, stats in dst_stats.items():
            share = stats['packets'] / max(1, total_packets)
            flow_norm = stats['flows'] / max(1, len(flows))
            syn_share = stats['syn_flows'] / max(1, stats['flows'])
            ent = 1 - (_shannon_entropy(stats['src_ips']) / max(1, log2(len(stats['src_ips']) + 1)))
            susp = 0.4 * share + 0.2 * flow_norm + 0.2 * syn_share + 0.2 * ent
            cands.append({'victim_ip': dst_ip, 'suspiciousness': susp})

        cands.sort(key=lambda x: x['suspiciousness'], reverse=True)
        top = cands[0] if cands else empty_v

        vip = top['victim_ip']
        all_atk = []
        if vip:
            src_stats = defaultdict(lambda: {'packets': 0, 'flows': 0})
            for f in [f for f in flows if f['dst_ip'] == vip]:
                s = f['src_ip']
                if not s: continue
                src_stats[s]['packets'] += f['packets']
                src_stats[s]['flows'] += 1
            for src_ip, st in src_stats.items():
                all_atk.append({'src_ip': src_ip, 'packets': st['packets']})
            all_atk.sort(key=lambda x: x['packets'], reverse=True)
            top['attacker'] = all_atk[0] if all_atk else empty_a.copy()
        else: top['attacker'] = empty_a.copy()
        top['attackers'] = all_atk
        return top


    # ─── Mitigation Exec ────────────────────────────────────────────────
    def get_host_location(target_ip):
        for node in ONOS_NODES:
            try:
                url = f"http://{node['ip']}:{node['port']}/onos/v1/hosts"
                r = requests.get(url, auth=ONOS_AUTH, timeout=1)
                if r.ok:
                    for host in r.json().get('hosts', []):
                        if target_ip in host.get('ipAddresses', []):
                            locs = host.get('locations', [])
                            if locs: return locs[0].get('elementId'), locs[0].get('port')
            except Exception: pass
        return None, None

    def find_port_for_ip(target_ip):
        device_id, port_num = get_host_location(target_ip)
        if device_id and port_num:
            try:
                sw_idx = int(device_id.split(':')[-1], 16)
                return f's{sw_idx}-eth{port_num}'
            except Exception: pass
        fallback = {'10.10.0.1': 's1-eth1', '10.10.0.2': 's1-eth2', '10.10.0.3': 's2-eth4', '10.10.0.4': 's2-eth5'}
        return fallback.get(target_ip)

    def install_drop_flow(attacker_ip, victim_ip, timeout=0):
        if not attacker_ip or not victim_ip or attacker_ip in CONTROLLER_IPS: return False
        ok = 0
        with mitigation_lock:
            for dev in SWITCH_DPIDS:
                master = get_master_controller(dev)
                flow = {
                    'priority': 65000, 'isPermanent': (timeout == 0), 'deviceId': dev,
                    'treatment': {'instructions': []},
                    'selector': {'criteria': [{'type': 'ETH_TYPE', 'ethType': '0x0800'},
                                            {'type': 'IPV4_SRC', 'ip': f'{attacker_ip}/32'},
                                            {'type': 'IPV4_DST', 'ip': f'{victim_ip}/32'}]},
                }
                if timeout > 0: flow['timeout'] = timeout; flow['isPermanent'] = False
                try:
                    url = f"http://{master['ip']}:{master['port']}/onos/v1/flows"
                    requests.post(url, json={'flows': [flow]}, auth=ONOS_AUTH, timeout=2)
                    ok += 1
                except Exception: pass
        print(f'🛡️  [DROP] {attacker_ip} → {victim_ip} on {ok}/4 switches')
        return ok > 0

    def clear_rate_limit(port):
        if not port: return
        try:
            subprocess.run(f'sudo ovs-vsctl -- clear Port {port} qos', shell=True, timeout=3, capture_output=True)
            subprocess.run('sudo ovs-vsctl --all destroy QoS', shell=True, timeout=3, capture_output=True)
            subprocess.run('sudo ovs-vsctl --all destroy Queue', shell=True, timeout=3, capture_output=True)
        except Exception: pass

    def apply_rate_limit(port, bps):
        if not port or port in PROTECTED_PORTS: return False
        try:
            cmd = (f'sudo ovs-vsctl -- set Port {port} qos=@newqos '
                f'-- --id=@newqos create QoS type=linux-htb '
                f'other-config:max-rate={bps} queues:1=@q1 '
                f'-- --id=@q1 create Queue other-config:max-rate={bps}')
            subprocess.run(cmd, shell=True, timeout=3)
            return True
        except Exception: return False

    def remove_drop_flows(attacker_ip, victim_ip):
        for sw in SWITCHES:
            match = f'ip,nw_src={attacker_ip},nw_dst={victim_ip}'
            subprocess.run(['sudo', 'ovs-ofctl', '-O', 'OpenFlow13', 'del-flows', sw, match], timeout=3, capture_output=True)

    def clear_all_mitigations():
        global active_mitigations
        for key, info in list(active_mitigations.items()):
            parts = key.split('->')
            if len(parts) == 2:
                a, v = parts
                if info.get('type') == 'DROP': remove_drop_flows(a, v)
                elif info.get('type') == 'RATE_LIMIT': clear_rate_limit(info.get('port'))
        active_mitigations.clear()
        for sw in SWITCHES:
            subprocess.run(['sudo', 'ovs-ofctl', '-O', 'OpenFlow13', 'del-flows', sw, 'priority=65000'], timeout=3, capture_output=True)
        subprocess.run('sudo ovs-vsctl --all destroy QoS', shell=True, timeout=3, capture_output=True)
        subprocess.run('sudo ovs-vsctl --all destroy Queue', shell=True, timeout=3, capture_output=True)
        print('    🧹 Rules successfully dropped/revoked!')


    # ────────────────────────────────────────────────────────────────────
    #  GYMNASIUM LIVE ENVIRONMENT FOR PPO ONLINE LEARNING
    # ────────────────────────────────────────────────────────────────────
    class LiveDDoSEnv(gym.Env):
        """
        Custom Gym Env that reads from `collected_samples`, runs LSTM, 
        and applies OpenFlow REST rules interactively. 
        It executes inside the `ppo_model.learn()` loop.
        """
        def __init__(self):
            super(LiveDDoSEnv, self).__init__()
            self.action_space = spaces.Discrete(3) # 0:ALLOW, 1:RATE_LIMIT, 2:DROP
            self.observation_space = spaces.Box(low=-10, high=10, shape=(19,), dtype=np.float32)
            
            self.t_start = time.time()
            self.last_processed = 0
            self.step_idx = 0
            self.feature_buffer = deque(maxlen=WINDOW_LEN) # Explicitly cap size 
            
            self.prev_prob = 0.0
            self.prev_confidence = 0.0
            self.prev_severity = 0.0
            self.prev_packets = 0
            self.current_obs = np.zeros(19, dtype=np.float32)

        def _get_ground_truth(self):
            try:
                with open('/tmp/label_state', 'r') as f:
                    lines = f.readlines()
                    if lines:
                        parts = lines[-1].strip().split()
                        return int(parts[0]), parts[2] if len(parts) > 2 else 'UNKNOWN'
            except Exception: pass
            return 0, 'UNKNOWN'

        def _wait_for_next_sample(self):
            """Wait until `collected_samples` has new data from the network"""
            global collected_samples, data_lock
            while True:
                time.sleep(1) # check faster
                with data_lock:
                    current_samples = list(collected_samples)
                if len(current_samples) > self.last_processed:
                    new_samples = current_samples[self.last_processed:]
                    self.last_processed = len(current_samples)
                    return current_samples[-1], new_samples

        def reset(self, seed=None, options=None):
            super().reset(seed=seed)
            print("🔄 Environment resetting for Online Learning...")
            clear_all_mitigations()
            self.feature_buffer.clear()
            
            # Warmup loop: fill buffer to at least WINDOW_LEN
            global collected_samples, data_lock
            with data_lock:
                # Sync to current head
                self.last_processed = len(collected_samples)

            print("  Warming up LSTM window...")
            while len(self.feature_buffer) < WINDOW_LEN:
                latest, new = self._wait_for_next_sample()
                if new:
                    df_new = pd.DataFrame(new)
                    X_new = preprocess_live_sample(df_new)
                    for r in X_new: self.feature_buffer.append(r)

            self.current_obs = self._build_obs(latest)
            pkts, _ = get_ovs_switch_stats()
            self.prev_packets = pkts
            return self.current_obs, {}

        def _build_obs(self, latest_sample):
            buffer_arr = np.array(list(self.feature_buffer))
            # Ensure we always grab the most recent 5 steps
            window = buffer_arr[-WINDOW_LEN:].reshape(1, WINDOW_LEN, N_FEATURES)
            
            import warnings
            with warnings.catch_warnings():
                warnings.simplefilter('ignore')
                raw_prob = model.predict(window, verbose=0).ravel()[0]
                
            cal_prob = float(calibrate_probability(raw_prob, TEMPERATURE))
            self.prev_prob = cal_prob
            self.prev_severity = min(1.0, cal_prob * 1.2)
            self.prev_confidence = abs(cal_prob - 0.5) * 2

            ppo_dict = {'attack_probability': cal_prob, 'confidence': self.prev_confidence, 'severity_score': self.prev_severity}
            for col in PPO_OBS_FEATURES[3:]:
                if col in latest_sample:
                    try: ppo_dict[col] = float(latest_sample[col])
                    except: ppo_dict[col] = 0.0
                else: ppo_dict[col] = 0.0
                    
            arr_obs = np.array([[ppo_dict[col] for col in PPO_OBS_FEATURES]], dtype=np.float32)
            if np.isnan(arr_obs).any(): arr_obs = np.nan_to_num(arr_obs, 0.0)
            return obs_scaler.transform(arr_obs).astype(np.float32).ravel()

        def step(self, action):
            """Execute PPO Action, wait for network physics to happen, get reward"""
            
            # Get Ground Truth AT THE TIME of the decision (not later)
            gt_label_start, gt_name_start = self._get_ground_truth()
            action_name = ACTION_MAP[action]
            result = {'action': action_name, 'applied': False, 'victim': None, 'attacker': None}
            
            # Get pre-action state
            pre_action_pkts = self.prev_packets
            pre_action_prob = self.prev_prob
            pre_action_conf = self.prev_confidence

            # ── EXECUTE OPENFLOW ACTION ── 
            if action_name != 'ALLOW':
                disc = discover_victims_and_attackers()
                attacker_ip = disc['attacker']['src_ip']
                victim_ip = disc['victim_ip']
                
                if attacker_ip and victim_ip:
                    result['victim'], result['attacker'] = victim_ip, attacker_ip
                    if attacker_ip not in attackers_seen: attackers_seen.add(attacker_ip)
                    key = f'{attacker_ip}->{victim_ip}'
                    
                    if action_name == 'DROP':
                        if install_drop_flow(attacker_ip, victim_ip, timeout=MITIGATION_DURATION):
                            active_mitigations[key] = {'type': 'DROP', 'ts': time.time()}
                            result['applied'] = True
                    elif action_name == 'RATE_LIMIT':
                        port = find_port_for_ip(attacker_ip)
                        if apply_rate_limit(port, RATE_LIMIT_BPS):
                            print(f"🚦 [RATE_LIMIT] {attacker_ip} → {victim_ip} ({RATE_LIMIT_BPS//1000}kbps on {port})")
                            active_mitigations[key] = {'type': 'RATE_LIMIT', 'port': port, 'ts': time.time()}
                            result['applied'] = True

            # ── WAIT FOR NETWORK PHYSICS TO TAKE EFFECT ── 
            if action_name != 'ALLOW':
                print(f"    ⏳ [{action_name}] Applied! Enforcing 15-second physical wait for LSTM memory shift...")
                time.sleep(15)  # Longer wait to ensure packets physically drop entirely and cross window len
                
                # Sync the buffer to the latest collected data AFTER sleep!
                global collected_samples, data_lock
                with data_lock:
                    current_samples = list(collected_samples)
                if len(current_samples) > self.last_processed:
                    new_samples = current_samples[self.last_processed:]
                    self.last_processed = len(current_samples)
                    # Fast forward the LSTM buffer
                    df_new = pd.DataFrame(new_samples)
                    X_new = preprocess_live_sample(df_new)
                    for r in X_new: self.feature_buffer.append(r)
                    latest = current_samples[-1]
                else:
                    # If network hangs, grab the next available
                    latest, _ = self._wait_for_next_sample()
            else:
                # If ALLOW, just grab next normal snapshot
                latest, new_samples = self._wait_for_next_sample()
                if new_samples:
                    df_new = pd.DataFrame(new_samples)
                    X_new = preprocess_live_sample(df_new)
                    for r in X_new: self.feature_buffer.append(r)

            # ── UPDATE EXPERT DL METRICS ──
            self.current_obs = self._build_obs(latest)
            post_action_prob = self.prev_prob
            post_action_conf = self.prev_confidence
            
            post_action_pkts, post_action_bytes = get_ovs_switch_stats()
            self.prev_packets = post_action_pkts
            packet_delta = post_action_pkts - pre_action_pkts

            # ── EXPERT DL-DRL ALIGNED REWARD FUNCTION (WITH WAIT) ──
            reward = 0.0
            prob_delta = pre_action_prob - post_action_prob  # Positive = probability dropped

            # 1. Oracle Safety Net (SLA Constraints on Benign Traffic) based on the STATE WHEN WE ACTED
            if gt_label_start == 0:
                if action_name == 'DROP': 
                    if pre_action_prob > 0.5:
                        reward = -1.0     # ❌ Bad: Dropped because LSTM hallucinated (blame LSTM, penalize slightly)
                    else:
                        reward = -5.0     # ❌ FATAL: Dropping legitimate users explicitly
                elif action_name == 'RATE_LIMIT': 
                    reward = -2.0         # ❌ PENALTY: Throttling legitimate users
                else:
                    if pre_action_prob < 0.5:
                        reward = +1.0     # ✅ Excellent: LSTM said safe, agent allowed.
                    else:
                        reward = +0.5     # ⚠️ Agent allowed despite LSTM False Positive (Agent saved the SLA!)
                
            # 2. Under Attack Constraints
            elif gt_label_start == 1:
                if action_name == 'ALLOW':
                    if pre_action_prob >= 0.5:
                        reward = -5.0   # ❌ FATAL: LSTM screamed attack, but agent ALLOWED it!
                    else:
                        reward = -2.0   # ❌ Bad: Agent allowed attack, but LSTM was also unaware.
                else:
                    # ── Agent Applied Mitigation! Evaluate the DL OUTCOME! ──
                    # Check 1: Complete Mitigation (LSTM agrees the threat is gone)
                    if post_action_prob <= 0.40:
                        reward = +5.0   # 🌟 PERFECT: Attack completely mitigated, probability is safe!
                    
                    # Check 2: Strong Positive Momentum
                    elif prob_delta >= 0.15:
                        reward = +3.0   # ✅ Major success: Action caused significant drop in attack prob.
                    
                    # Check 3: Mild Positive Momentum
                    elif prob_delta >= 0.02:
                        reward = +1.5   # ✅ Good success: Prob is going down.
                    
                    # Check 4: Physics changed, but LSTM probability lagging
                    elif packet_delta < -500:
                        if action_name == 'RATE_LIMIT':
                            reward = +1.0 # ✅ Valid Rate Limit: Hardware confirmed drop in traffic volume.
                        else:
                            reward = +0.5 # ✅ Valid Drop: Volume dropped, waiting for LSTM to catch up.
                    
                    # Failure Checks:
                    elif prob_delta < -0.05:
                        reward = -2.0   # ❌ Active Failure: Took action, but attack is surging faster!
                    else:
                        reward = -1.0   # ❌ Ineffective: Wrong IP dropped or rate limit too weak.

            # ── Logging ──
            elapsed_s = round(time.time() - self.t_start, 1)
            ACTION_ICONS = {'ALLOW': '✅ ALLOW', 'RATE_LIMIT': '🚦 RATE', 'DROP': '🛡️ DROP'}
            print(f'[{self.step_idx:>4}] P={pre_action_prob:.4f}→{post_action_prob:.4f} | '
                f'{ACTION_ICONS[result["action"]]:10s} | R={reward:+.1f} | gt={gt_label_start}({gt_name_start}) '
                f'[{result.get("attacker")}→{result.get("victim")}]')

            # ── STATE REVOCATION: Always reset rules AFTER receiving reinforcement reward! ──
            if action_name != 'ALLOW':
                print(f"    🧹 Action evaluated. Revoking mitigations to return to raw state...")
                clear_all_mitigations()

            step_log.append({
                'step': self.step_idx, 'elapsed_s': elapsed_s, 'pre_prob': pre_action_prob, 'post_prob': post_action_prob,
                'packet_delta': packet_delta,
                'action': result['action'], 'reward': reward,
                'applied': result['applied'], 'victim': result['victim'], 'attacker': result['attacker'],
                'gt_label': gt_label_start, 'gt_name': gt_name_start
            })

            self.step_idx += 1
            done = False # Run indefinitely until Ctrl+C
            
            return self.current_obs, reward, done, False, {}

    # ════════════════════════════════════════════════════════════════════
    #  ONLINE LEARNING BOOTSTRAP
    # ════════════════════════════════════════════════════════════════════
    os.makedirs(SAVE_DIR, exist_ok=True)
    print('╔════════════════════════════════════════════════════════════╗')
    print('║   PPO CONTINUAL LEARNING — EXPERT DL ALIGNED REWARDS       ║')
    print(f'║   Duration: INFINITE (Press Stop/Ctrl+C to finish)         ║')
    print('╚════════════════════════════════════════════════════════════╝')

    # Wrap the Live Architecture into the Gym Interface
    live_env = LiveDDoSEnv()

    # Bind the existing pre-trained PPO Agent to the live env
    ppo_model.set_env(live_env)

    # Actively Learn on live network!
    try:
        ppo_model.learn(total_timesteps=999_999, reset_num_timesteps=False)
    except KeyboardInterrupt:
        print("\n🛑 Learning Interrupted by User")

    # Save the updated model!
    updated_model_path = os.path.join(SAVE_DIR, 'ppo_model_online_updated.zip')
    ppo_model.save(updated_model_path)
    print(f"💾 Updated PPO Model saved to: {updated_model_path}")

    clear_all_mitigations()

    if step_log:
        df_log = pd.DataFrame(step_log)
        df_log.to_csv(os.path.join(SAVE_DIR, 'ppo_online_learning_log.csv'), index=False)
        print(f'💾 ppo_online_learning_log.csv saved')
        
        total = len(df_log)
        print('\n═' * 60)
        print('  PPO ONLINE LEARNING SUMMARY')
        print('═' * 60)
        print(f'  Avg Reward/step: {df_log["reward"].mean():.2f}')
        for a in ['ALLOW', 'RATE_LIMIT', 'DROP']:
            n = len(df_log[df_log['action'] == a])
            print(f'  {a:15s}: {n:>4} ({100*n/total:5.1f}%)')


loading PPO models for ONLINE CONTINUAL LEARNING...
╔════════════════════════════════════════════════════════════╗
║   PPO CONTINUAL LEARNING — EXPERT DL ALIGNED REWARDS       ║
║   Duration: INFINITE (Press Stop/Ctrl+C to finish)         ║
╚════════════════════════════════════════════════════════════╝
Wrapping the env with a `Monitor` wrapper
Wrapping the env in a DummyVecEnv.
🔄 Environment resetting for Online Learning...
    🧹 Rules successfully dropped/revoked!
  Warming up LSTM window...
b8cb71e1-7665-4aab-b5f7-d50d1ffd3e53
9401bdc4-0562-454b-9103-ab50c863707d
🚦 [RATE_LIMIT] 10.10.0.3 → 10.10.0.200 (100kbps on s2-eth4)
    ⏳ [RATE_LIMIT] Applied! Enforcing 15-second physical wait for LSTM memory shift...
[   0] P=0.0003→0.0006 | 🚦 RATE     | R=-2.0 | gt=0(UNKNOWN) [10.10.0.3→10.10.0.200]
    🧹 Action evaluated. Revoking mitigations to return to raw state...
    🧹 Rules successfully dropped/revoked!
[   1] P=0.0006→0.0006 | ✅ ALLOW    | R=+1.0 | gt=0(UNKNOWN) [None→None]
[   2] P=0

In [4]:
start_monitoring(label=0)
print(f'Monitoring started. Waiting for initial {WINDOW_LEN} samples...')
time.sleep(WINDOW_LEN * 2 + 2)  # wait for enough samples
show_status()

[RESET] prev_* cleared. reason=start_monitoring
✓ Monitoring started (label=0). Sampling every 2s
Monitoring started. Waiting for initial 5 samples...
MONITORING STATUS (schema v1.3.3)
Active: True
Collected samples (in memory): 5
Training queue size: 5

------------------------------------------------------------
LAST SAMPLE
------------------------------------------------------------
Timestamp: 2026-02-27T07:57:31.925298+00:00
Sample ID: 5 | Label: 0 | Warmup: 0
Total flows: 7 | Total packets: 2721


[Label Sync] Changed to 1
[RESET] prev_* cleared. reason=label_flip 0->1
[Label Sync] Changed to 0
[RESET] prev_* cleared. reason=label_flip 1->0
[Label Sync] Changed to 1
[RESET] prev_* cleared. reason=label_flip 0->1
[Label Sync] Changed to 0
[RESET] prev_* cleared. reason=label_flip 1->0
[Label Sync] Changed to 1
[RESET] prev_* cleared. reason=label_flip 0->1
[Label Sync] Changed to 0
[RESET] prev_* cleared. reason=label_flip 1->0


### 4. LIVE AGENT EVALUATION: DQN Mitigation Test (Uses LSTM Signals)

Runs the trained DQN policy online using the current live stream (`collected_samples`) and prints each decision with:
- LSTM probability
- confidence
- severity
- chosen action (`ALLOW` / `RATE_LIMIT_5MBPS` / `DROP`)
- top candidate attacker/victim (if candidate builder is available).

In [7]:
# ===================================================================
# LIVE DQN MITIGATION TEST (SELF-CONTAINED + ATTACK LOCK)
# Guarantees:
# 1) LSTM probability is computed every loop (fresh sample OR stale reuse)
# 2) Once DROP is applied during attack, it stays locked until label=benign,
#    then all mitigations are cleared.
# ===================================================================

import os
import re
import json
import time
import pickle
import joblib
import threading
import subprocess
from collections import deque, defaultdict
from math import log2

import numpy as np
import pandas as pd
import requests
import tensorflow as tf
from stable_baselines3 import DQN

# -----------------------------
# Helpers
# -----------------------------
def _pick_latest_existing(paths):
    existing = [os.path.abspath(p) for p in paths if p and os.path.isfile(p)]
    return max(existing, key=os.path.getmtime) if existing else None


def _resolve_newtrail_root():
    cwd = os.getcwd()
    candidates = [cwd, os.path.join(cwd, "Models", "newTrail")]
    for c in candidates:
        if os.path.isfile(os.path.join(c, "outputs_detector", "config.json")):
            return os.path.abspath(c)
    return os.path.abspath(cwd)


def _load_obs_scaler(path):
    errs = []
    try:
        return joblib.load(path)
    except Exception as e:
        errs.append(f"joblib.load failed: {e}")
    try:
        with open(path, "rb") as f:
            return pickle.load(f)
    except Exception as e:
        errs.append(f"pickle.load failed: {e}")
    raise RuntimeError(f"Failed to load obs scaler: {path}\n" + "\n".join(errs))


def _parse_numeric(v, default=0.0):
    try:
        if isinstance(v, str):
            m = re.search(r"[-+]?\d*\.?\d+", v)
            if m:
                return float(m.group())
            return float(default)
        return float(v)
    except Exception:
        return float(default)


# -----------------------------
# Paths
# -----------------------------
ROOT = _resolve_newtrail_root()
DETECTOR_DIR = os.path.join(ROOT, "outputs_detector")
DQN_DIR_CANDIDATES = [
    os.path.join(ROOT, "outputs_dqn_offline"),
    os.path.join(os.getcwd(), "outputs_dqn_offline"),
    os.path.join(os.getcwd(), "Models", "newTrail", "outputs_dqn_offline"),
]
DQN_DIRS = [d for d in dict.fromkeys(os.path.abspath(x) for x in DQN_DIR_CANDIDATES) if os.path.isdir(d)]

if not DQN_DIRS:
    raise FileNotFoundError("outputs_dqn_offline directory not found.")

# -----------------------------
# Detector config/model (load if needed)
# -----------------------------
detector_cfg = {}
cfg_path = os.path.join(DETECTOR_DIR, "config.json")
if os.path.isfile(cfg_path):
    with open(cfg_path, "r") as f:
        detector_cfg = json.load(f)

if "FEATURE_COLS" not in globals():
    FEATURE_COLS = detector_cfg.get("feature_cols")
if "BEST_THRESHOLD" not in globals():
    BEST_THRESHOLD = float(detector_cfg.get("BEST_THRESHOLD", 0.5))
if "WINDOW_LEN" not in globals():
    WINDOW_LEN = int(detector_cfg.get("BEST_WINDOW_LEN", 10))
if "N_FEATURES" not in globals():
    N_FEATURES = int(detector_cfg.get("n_features", len(FEATURE_COLS) if FEATURE_COLS else 0))
if "TEMPERATURE" not in globals():
    TEMPERATURE = float(detector_cfg.get("TEMPERATURE", 1.0))

if FEATURE_COLS is None or len(FEATURE_COLS) == 0:
    raise RuntimeError("FEATURE_COLS missing. Run detector init cells or provide outputs_detector/config.json.")

if "model" not in globals():
    detector_model_path = _pick_latest_existing([
        os.path.join(DETECTOR_DIR, "model_final.keras"),
        os.path.join(ROOT, "lstm_online_trained_v2.keras"),
        os.path.join(ROOT, "lstm_online_trained.keras"),
    ])
    if detector_model_path is None:
        raise FileNotFoundError("Detector model not found.")
    model = tf.keras.models.load_model(detector_model_path, compile=False)
    print(f"Loaded detector model: {detector_model_path}")

if "preprocess_live_sample" not in globals():
    imputer = None
    scaler = None
    imp_path = _pick_latest_existing([os.path.join(DETECTOR_DIR, "imputer.pkl")])
    scl_path = _pick_latest_existing([os.path.join(DETECTOR_DIR, "scaler.pkl")])

    if imp_path:
        with open(imp_path, "rb") as f:
            imputer = pickle.load(f)
    if scl_path:
        with open(scl_path, "rb") as f:
            scaler = pickle.load(f)

    def preprocess_live_sample(raw_df):
        df = raw_df.copy()
        for c in FEATURE_COLS:
            if c not in df.columns:
                df[c] = 0.0
        for c in FEATURE_COLS:
            if df[c].dtype == "object":
                ex = df[c].astype(str).str.extract(r"([-+]?\d*\.?\d+)")[0]
                df[c] = pd.to_numeric(ex, errors="coerce")
            else:
                df[c] = pd.to_numeric(df[c], errors="coerce")
            df[c] = df[c].replace([np.inf, -np.inf], np.nan).fillna(0.0)

        X = df[FEATURE_COLS].values.astype(np.float64)
        if imputer is not None:
            X = imputer.transform(X)
        if scaler is not None:
            X = scaler.transform(X)
        X = np.nan_to_num(X, nan=0.0, posinf=0.0, neginf=0.0)
        return X.astype(np.float32)

    print("Defined preprocess_live_sample() from detector artifacts.")

if "calibrate_probability" not in globals():
    def calibrate_probability(raw_prob, temperature):
        p = float(np.clip(raw_prob, 1e-6, 1 - 1e-6))
        t = max(1e-6, float(temperature))
        logit = np.log(p / (1.0 - p))
        z = logit / t
        return float(1.0 / (1.0 + np.exp(-z)))

    print("Defined calibrate_probability().")

# -----------------------------
# Live stream guard
# -----------------------------
if "data_lock" not in globals() or "collected_samples" not in globals():
    raise RuntimeError("data_lock / collected_samples not found. Start monitoring first.")

# -----------------------------
# Load DQN artifacts
# -----------------------------
recipe_candidates = [os.path.join(d, "online_tuning_recipe_dqn.json") for d in DQN_DIRS]
RECIPE_PATH = _pick_latest_existing(recipe_candidates)
recipe = {}
if RECIPE_PATH:
    with open(RECIPE_PATH, "r") as f:
        recipe = json.load(f)

model_fallbacks = []
for d in DQN_DIRS:
    model_fallbacks.extend([
        os.path.join(d, "dqn_fast_mitigation_offline_final.zip"),
        os.path.join(d, "dqn_attacker_hunt_offline_final.zip"),
    ])
    model_fallbacks.extend([os.path.join(d, x) for x in os.listdir(d) if x.endswith(".zip") and "dqn" in x.lower()])

obs_feature_fallbacks = [os.path.join(d, "obs_feature_list.json") for d in DQN_DIRS]
obs_scaler_fallbacks = [os.path.join(d, "obs_scaler.pkl") for d in DQN_DIRS]

DQN_MODEL_PATH = _pick_latest_existing([recipe.get("final_model_path"), *model_fallbacks])
OBS_FEATURE_PATH = _pick_latest_existing([recipe.get("obs_feature_list_path"), *obs_feature_fallbacks])
OBS_SCALER_PATH = _pick_latest_existing([recipe.get("obs_scaler_path"), *obs_scaler_fallbacks])

if DQN_MODEL_PATH is None:
    raise FileNotFoundError("DQN model .zip not found.")
if OBS_FEATURE_PATH is None:
    raise FileNotFoundError("obs_feature_list.json not found.")
if OBS_SCALER_PATH is None:
    raise FileNotFoundError("obs_scaler.pkl not found.")

print(f"Using recipe: {RECIPE_PATH}")
print(f"Loading DQN model: {DQN_MODEL_PATH}")
print(f"Using obs schema: {OBS_FEATURE_PATH}")
print(f"Using obs scaler: {OBS_SCALER_PATH} | size={os.path.getsize(OBS_SCALER_PATH)} bytes")

dqn_model = DQN.load(DQN_MODEL_PATH)
with open(OBS_FEATURE_PATH, "r") as f:
    DQN_OBS_FEATURES = json.load(f)
dqn_obs_scaler = _load_obs_scaler(OBS_SCALER_PATH)

MODEL_OBS_DIM = int(np.prod(dqn_model.observation_space.shape))
BASE_FEAT_DIM = len(DQN_OBS_FEATURES)
if BASE_FEAT_DIM <= 0:
    raise RuntimeError("DQN obs feature schema is empty.")
if MODEL_OBS_DIM % BASE_FEAT_DIM != 0:
    raise RuntimeError(f"Incompatible obs dims: model={MODEL_OBS_DIM}, features={BASE_FEAT_DIM}")
DQN_LOOKBACK = MODEL_OBS_DIM // BASE_FEAT_DIM

ACTION_MAP = {0: "ALLOW", 1: "RATE_LIMIT_5MBPS", 2: "DROP"}
print(f"DQN obs_dim={MODEL_OBS_DIM}, base_features={BASE_FEAT_DIM}, lookback={DQN_LOOKBACK}")
print(f"DQN actions={dqn_model.action_space.n} -> {ACTION_MAP}")

# -----------------------------
# ONOS helpers
# -----------------------------
ONOS_NODES = globals().get("ONOS_NODES", [
    {"id": "onos1", "ip": "175.24.1.5", "port": 8181},
    {"id": "onos2", "ip": "175.24.1.6", "port": 8181},
    {"id": "onos3", "ip": "175.24.1.7", "port": 8181},
])
ONOS_AUTH = globals().get("ONOS_AUTH", ("onos", "rocks"))
SWITCH_DPIDS = globals().get("SWITCH_DPIDS", [
    "of:0000000000000001",
    "of:0000000000000002",
    "of:0000000000000003",
    "of:0000000000000004",
])
FLOW_PRIORITY = int(globals().get("FLOW_PRIORITY", 65000))
CONTROLLER_IPS = {n["ip"] for n in ONOS_NODES}
mitigation_lock = threading.Lock()

STATIC_EXCLUDE_ATTACKER_IPS = set(CONTROLLER_IPS)
STATIC_EXCLUDE_ATTACKER_IPS.update(set(globals().get("NON_ATTACKER_IPS", [])))
STATIC_EXCLUDE_ATTACKER_IPS.update({"10.10.0.200"})
if "SERVER_IPS" in globals():
    STATIC_EXCLUDE_ATTACKER_IPS.update(set(globals().get("SERVER_IPS", [])))

active_drop_targets = set()   # {(src_ip, dst_ip)}
active_rate_ports = set()     # {port_name}

def get_master_controller(device_id):
    for node in ONOS_NODES:
        try:
            u = f"http://{node['ip']}:{node['port']}/onos/v1/mastership/{device_id}/master"
            r = requests.get(u, auth=ONOS_AUTH, timeout=1.5)
            if r.ok:
                master_id = r.json().get("nodeId", "")
                for n2 in ONOS_NODES:
                    if master_id and (master_id in n2["ip"] or master_id == n2["id"]):
                        return n2
        except Exception:
            pass
    return ONOS_NODES[0]

def _extract_ip_criterion(criteria, ctype):
    for c in criteria:
        if c.get("type") == ctype:
            return str(c.get("ip", "")).split("/")[0]
    return None

def _is_ipv4_drop_rule(flow, attacker_ip=None, victim_ip=None):
    try:
        if int(flow.get("priority", 0)) != FLOW_PRIORITY:
            return False
    except Exception:
        return False

    criteria = flow.get("selector", {}).get("criteria", [])
    has_eth_ipv4 = any(
        c.get("type") == "ETH_TYPE" and str(c.get("ethType", "")).lower() in ("0x0800", "0x800")
        for c in criteria
    )
    if not has_eth_ipv4:
        return False

    src = _extract_ip_criterion(criteria, "IPV4_SRC")
    dst = _extract_ip_criterion(criteria, "IPV4_DST")
    if src is None or dst is None:
        return False
    if attacker_ip and src != attacker_ip:
        return False
    if victim_ip and dst != victim_ip:
        return False
    return True

def _delete_matching_drop_flows(attacker_ip=None, victim_ip=None):
    removed = 0
    seen = set()

    for node in ONOS_NODES:
        try:
            r = requests.get(f"http://{node['ip']}:{node['port']}/onos/v1/flows", auth=ONOS_AUTH, timeout=2)
            if not r.ok:
                continue
            for fl in r.json().get("flows", []):
                if not _is_ipv4_drop_rule(fl, attacker_ip=attacker_ip, victim_ip=victim_ip):
                    continue

                dev = fl.get("deviceId")
                fid = str(fl.get("id", "")).strip()
                if not dev or not fid:
                    continue

                key = (dev, fid)
                if key in seen:
                    continue
                seen.add(key)

                for n2 in ONOS_NODES:
                    try:
                        d = requests.delete(
                            f"http://{n2['ip']}:{n2['port']}/onos/v1/flows/{dev}/{fid}",
                            auth=ONOS_AUTH,
                            timeout=1.5,
                        )
                        if d.status_code in (200, 202, 204):
                            removed += 1
                            break
                    except Exception:
                        pass
        except Exception:
            pass

    return removed

def get_all_flows_from_onos():
    flows = []
    seen = set()
    for node in ONOS_NODES:
        try:
            r = requests.get(f"http://{node['ip']}:{node['port']}/onos/v1/flows", auth=ONOS_AUTH, timeout=2.5)
            if not r.ok:
                continue
            for fl in r.json().get("flows", []):
                pkts = float(fl.get("packets", 0) or 0)
                if pkts <= 0:
                    continue

                src_ip, dst_ip, proto = None, None, None
                criteria = fl.get("selector", {}).get("criteria", [])
                for c in criteria:
                    if c.get("type") == "IPV4_SRC":
                        src_ip = str(c.get("ip", "")).split("/")[0]
                    elif c.get("type") == "IPV4_DST":
                        dst_ip = str(c.get("ip", "")).split("/")[0]
                    elif c.get("type") == "IP_PROTO":
                        proto = c.get("protocol")

                if not src_ip or not dst_ip:
                    continue

                sig = (fl.get("deviceId"), src_ip, dst_ip, proto, int(pkts), int(fl.get("bytes", 0) or 0), fl.get("id"))
                if sig in seen:
                    continue
                seen.add(sig)

                flows.append({
                    "src_ip": src_ip,
                    "dst_ip": dst_ip,
                    "proto": proto,
                    "packets": pkts,
                    "bytes": float(fl.get("bytes", 0) or 0),
                    "life": float(fl.get("life", 1) or 1),
                })
        except Exception:
            pass
    return flows

def _shannon_entropy(values):
    if not values:
        return 0.0
    total = sum(values)
    if total <= 0:
        return 0.0
    probs = [v / total for v in values if v > 0]
    return -sum(p * log2(p) for p in probs)

def _norm_rate(v, cap=5000.0):
    v = max(0.0, float(v))
    return float(np.clip(np.log1p(v) / np.log1p(cap), 0.0, 1.0))

def _norm_ratio(v):
    return float(np.clip(float(v), 0.0, 1.0))

def build_topk_candidates(now_ts=None, top_k=5):
    flows = get_all_flows_from_onos()
    if not flows:
        return []

    dst_stats = defaultdict(lambda: {"packets": 0.0, "src_loads": []})
    total_packets = sum(f["packets"] for f in flows)

    for f in flows:
        dst = f.get("dst_ip")
        if not dst:
            continue
        dst_stats[dst]["packets"] += f["packets"]
        dst_stats[dst]["src_loads"].append(f["packets"])

    victims = []
    for dst_ip, st in dst_stats.items():
        share = st["packets"] / max(1.0, total_packets)
        ent = 1.0 - (_shannon_entropy(st["src_loads"]) / max(1.0, log2(len(st["src_loads"]) + 1)))
        victims.append({"victim_ip": dst_ip, "score": 0.5 * share + 0.5 * ent})
    if not victims:
        return []

    victims.sort(key=lambda x: x["score"], reverse=True)
    victim_ip = victims[0]["victim_ip"]

    src_stats = defaultdict(lambda: {"rate": 0.0, "bytes_per_life": 0.0, "syn_flows": 0, "flows": 0, "packets": 0.0})
    victim_total_packets = 0.0

    for f in flows:
        if f.get("dst_ip") != victim_ip:
            continue
        src = f.get("src_ip")
        if src is None or src == victim_ip or src in STATIC_EXCLUDE_ATTACKER_IPS:
            continue

        life = max(1.0, float(f.get("life", 1)))
        pkts = float(f.get("packets", 0) or 0)
        byts = float(f.get("bytes", 0) or 0)
        proto = f.get("proto")

        src_stats[src]["rate"] += pkts / life
        src_stats[src]["bytes_per_life"] += byts / life
        src_stats[src]["packets"] += pkts
        src_stats[src]["flows"] += 1
        victim_total_packets += pkts

        if proto in (6, "6", "TCP", "tcp") and pkts <= 5:
            src_stats[src]["syn_flows"] += 1

    cands = []
    for src_ip, st in src_stats.items():
        syn_ratio = st["syn_flows"] / max(1, st["flows"])
        dst_share = st["packets"] / max(1.0, victim_total_packets)
        score = (
            0.50 * _norm_rate(st["rate"])
            + 0.25 * _norm_rate(st["bytes_per_life"], cap=2e5)
            + 0.15 * _norm_ratio(syn_ratio)
            + 0.10 * _norm_ratio(dst_share)
        )
        cands.append({
            "src_ip": src_ip,
            "victim_ip": victim_ip,
            "score": float(score),
            "pps": float(st["rate"]),
            "bytes_per_life": float(st["bytes_per_life"]),
            "syn_ratio": float(syn_ratio),
            "dst_share": float(dst_share),
        })

    cands.sort(key=lambda x: x["score"], reverse=True)
    return cands[:top_k]

def install_drop_flow(attacker_ip, victim_ip, timeout_s=45):
    if not attacker_ip or not victim_ip or attacker_ip in CONTROLLER_IPS:
        return False

    ok = 0
    with mitigation_lock:
        for dev in SWITCH_DPIDS:
            master = get_master_controller(dev)
            flow = {
                "priority": FLOW_PRIORITY,
                "isPermanent": False,
                "timeout": int(timeout_s),
                "deviceId": dev,
                "treatment": {"instructions": []},
                "selector": {"criteria": [
                    {"type": "ETH_TYPE", "ethType": "0x0800"},
                    {"type": "IPV4_SRC", "ip": f"{attacker_ip}/32"},
                    {"type": "IPV4_DST", "ip": f"{victim_ip}/32"},
                ]},
            }
            try:
                r = requests.post(
                    f"http://{master['ip']}:{master['port']}/onos/v1/flows",
                    json={"flows": [flow]},
                    auth=ONOS_AUTH,
                    timeout=2.0,
                )
                if r.status_code in (200, 201, 202):
                    ok += 1
            except Exception:
                pass

    success = ok > 0
    if success:
        active_drop_targets.add((attacker_ip, victim_ip))
    return success

def find_port_for_ip(target_ip):
    for node in ONOS_NODES:
        try:
            r = requests.get(f"http://{node['ip']}:{node['port']}/onos/v1/hosts", auth=ONOS_AUTH, timeout=1.5)
            if not r.ok:
                continue
            for host in r.json().get("hosts", []):
                if target_ip in host.get("ipAddresses", []):
                    locs = host.get("locations", [])
                    if not locs:
                        continue
                    dev = locs[0].get("elementId", "")
                    port = str(locs[0].get("port", ""))
                    try:
                        sw = f"s{int(dev.split(':')[-1], 16)}"
                        return f"{sw}-eth{port}"
                    except Exception:
                        return None
        except Exception:
            pass
    return None

def apply_rate_limit(port, bps=5_000_000):
    if not port:
        return False
    try:
        cmd = (
            f"sudo ovs-vsctl -- set Port {port} qos=@newqos "
            f"-- --id=@newqos create QoS type=linux-htb other-config:max-rate={int(bps)} queues:1=@q1 "
            f"-- --id=@q1 create Queue other-config:max-rate={int(bps)}"
        )
        out = subprocess.run(cmd, shell=True, timeout=4, capture_output=True, text=True)
        if out.returncode == 0:
            active_rate_ports.add(port)
            return True
        return False
    except Exception:
        return False

def clear_rate_limit(port):
    if not port:
        return False
    try:
        subprocess.run(f"sudo ovs-vsctl -- clear Port {port} qos", shell=True, timeout=3, capture_output=True)
        subprocess.run("sudo ovs-vsctl --all destroy QoS", shell=True, timeout=3, capture_output=True)
        subprocess.run("sudo ovs-vsctl --all destroy Queue", shell=True, timeout=3, capture_output=True)
        return True
    except Exception:
        return False

def clear_all_mitigations():
    for (src, dst) in list(active_drop_targets):
        _delete_matching_drop_flows(attacker_ip=src, victim_ip=dst)
    active_drop_targets.clear()

    for p in list(active_rate_ports):
        clear_rate_limit(p)
    active_rate_ports.clear()

def get_ground_truth():
    try:
        with open("/tmp/label_state", "r") as f:
            lines = f.readlines()
        if lines:
            parts = lines[-1].strip().split()
            return int(parts[0]), (parts[2] if len(parts) > 2 else "UNKNOWN")
    except Exception:
        pass
    return None, "UNKNOWN"

# -----------------------------
# Runtime config
# -----------------------------
TEST_STEPS = int(globals().get("DQN_TEST_STEPS", 120))
POLL_SLEEP_S = float(globals().get("DQN_POLL_SLEEP_S", 0.5))
APPLY_ACTIONS = bool(globals().get("DQN_APPLY_ACTIONS", True))
RATE_LIMIT_BPS = int(globals().get("DQN_RATE_LIMIT_BPS", 5_000_000))

LOCK_REAPPLY_INTERVAL_S = float(globals().get("DQN_LOCK_REAPPLY_INTERVAL_S", 8.0))
FORCE_DROP_WHILE_ATTACK = True

print(f"TEST_STEPS={TEST_STEPS}, APPLY_ACTIONS={APPLY_ACTIONS}, RATE_LIMIT_BPS={RATE_LIMIT_BPS}")
print(f"FORCE_DROP_WHILE_ATTACK={FORCE_DROP_WHILE_ATTACK}, LOCK_REAPPLY_INTERVAL_S={LOCK_REAPPLY_INTERVAL_S}")

FEATURE_IDX = {c: i for i, c in enumerate(FEATURE_COLS)}

feature_buffer = deque(maxlen=max(5000, WINDOW_LEN + 32))
obs_buffer = deque(maxlen=max(1, DQN_LOOKBACK))
prob_hist = deque(maxlen=32)
conf_hist = deque(maxlen=32)
sev_hist = deque(maxlen=32)

last_sample_sig = None
last_raw_sample = None
last_x = np.zeros(N_FEATURES, dtype=np.float32)

locked_target = None  # {"src_ip": ..., "victim_ip": ...}
locked_since = 0.0
last_lock_reapply = 0.0

def _wait_for_new_sample(timeout_s=8.0):
    global last_sample_sig
    t0 = time.time()
    while time.time() - t0 < timeout_s:
        with data_lock:
            sample = dict(collected_samples[-1]) if collected_samples else None

        if sample is None:
            time.sleep(POLL_SLEEP_S)
            continue

        sig = (sample.get("sample_id"), sample.get("timestamp_unix"), sample.get("timestamp_utc"))
        if sig != last_sample_sig:
            last_sample_sig = sig
            return sample

        time.sleep(POLL_SLEEP_S)
    return None

def _sample_to_x(sample):
    x = preprocess_live_sample(pd.DataFrame([sample]))[0]
    x = np.asarray(x, dtype=np.float32)
    return np.nan_to_num(x, nan=0.0, posinf=0.0, neginf=0.0)

def _lstm_prob_from_buffer():
    global last_x
    if len(feature_buffer) == 0:
        arr = np.tile(last_x, (WINDOW_LEN, 1)).astype(np.float32)
    else:
        arr = np.array(feature_buffer, dtype=np.float32)
        if len(arr) < WINDOW_LEN:
            pad = np.tile(arr[-1], (WINDOW_LEN - len(arr), 1))
            arr = np.vstack([pad, arr])
        else:
            arr = arr[-WINDOW_LEN:]

    window = arr.reshape(1, WINDOW_LEN, N_FEATURES)
    raw_prob = float(model.predict(window, verbose=0).ravel()[0])
    cal_prob = float(calibrate_probability(raw_prob, TEMPERATURE))
    cal_prob = float(np.clip(cal_prob, 1e-6, 1 - 1e-6))
    confidence = float(abs(cal_prob - 0.5) * 2.0)
    severity = float(min(1.0, cal_prob * 1.2))
    return cal_prob, confidence, severity

def _ema(values, span):
    if not values:
        return 0.0
    arr = np.asarray(values, dtype=np.float32)
    alpha = 2.0 / (span + 1.0)
    out = float(arr[0])
    for v in arr[1:]:
        out = alpha * float(v) + (1.0 - alpha) * out
    return out

def _build_single_obs(sample, x_row, cal_prob, confidence, severity):
    prob_hist.append(cal_prob)
    conf_hist.append(confidence)
    sev_hist.append(severity)

    p_ema_3 = _ema(prob_hist, 3)
    p_ema_8 = _ema(prob_hist, 8)
    p_ema_10 = _ema(prob_hist, 10)

    p_prev = prob_hist[-2] if len(prob_hist) >= 2 else prob_hist[-1]
    c_prev = conf_hist[-2] if len(conf_hist) >= 2 else conf_hist[-1]
    s_prev = sev_hist[-2] if len(sev_hist) >= 2 else sev_hist[-1]

    p_delta = float(cal_prob - p_prev)
    c_delta = float(confidence - c_prev)
    s_delta = float(severity - s_prev)
    is_uncertain = float((BEST_THRESHOLD - 0.18) <= cal_prob <= (BEST_THRESHOLD + 0.18) or confidence < 0.93)

    row_vals = []
    for feat in DQN_OBS_FEATURES:
        if feat == "attack_probability":
            val = cal_prob
        elif feat == "confidence":
            val = confidence
        elif feat == "severity_score":
            val = severity
        elif feat == "attack_prob_ema_3":
            val = p_ema_3
        elif feat == "attack_prob_ema_8":
            val = p_ema_8
        elif feat == "attack_prob_ema_10":
            val = p_ema_10
        elif feat == "attack_prob_delta":
            val = p_delta
        elif feat == "confidence_delta":
            val = c_delta
        elif feat == "severity_delta":
            val = s_delta
        elif feat == "is_uncertain":
            val = is_uncertain
        elif feat in sample:
            val = _parse_numeric(sample.get(feat, 0.0), 0.0)
        elif feat in FEATURE_IDX:
            val = float(x_row[FEATURE_IDX[feat]])
        else:
            val = 0.0
        row_vals.append(float(val))

    row_arr = np.asarray(row_vals, dtype=np.float32).reshape(1, -1)
    try:
        row_scaled = dqn_obs_scaler.transform(row_arr).astype(np.float32).ravel()
    except Exception as e:
        raise RuntimeError(f"obs scaler transform failed. row_shape={row_arr.shape}, expected_features={BASE_FEAT_DIM}, err={e}")
    row_scaled = np.nan_to_num(row_scaled, nan=0.0, posinf=0.0, neginf=0.0)
    return np.clip(row_scaled, -10.0, 10.0)

def _stack_obs(row_vec):
    obs_buffer.append(row_vec)
    arr = np.array(obs_buffer, dtype=np.float32)
    if len(arr) < DQN_LOOKBACK:
        pad = np.tile(arr[-1], (DQN_LOOKBACK - len(arr), 1))
        arr = np.vstack([pad, arr])
    else:
        arr = arr[-DQN_LOOKBACK:]
    return np.clip(arr.reshape(-1).astype(np.float32), -10.0, 10.0)

def _apply_action(action_id, target):
    if not APPLY_ACTIONS:
        return "DRY_RUN", True

    if action_id == 0:
        return "ALLOW", True

    if target is None:
        return "NO_TARGET", False

    attacker_ip = target.get("src_ip")
    victim_ip = target.get("victim_ip")

    if action_id == 2:
        ok = install_drop_flow(attacker_ip, victim_ip, timeout_s=45)
        return f"DROP_APPLIED={ok}", ok

    if action_id == 1:
        port = find_port_for_ip(attacker_ip)
        if port:
            ok = apply_rate_limit(port, RATE_LIMIT_BPS)
            return f"RATE_LIMIT_APPLIED={ok}@{port}", ok
        return "RATE_LIMIT_NO_PORT", False

    return "UNKNOWN_ACTION", False

# -----------------------------
# Run
# -----------------------------
print("")
print("==============================================================")
print("DQN LIVE MITIGATION TEST (LOCK UNTIL BENIGN)")
print("==============================================================")
print("Per step: p/conf/sev + req_action + eff_action + lock + target + apply_status")

with data_lock:
    warm_samples = list(collected_samples)

if warm_samples:
    warm_tail = warm_samples[-max(WINDOW_LEN, DQN_LOOKBACK):]
    x_warm = preprocess_live_sample(pd.DataFrame(warm_tail))
    for row in x_warm:
        feature_buffer.append(np.nan_to_num(np.asarray(row, dtype=np.float32), nan=0.0, posinf=0.0, neginf=0.0))
    last_x = np.asarray(x_warm[-1], dtype=np.float32)
    last_raw_sample = dict(warm_tail[-1])

    p0, c0, s0 = _lstm_prob_from_buffer()
    seed_row = _build_single_obs(last_raw_sample, last_x, p0, c0, s0)
    for _ in range(DQN_LOOKBACK):
        obs_buffer.append(seed_row)

print(f"Warm buffers: feature={len(feature_buffer)}, obs={len(obs_buffer)}")

live_log = []

for step in range(TEST_STEPS):
    now = time.time()
    sample = _wait_for_new_sample(timeout_s=max(4.0, 2.5 * float(globals().get("INTERVAL", 2))))
    sample_fresh = sample is not None

    if sample_fresh:
        last_raw_sample = dict(sample)
        x_row = _sample_to_x(sample)
        last_x = x_row
    else:
        # keep predicting even when no fresh sample
        if last_raw_sample is None:
            with data_lock:
                last_raw_sample = dict(collected_samples[-1]) if collected_samples else None
            if last_raw_sample is None:
                print(f"[{step:03d}] no sample available at all...")
                continue
        sample = dict(last_raw_sample)
        x_row = last_x.copy()

    feature_buffer.append(x_row)

    cal_prob, confidence, severity = _lstm_prob_from_buffer()
    single_obs = _build_single_obs(sample, x_row, cal_prob, confidence, severity)
    obs = _stack_obs(single_obs)

    req_action, _ = dqn_model.predict(obs, deterministic=True)
    req_action = int(req_action)
    req_name = ACTION_MAP.get(req_action, f"UNK_{req_action}")

    gt_label, _ = get_ground_truth()

    try:
        cands = build_topk_candidates(time.time(), top_k=5)
        top_target = cands[0] if cands else None
    except Exception:
        top_target = None

    eff_action = req_action
    eff_name = ACTION_MAP.get(eff_action, f"UNK_{eff_action}")
    reason = "model"

    # Hard requested behavior:
    # - if benign: clear all rules and force ALLOW
    # - if attack and locked target exists: force DROP on locked target
    if gt_label == 0:
        if locked_target is not None or len(active_drop_targets) > 0 or len(active_rate_ports) > 0:
            clear_all_mitigations()
        locked_target = None
        locked_since = 0.0
        last_lock_reapply = 0.0
        eff_action = 0
        eff_name = ACTION_MAP[0]
        reason = "benign_clear_all"
    elif gt_label == 1 and FORCE_DROP_WHILE_ATTACK and locked_target is not None:
        eff_action = 2
        eff_name = ACTION_MAP[2]
        reason = "attack_lock_hold"

    # Select target
    if eff_action == 2 and locked_target is not None and gt_label == 1:
        target = {"src_ip": locked_target["src_ip"], "victim_ip": locked_target["victim_ip"]}
    else:
        target = top_target

    # Apply effective action
    if eff_action == 2 and gt_label == 1 and locked_target is not None:
        # periodic reapply while attack ongoing
        if APPLY_ACTIONS and (now - last_lock_reapply) >= LOCK_REAPPLY_INTERVAL_S:
            apply_status, ok = _apply_action(2, target)
            apply_status = f"LOCK_REAPPLY|{apply_status}"
            if ok:
                last_lock_reapply = now
        else:
            apply_status = "LOCK_HOLD"
            ok = True
    else:
        apply_status, ok = _apply_action(eff_action, target)

    # Create lock when first successful DROP happens during attack
    if gt_label == 1 and locked_target is None and eff_action == 2 and ok and isinstance(target, dict):
        locked_target = {"src_ip": target.get("src_ip"), "victim_ip": target.get("victim_ip")}
        locked_since = now
        last_lock_reapply = now

    ts = sample.get("timestamp_utc", sample.get("timestamp_unix", now))
    src = target.get("src_ip") if isinstance(target, dict) else None
    dst = target.get("victim_ip") if isinstance(target, dict) else None
    freshness = "fresh" if sample_fresh else "stale_reuse"
    lock_txt = f"{locked_target['src_ip']}->{locked_target['victim_ip']}" if locked_target else "None"

    print(
        f"[{step:03d}] ({freshness}) ts={ts} | p={cal_prob:.3f} conf={confidence:.3f} sev={severity:.3f} "
        f"| req={req_name} eff={eff_name}({reason}) | lock={lock_txt} "
        f"| target={src}->{dst} | gt={gt_label} | {apply_status}"
    )

    live_log.append({
        "step": step,
        "timestamp": ts,
        "sample_fresh": sample_fresh,
        "attack_probability": float(cal_prob),
        "confidence": float(confidence),
        "severity_score": float(severity),
        "requested_action_id": req_action,
        "requested_action_name": req_name,
        "effective_action_id": eff_action,
        "effective_action_name": eff_name,
        "effective_reason": reason,
        "locked_target": lock_txt,
        "target_attacker_ip": src,
        "target_victim_ip": dst,
        "ground_truth": gt_label,
        "apply_status": apply_status,
    })

save_dir = os.path.join(os.path.dirname(DQN_MODEL_PATH), "outputs_dqn_live_test")
os.makedirs(save_dir, exist_ok=True)
log_path = os.path.join(save_dir, f"dqn_live_test_{int(time.time())}.csv")
pd.DataFrame(live_log).to_csv(log_path, index=False)

print("")
print(f"DQN live test finished. Steps logged: {len(live_log)}")
print(f"Log saved: {log_path}")


Using recipe: /home/shokran/PycharmProjects/JupyterProject/Models/newTrail/outputs_dqn_offline/online_tuning_recipe_dqn.json
Loading DQN model: /home/shokran/PycharmProjects/JupyterProject/Models/newTrail/outputs_dqn_offline/dqn_fast_mitigation_offline_final.zip
Using obs schema: /home/shokran/PycharmProjects/JupyterProject/Models/newTrail/outputs_dqn_offline/obs_feature_list.json
Using obs scaler: /home/shokran/PycharmProjects/JupyterProject/Models/newTrail/outputs_dqn_offline/obs_scaler.pkl | size=1887 bytes
DQN obs_dim=100, base_features=25, lookback=4
DQN actions=3 -> {0: 'ALLOW', 1: 'RATE_LIMIT_5MBPS', 2: 'DROP'}
TEST_STEPS=120, APPLY_ACTIONS=True, RATE_LIMIT_BPS=5000000
FORCE_DROP_WHILE_ATTACK=True, LOCK_REAPPLY_INTERVAL_S=8.0

DQN LIVE MITIGATION TEST (LOCK UNTIL BENIGN)
Per step: p/conf/sev + req_action + eff_action + lock + target + apply_status
Warm buffers: feature=5, obs=4
[000] (fresh) ts=2026-02-27T08:03:12.548966+00:00 | p=0.001 conf=0.998 sev=0.001 | req=ALLOW eff=ALLOW

In [7]:
stop_monitoring()

✓ Monitoring stopped. 255 samples saved to: Online_test.csv


In [2]:
"""
Full SDN Controller-Level Monitoring (LSTM → DRL ready, updated v1.4.0)
CHANGES from v1.3.3:
- ADDED per-source-IP features (Approach 1) for DRL attacker identification
- Top-6 sources ranked by packet count with 10 features each (60 new cols)
- 3 global source diversity metrics (active_sources, source_entropy, top_source_dominance)
- All existing 162 columns UNCHANGED — LSTM works without modification
"""

import requests, time, pandas as pd, numpy as np, subprocess, re, json, threading, hashlib, math, os, queue
from requests.auth import HTTPBasicAuth
from collections import defaultdict, deque
from datetime import datetime, timezone
from math import log2

# ===== Config (edit) =====
CONTROLLERS = [
    {"id": "onos1", "ip": "175.24.1.5", "port": 8181},
    {"id": "onos2", "ip": "175.24.1.6", "port": 8181},
    {"id": "onos3", "ip": "175.24.1.7", "port": 8181},
]
SWITCH_TO_CONTROLLER = {"s1":"175.24.1.7","s2":"175.24.1.6","s3":"175.24.1.6","s4":"175.24.1.5"}
AUTH = HTTPBasicAuth("onos", "rocks")
INTERVAL = 2
SWITCHES = ["s1","s2","s3","s4"]
CSV_PATH = "DRL_test.csv"
PARQUET_PATH = "DRL_test.parquet"
PARQUET_SYNC_EVERY = 100

SCHEMA_VERSION = "1.5.0"

# ---- Smoothing / evaluation knobs ----
REST_LAT_WINDOW = 2
WARMUP_SAMPLES = 0

# ---- Per-source-IP features (Approach 1) ----
TOP_K_SOURCES = 6   # track top-6 sources by packet count

# Host-to-IP mapping (must match auto_attack_cycle.sh)
HOST_TO_IP = {"h2": "10.10.0.2", "h4": "10.10.0.4", "iot2": "10.10.0.102"}

# ===== Globals =====
monitoring_active = False
monitoring_thread = None
prev_stats = {}
prev_metrics = {}
prev_flows_set = set()
collected_samples = []
parquet_pending_counter = 0
training_queue = queue.Queue()
current_label = 0
current_attacker_ip = "none"
current_attacker_host = "none"
flow_database = {}
attack_markers = []
monotonic_start = time.monotonic()

rest_lat_hist = defaultdict(lambda: deque(maxlen=REST_LAT_WINDOW))

data_lock = threading.Lock()

# ===== Helpers =====
def now_utc_iso():
    return datetime.now(timezone.utc).strftime("%Y-%m-%dT%H:%M:%S.%fZ")

def query(ctrl, endpoint, timeout=3.0):
    url = f"http://{ctrl['ip']}:{ctrl['port']}/onos/v1{endpoint}"
    t0 = time.time()
    try:
        r = requests.get(url, auth=AUTH, timeout=timeout)
        lat_ms = (time.time() - t0) * 1000
        rest_lat_hist[(ctrl['id'], endpoint)].append(lat_ms)
        return (r.json() if r.status_code == 200 else None, lat_ms)
    except Exception:
        rest_lat_hist[(ctrl['id'], endpoint)].append(-1)
        return (None, -1)

def safe_rate(current, previous, interval):
    if current is None or previous is None: return 0.0
    if pd.isna(current) or pd.isna(previous): return 0.0
    try:
        curr_val = float(current)
        prev_val = float(previous)
    except (ValueError, TypeError):
        return 0.0
    delta = curr_val - prev_val
    if delta < 0: return 0.0
    safe_interval = max(1, interval)
    rate = delta / safe_interval
    if rate > 1e9: return 0.0
    return rate

def reset_prev_state(reason=""):
    global prev_stats, prev_metrics, prev_flows_set
    prev_stats.clear()
    prev_metrics.clear()
    prev_flows_set.clear()
    rest_lat_hist.clear()
    attack_markers.append({
        "ts_utc": now_utc_iso(),
        "t_monotonic": time.monotonic() - monotonic_start,
        "event": "reset_prev_state",
        "reason": reason
    })
    print(f"[RESET] prev_* cleared. reason={reason}")

def get_docker_stats():
    try:
        p = subprocess.run(
            ['docker','stats','--no-stream','--format','{{.Name}}\t{{.CPUPerc}}\t{{.MemUsage}}'],
            capture_output=True, text=True, timeout=3
        )
        out = {}
        for line in p.stdout.strip().splitlines():
            parts = line.split('\t')
            if len(parts) >= 3:
                name = parts[0]
                try: cpu = float(parts[1].replace('%','').strip())
                except: cpu = 0.0
                mem = parts[2].split('/')[0].strip()
                out[name] = {"cpu": cpu, "mem": mem}
        return out
    except Exception:
        return {}

def get_hosts_mac_map():
    mac_map = {}
    for ctrl in CONTROLLERS:
        hosts_data, _ = query(ctrl, "/hosts")
        if hosts_data:
            for h in hosts_data.get("hosts", []):
                mac = (h.get("mac") or "").lower()
                ips = h.get("ipAddresses") or []
                if mac and ips:
                    mac_map[mac] = {"ip": ips[0]}
    return mac_map

def dump_switch_flows(sw):
    try:
        p = subprocess.run(['sudo','ovs-ofctl','-O','OpenFlow13','dump-flows',sw],
                           capture_output=True, text=True, timeout=5)
        return p.stdout
    except Exception:
        return ""

def generate_flow_id(switch, src_ip, dst_ip, src_port, dst_port, protocol):
    key = f"{switch}|{src_ip}|{dst_ip}|{src_port}|{dst_port}|{protocol}"
    return hashlib.md5(key.encode()).hexdigest()[:16]

def shannon_entropy(values):
    if not values: return 0.0
    total = sum(values)
    if total <= 0: return 0.0
    probs = [v/total for v in values if v > 0]
    return -sum(p*log2(p) for p in probs)

def parse_flows_enhanced(sw, text, mac_map):
    global flow_database
    flows = []
    if not text: return flows
    for line in text.strip().splitlines():
        if "n_packets=" not in line or "actions=" not in line: continue
        m_pkts = re.search(r'n_packets=(\d+)', line)
        if not m_pkts or int(m_pkts.group(1)) == 0: continue

        m_bytes = re.search(r'n_bytes=(\d+)', line)
        m_dur   = re.search(r'duration=(\d+(?:\.\d+)?)s', line)
        m_prio  = re.search(r'priority=(\d+)', line)
        m_cookie= re.search(r'cookie=(0x[0-9a-f]+)', line)

        m_srcmac= re.search(r'dl_src=([0-9a-f:]{17})', line, re.I)
        m_dstmac= re.search(r'dl_dst=([0-9a-f:]{17})', line, re.I)
        m_srcip = re.search(r'nw_src=([0-9.]+)', line)
        m_dstip = re.search(r'nw_dst=([0-9.]+)', line)

        m_proto = re.search(r'nw_proto=(\d+)', line)
        m_proto_token = re.search(r'\b(tcp|udp|icmp)\b', line, re.IGNORECASE)
        m_sport = re.search(r'tp_src=(\d+)', line)
        m_dport = re.search(r'tp_dst=(\d+)', line)

        m_tflags= re.search(r'tcp_flags=([^,\s]+)', line)
        m_itype = re.search(r'icmp_type=(\d+)', line)
        m_icode = re.search(r'icmp_code=(\d+)', line)
        m_inp   = re.search(r'in_port=(?:"([^"]+)|(\d+))', line)
        m_act   = re.search(r'actions=([^\s]+)', line)

        src_ip = m_srcip.group(1) if m_srcip else None
        dst_ip = m_dstip.group(1) if m_dstip else None
        if not src_ip and m_srcmac: src_ip = (mac_map.get(m_srcmac.group(1).lower()) or {}).get("ip")
        if not dst_ip and m_dstmac: dst_ip = (mac_map.get(m_dstmac.group(1).lower()) or {}).get("ip")
        if not (src_ip and dst_ip): continue

        proto_num = int(m_proto.group(1)) if m_proto else None
        if proto_num is not None:
            proto_map = {1: 'ICMP', 6: 'TCP', 17: 'UDP'}
            proto_name = proto_map.get(proto_num, f"other_{proto_num}")
        elif m_proto_token:
            token = m_proto_token.group(1).lower()
            token_map = {'tcp': ('TCP', 6), 'udp': ('UDP', 17), 'icmp': ('ICMP', 1)}
            proto_name, proto_num = token_map.get(token, (token.upper(), 0))
        else:
            proto_num = 0
            proto_name = 'OTHER'

        src_port  = int(m_sport.group(1)) if m_sport else 0
        dst_port  = int(m_dport.group(1)) if m_dport else 0

        fid = generate_flow_id(sw, src_ip, dst_ip, src_port, dst_port, proto_name)
        pkt = int(m_pkts.group(1))
        byt = int(m_bytes.group(1)) if m_bytes else 0
        dur = float(m_dur.group(1)) if m_dur else 0.0

        flow = {
            "flow_id": fid, "switch": sw,
            "cookie": m_cookie.group(1) if m_cookie else None,
            "priority": int(m_prio.group(1)) if m_prio else 0,
            "src_ip": src_ip, "dst_ip": dst_ip,
            "src_port": src_port, "dst_port": dst_port,
            "protocol": proto_name, "proto_num": proto_num,
            "packets": pkt, "bytes": byt, "duration": dur,
            "tcp_flags": m_tflags.group(1) if m_tflags else None,
            "icmp_type": int(m_itype.group(1)) if m_itype else None,
            "icmp_code": int(m_icode.group(1)) if m_icode else None,
            "in_port": (m_inp.group(1) if m_inp and m_inp.group(1) else
                        (int(m_inp.group(2)) if m_inp and m_inp.group(2) else None)),
            "actions": m_act.group(1) if m_act else None,
            "master_controller": SWITCH_TO_CONTROLLER.get(sw,"unknown"),
        }
        flows.append(flow)

        flow_database[fid] = {
            "switch": sw, "src_ip": src_ip, "dst_ip": dst_ip,
            "src_port": src_port, "dst_port": dst_port, "protocol": proto_name,
            "cookie": flow["cookie"], "priority": flow["priority"],
            "last_seen": now_utc_iso(), "total_packets": pkt, "total_bytes": byt
        }

    return flows

def get_onos_metrics_best_effort(ctrl):
    out = {"pktin_total": 0.0, "pktout_total": 0.0, "flowmiss_total": np.nan,
           "pktin_rate_direct": 0.0, "pktout_rate_direct": 0.0,
           "event_queue": np.nan, "atomix_nodes": np.nan, "gc_pause_ms": np.nan}

    mjson, _ = query(ctrl, "/metrics")
    if not mjson or not isinstance(mjson.get("metrics"), list):
        return out

    pktin_counters = []
    pktout_counters = []
    pktin_rates = []
    pktout_rates = []

    for m in mjson["metrics"]:
        name = (m.get("name") or "").lower()

        fval_count = None
        fval_mean_rate = None
        metric_obj = m.get('metric') if isinstance(m, dict) else None
        if isinstance(metric_obj, dict):
            meter = metric_obj.get('meter')
            if isinstance(meter, dict):
                fval_count = meter.get('counter')
                fval_mean_rate = meter.get('mean_rate')
            if fval_count is None and 'counter' in metric_obj and isinstance(metric_obj.get('counter'), dict):
                fval_count = metric_obj.get('counter').get('counter')

        if fval_count is None:
            if 'value' in m and m.get('value') is not None:
                try: fval_count = float(m.get('value'))
                except: fval_count = None
            elif 'measurements' in m and isinstance(m.get('measurements'), list) and m.get('measurements'):
                first = m.get('measurements')[0]
                if isinstance(first, dict):
                    fval_count = first.get('value') or first.get('count')
                else:
                    fval_count = first

        try:
            if fval_count is not None: fval_count = float(fval_count)
        except: fval_count = None
        try:
            if fval_mean_rate is not None: fval_mean_rate = float(fval_mean_rate)
        except: fval_mean_rate = None

        if re.search(r'of:[0-9a-f]+\.packet[_\.-]?in\.count', name):
            if fval_count is not None and fval_count > 0:
                pktin_counters.append(fval_count)

        if re.search(r'of:[0-9a-f]+\.packet[_\.-]?in\.rate', name):
            if fval_mean_rate is not None and fval_mean_rate > 0:
                pktin_rates.append(fval_mean_rate)

        if re.search(r'of:[0-9a-f]+\.packet[_\.-]?out\.count', name):
            if fval_count is not None and fval_count > 0:
                pktout_counters.append(fval_count)

        if re.search(r'of:[0-9a-f]+\.packet[_\.-]?out\.rate', name):
            if fval_mean_rate is not None and fval_mean_rate > 0:
                pktout_rates.append(fval_mean_rate)

        if re.search(r'flow[_\.-]?miss|miss[_\.-]?flow|flowmiss', name) and not re.search(r'rate', name):
            if fval_count is not None:
                out['flowmiss_total'] = fval_count

        if re.search(r'event[_\.-]?queue|queue[_\.-]?size', name):
            if fval_count is not None:
                out['event_queue'] = fval_count

        if 'atomix' in name and re.search(r'pending|backlog|queue', name):
            if fval_count is not None:
                out['atomix_nodes'] = fval_count

    if pktin_counters:
        out['pktin_total'] = sum(pktin_counters)
    if pktin_rates:
        out['pktin_rate_direct'] = sum(pktin_rates)
    if pktout_counters:
        out['pktout_total'] = sum(pktout_counters)
    if pktout_rates:
        out['pktout_rate_direct'] = sum(pktout_rates)

    fjson, _ = query(ctrl, "/flows")
    if fjson and isinstance(fjson.get("flows"), list):
        miss = sum(1 for f in fjson["flows"] if int(f.get("packets", 0)) == 0)
        if np.isnan(out.get("flowmiss_total", np.nan)):
            out["flowmiss_total"] = miss

    cjson, _ = query(ctrl, "/cluster")
    if cjson and isinstance(cjson.get("nodes"), list):
        out["atomix_nodes"] = sum(1 for n in cjson["nodes"] if n.get("status") == "READY")

    try:
        p = subprocess.run(['docker','logs','--tail','50', ctrl['id']],
                           capture_output=True, text=True, timeout=1.0)
        for line in (p.stderr or "").splitlines():
            if 'GC' in line and 'pause' in line.lower():
                m_gc = re.search(r'(\d+\.?\d*)\s*ms', line)
                if m_gc: out["gc_pause_ms"] = float(m_gc.group(1)); break
    except Exception:
        pass

    return out

def get_onos_flows_with_protocol(ctrl):
    proto_counts = {"tcp": 0, "udp": 0, "icmp": 0, "other": 0, "syn_flows": 0}
    fjson, _ = query(ctrl, "/flows")
    if not fjson or not isinstance(fjson.get("flows"), list):
        return proto_counts

    for flow in fjson["flows"]:
        packets = flow.get("packets", 0)
        if packets == 0:
            continue

        criteria = flow.get("selector", {}).get("criteria", [])
        proto_found = False
        is_tcp = False

        for c in criteria:
            if c.get("type") == "IP_PROTO":
                proto_num = c.get("protocol")
                if proto_num == 6:
                    proto_counts["tcp"] += 1
                    proto_found = True
                    is_tcp = True
                elif proto_num == 17:
                    proto_counts["udp"] += 1
                    proto_found = True
                elif proto_num == 1:
                    proto_counts["icmp"] += 1
                    proto_found = True
                break

        if is_tcp and packets <= 5:
            proto_counts["syn_flows"] += 1

        if not proto_found:
            for c in criteria:
                if c.get("type") == "ETH_TYPE" and c.get("ethType") in ["0x800", "0x0800"]:
                    proto_counts["other"] += 1
                    break

    return proto_counts


# ────────────────────────────────────────────────────────────────────
#  PER-SOURCE-IP FEATURES (Approach 1 — NEW in v1.4.0)
# ────────────────────────────────────────────────────────────────────
def compute_per_source_features(all_switch_flows):
    """
    Group flows by src_ip → compute per-source stats for top-K sources.
    Returns dict with fixed columns: src1_ip, src1_packets, ..., src6_syn_ratio, etc.
    Produces 10 features × TOP_K_SOURCES + 3 global = 63 new columns.
    """
    if not all_switch_flows:
        out = {}
        for i in range(1, TOP_K_SOURCES + 1):
            out[f'src{i}_ip'] = 'none'
            out[f'src{i}_packets'] = 0
            out[f'src{i}_bytes'] = 0
            out[f'src{i}_flows'] = 0
            out[f'src{i}_pps'] = 0.0
            out[f'src{i}_syn_ratio'] = 0.0
            out[f'src{i}_dst_diversity'] = 0
            out[f'src{i}_top_dst_share'] = 0.0
            out[f'src{i}_avg_duration'] = 0.0
            out[f'src{i}_pkt_share'] = 0.0
        out['active_sources'] = 0
        out['source_entropy'] = 0.0
        out['top_source_dominance'] = 0.0
        return out

    src_stats = defaultdict(lambda: {
        'packets': 0, 'bytes': 0, 'flows': 0, 'syn_flows': 0,
        'dsts': set(), 'dst_pkts': defaultdict(int), 'durations': [],
    })

    total_packets = 0
    for f in all_switch_flows:
        src = f.get('src_ip')
        if not src:
            continue
        s = src_stats[src]
        pkts = f.get('packets', 0)
        s['packets'] += pkts
        s['bytes'] += f.get('bytes', 0)
        s['flows'] += 1
        s['dsts'].add(f.get('dst_ip', ''))
        s['dst_pkts'][f.get('dst_ip', '')] += pkts
        dur = f.get('duration', 0.0)
        s['durations'].append(dur)
        total_packets += pkts
        # SYN heuristic: TCP flow with very few packets
        proto = f.get('protocol', '')
        if proto == 'TCP' and pkts <= 5:
            s['syn_flows'] += 1

    # Rank by packets descending
    ranked = sorted(src_stats.items(), key=lambda kv: kv[1]['packets'], reverse=True)

    out = {}
    for i in range(1, TOP_K_SOURCES + 1):
        if i <= len(ranked):
            src_ip, st = ranked[i - 1]
            top_dst_pkts = max(st['dst_pkts'].values()) if st['dst_pkts'] else 0
            out[f'src{i}_ip'] = src_ip
            out[f'src{i}_packets'] = st['packets']
            out[f'src{i}_bytes'] = st['bytes']
            out[f'src{i}_flows'] = st['flows']
            out[f'src{i}_pps'] = st['packets'] / max(1, INTERVAL)
            out[f'src{i}_syn_ratio'] = st['syn_flows'] / max(1, st['flows'])
            out[f'src{i}_dst_diversity'] = len(st['dsts'])
            out[f'src{i}_top_dst_share'] = top_dst_pkts / max(1, st['packets'])
            out[f'src{i}_avg_duration'] = float(np.mean(st['durations'])) if st['durations'] else 0.0
            out[f'src{i}_pkt_share'] = st['packets'] / max(1, total_packets)
        else:
            out[f'src{i}_ip'] = 'none'
            out[f'src{i}_packets'] = 0
            out[f'src{i}_bytes'] = 0
            out[f'src{i}_flows'] = 0
            out[f'src{i}_pps'] = 0.0
            out[f'src{i}_syn_ratio'] = 0.0
            out[f'src{i}_dst_diversity'] = 0
            out[f'src{i}_top_dst_share'] = 0.0
            out[f'src{i}_avg_duration'] = 0.0
            out[f'src{i}_pkt_share'] = 0.0

    # Global source diversity metrics
    out['active_sources'] = len(src_stats)
    pkt_counts = [st['packets'] for _, st in ranked]
    out['source_entropy'] = shannon_entropy(pkt_counts)
    out['top_source_dominance'] = (ranked[0][1]['packets'] / max(1, total_packets)) if ranked else 0.0

    return out


def compute_features(all_switch_flows, per_ctrl_stats, per_switch_stats, docker_stats, per_ctrl_metrics, per_ctrl_protos):
    global prev_stats, prev_flows_set, current_label, prev_metrics

    five_seen = set()
    dedup = []
    for f in all_switch_flows:
        key5 = (f["src_ip"], f["dst_ip"], f["src_port"], f["dst_port"], f["protocol"])
        if key5 in five_seen: continue
        five_seen.add(key5)
        dedup.append(f)
    flows = dedup

    utc_now = datetime.now(timezone.utc)
    features = {
        "schema": SCHEMA_VERSION,
        "timestamp_utc": utc_now.isoformat(),
        "timestamp_unix": utc_now.timestamp(),
        "t_monotonic": time.monotonic() - monotonic_start,
        "sample_id": len(collected_samples) + 1,
        "label": current_label,
        "attacker_ip": current_attacker_ip,
        "attacker_host": current_attacker_host
    }

    for ctrl in CONTROLLERS:
        cid = ctrl['id']
        agg = per_ctrl_stats.get(cid, {"flows":0,"packets":0,"bytes":0})
        features[f"{cid}_flows"]   = agg["flows"]
        features[f"{cid}_packets"] = agg["packets"]
        features[f"{cid}_bytes"]   = agg["bytes"]

        for k in ("flows","packets","bytes"):
            kk = f"{cid}_{k}"
            prev = prev_stats.get(kk)
            features[f"{kk}_rate"] = safe_rate(agg[k], prev, INTERVAL)
            prev_stats[kk] = agg[k]

        features[f"{cid}_cpu"] = (docker_stats.get(cid) or {}).get("cpu", 0.0)
        features[f"{cid}_mem"] = (docker_stats.get(cid) or {}).get("mem", "0B")

        m = per_ctrl_metrics.get(cid, {})
        for key in ("pktin_total","pktout_total","flowmiss_total"):
            v = m.get(key, np.nan)
            features[f"{cid}_{key}"] = v if not pd.isna(v) else 0.0
            prev_v = prev_metrics.get(f"{cid}_{key}")
            features[f"{cid}_{key}_rate"] = safe_rate(v, prev_v, INTERVAL)
            prev_metrics[f"{cid}_{key}"] = v if not pd.isna(v) else 0.0

        features[f"{cid}_event_queue"]  = m.get("event_queue", np.nan)
        features[f"{cid}_atomix_nodes"] = m.get("atomix_nodes", np.nan)
        features[f"{cid}_gc_pause_ms"]  = m.get("gc_pause_ms", np.nan)

        for ep in ("/flows","/hosts","/metrics"):
            hist = rest_lat_hist.get((cid, ep), [])
            if hist:
                clean = [x for x in hist if x >= 0]
                features[f"{cid}_rest{ep.replace('/','_')}_avg_ms"] = float(np.mean(clean)) if clean else np.nan
                features[f"{cid}_rest{ep.replace('/','_')}_max_ms"] = float(np.max(clean)) if clean else np.nan
            else:
                features[f"{cid}_rest{ep.replace('/','_')}_avg_ms"] = np.nan
                features[f"{cid}_rest{ep.replace('/','_')}_max_ms"] = np.nan

        proto = per_ctrl_protos.get(cid, {"tcp": 0, "udp": 0, "icmp": 0, "other": 0, "syn_flows": 0})
        features[f"{cid}_tcp_flows"] = proto["tcp"]
        features[f"{cid}_udp_flows"] = proto["udp"]
        features[f"{cid}_icmp_flows"] = proto["icmp"]
        features[f"{cid}_other_flows"] = proto["other"]
        features[f"{cid}_syn_flows"] = proto["syn_flows"]

    for sw in SWITCHES:
        s = per_switch_stats.get(sw, {"flows":0,"packets":0,"bytes":0})
        features[f"{sw}_flows"] = s["flows"]
        features[f"{sw}_packets"] = s["packets"]
        features[f"{sw}_bytes"] = s["bytes"]
        features[f"{sw}_master"] = SWITCH_TO_CONTROLLER.get(sw,"unknown")

    if flows:
        df = pd.DataFrame(flows)
        features["total_flows"]   = len(df)
        features["total_packets"] = int(df["packets"].sum())
        features["total_bytes"]   = int(df["bytes"].sum())
        features["avg_pkt_size"]  = features["total_bytes"]/max(1,features["total_packets"])

        counts = df["protocol"].value_counts()
        features["tcp_flows"] = int(counts.get("TCP",0))
        features["udp_flows"] = int(counts.get("UDP",0))
        features["icmp_flows"]= int(counts.get("ICMP",0))
        total = max(1,features["total_flows"])
        features["tcp_ratio"]  = features["tcp_flows"]/total
        features["udp_ratio"]  = features["udp_flows"]/total
        features["icmp_ratio"] = features["icmp_flows"]/total

        cluster_tcp = sum(per_ctrl_protos.get(c["id"], {}).get("tcp", 0) for c in CONTROLLERS)
        cluster_udp = sum(per_ctrl_protos.get(c["id"], {}).get("udp", 0) for c in CONTROLLERS)
        cluster_icmp = sum(per_ctrl_protos.get(c["id"], {}).get("icmp", 0) for c in CONTROLLERS)
        cluster_other = sum(per_ctrl_protos.get(c["id"], {}).get("other", 0) for c in CONTROLLERS)
        cluster_syn = sum(per_ctrl_protos.get(c["id"], {}).get("syn_flows", 0) for c in CONTROLLERS)
        cluster_total = max(1, cluster_tcp + cluster_udp + cluster_icmp + cluster_other)
        features["cluster_tcp_flows"] = cluster_tcp
        features["cluster_udp_flows"] = cluster_udp
        features["cluster_icmp_flows"] = cluster_icmp
        features["cluster_other_flows"] = cluster_other
        features["cluster_syn_flows"] = cluster_syn
        features["cluster_tcp_ratio"] = cluster_tcp / cluster_total
        features["cluster_udp_ratio"] = cluster_udp / cluster_total
        features["cluster_icmp_ratio"] = cluster_icmp / cluster_total
        features["cluster_syn_ratio"] = cluster_syn / max(1, cluster_tcp) if cluster_tcp > 0 else 0.0

        features["unique_srcs"] = df["src_ip"].nunique()
        features["unique_dsts"] = df["dst_ip"].nunique()
        features["src_dst_ratio"] = features["unique_srcs"]/max(1,features["unique_dsts"])
        features["src_entropy"] = shannon_entropy(df.groupby("src_ip")["packets"].sum().tolist())
        features["dst_entropy"] = shannon_entropy(df.groupby("dst_ip")["packets"].sum().tolist())

        features["unique_dst_ports"] = df["dst_port"].nunique()
        features["unique_src_ports"] = df["src_port"].nunique()
        features["port_diversity"] = (features["unique_src_ports"] + features["unique_dst_ports"])/2

        top_dst = df.groupby("dst_ip")["packets"].sum().idxmax()
        top_dst_pkts = int(df[df["dst_ip"] == top_dst]["packets"].sum())
        features["top_dst"] = top_dst
        features["top_dst_packets"] = top_dst_pkts
        features["top_dst_share"] = top_dst_pkts/max(1,features["total_packets"])

        curr = set(df["flow_id"])
        new_flows = len(curr - prev_flows_set)
        features["flow_arrival_rate"] = new_flows/max(1,INTERVAL)
        prev_flows_set.clear(); prev_flows_set.update(curr)

        features["avg_flow_duration"] = float(df["duration"].mean())
        features["max_flow_duration"] = float(df["duration"].max())
        features["min_flow_duration"] = float(df["duration"].min())
        features["avg_pkts_per_flow"] = features["total_packets"]/max(1,features["total_flows"])
        features["max_pkts_per_flow"] = int(df["packets"].max())
        stdv = df["packets"].std()
        features["std_pkts_per_flow"] = float(stdv if not pd.isna(stdv) else 0.0)

    else:
        features.update({
            "total_flows":0,"total_packets":0,"total_bytes":0,"avg_pkt_size":0.0,
            "tcp_flows":0,"udp_flows":0,"icmp_flows":0,"tcp_ratio":0.0,"udp_ratio":0.0,"icmp_ratio":0.0,
            "cluster_tcp_flows":0,"cluster_udp_flows":0,"cluster_icmp_flows":0,"cluster_other_flows":0,
            "cluster_syn_flows":0,"cluster_tcp_ratio":0.0,"cluster_udp_ratio":0.0,"cluster_icmp_ratio":0.0,"cluster_syn_ratio":0.0,
            "unique_srcs":0,"unique_dsts":0,"src_dst_ratio":0.0,"src_entropy":0.0,"dst_entropy":0.0,
            "unique_dst_ports":0,"unique_src_ports":0,"port_diversity":0.0,
            "top_dst":"none","top_dst_packets":0,"top_dst_share":0.0,
            "flow_arrival_rate":0.0,
            "avg_flow_duration":0.0,"max_flow_duration":0.0,"min_flow_duration":0.0,
            "avg_pkts_per_flow":0.0,"max_pkts_per_flow":0,"std_pkts_per_flow":0.0
        })

    ctrl_loads = [per_ctrl_stats.get(c["id"],{}).get("packets",0) for c in CONTROLLERS]
    sw_loads   = [per_switch_stats.get(sw,{}).get("packets",0) for sw in SWITCHES]
    features["ctrl_load_std"]  = float(np.std(ctrl_loads)) if ctrl_loads else 0.0
    features["ctrl_load_mean"] = float(np.mean(ctrl_loads)) if ctrl_loads else 0.0
    features["ctrl_load_max"]  = int(max(ctrl_loads)) if ctrl_loads else 0
    features["ctrl_load_min"]  = int(min(ctrl_loads)) if ctrl_loads else 0
    features["switch_load_std"]= float(np.std(sw_loads)) if sw_loads else 0.0
    features["switch_load_mean"]=float(np.mean(sw_loads)) if sw_loads else 0.0

    features["cluster_size"] = len(CONTROLLERS)
    features["active_controllers"] = sum(1 for c in CONTROLLERS if per_ctrl_stats.get(c["id"]))

    features["tracked_flows"] = len(flow_database)
    features["attack_in_progress"] = 1 if current_label==1 else 0
    features["cfg_interval_s"] = INTERVAL
    features["cfg_switches"] = ",".join(SWITCHES)
    features["cfg_controllers"] = ",".join(c["id"] for c in CONTROLLERS)

    features["warmup"] = 1 if features["sample_id"] <= WARMUP_SAMPLES else 0

    # ── Per-source-IP features (Approach 1 — NEW) ──
    features.update(compute_per_source_features(all_switch_flows))

    return features

def monitoring_loop():
    global monitoring_active, collected_samples, current_label, parquet_pending_counter, current_attacker_ip, current_attacker_host
    last_label_mtime = 0
    label_file = '/tmp/label_state'

    while monitoring_active:
        t_start_loop = time.time()
        old_label = current_label

        try:
            try:
                if os.path.exists(label_file):
                    mtime = os.path.getmtime(label_file)
                    if mtime > last_label_mtime:
                        with open(label_file, 'r') as f:
                            lines = f.readlines()
                            if lines:
                                last_line = lines[-1].strip()
                                parts = last_line.split()
                                if len(parts) >= 1:
                                    try:
                                        new_label = int(parts[0])
                                        if new_label != current_label:
                                            current_label = new_label
                                            print(f"[Label Sync] Changed to {current_label}")
                                        # Parse attacker identity from label_state
                                        if new_label == 1 and len(parts) >= 3:
                                            current_attacker_ip = parts[2].strip()
                                            current_attacker_host = parts[1].split("_")[0]
                                            print(f"[Label Sync] Attacker: {current_attacker_host} ({current_attacker_ip})")
                                        elif new_label == 1 and len(parts) >= 2:
                                            host_tag = parts[1].split("_")[0]
                                            current_attacker_ip = HOST_TO_IP.get(host_tag, "none")
                                            current_attacker_host = host_tag
                                            print(f"[Label Sync] Attacker inferred: {host_tag} ({current_attacker_ip})")
                                        elif new_label == 0:
                                            current_attacker_ip = "none"
                                            current_attacker_host = "none"
                                    except ValueError:
                                        pass
                        last_label_mtime = mtime
            except Exception as e:
                print(f"[Label Sync Error] {e}")

            if current_label != old_label:
                reset_prev_state(reason=f"label_flip {old_label}->{current_label}")

            mac_map = get_hosts_mac_map()
            docker = get_docker_stats()

            per_switch_stats = {}
            per_ctrl_stats = {}
            all_switch_flows = []

            for sw in SWITCHES:
                text = dump_switch_flows(sw)
                flows = parse_flows_enhanced(sw, text, mac_map)
                all_switch_flows.extend(flows)

                s_flows = len(flows)
                s_packets = sum(f.get("packets", 0) for f in flows)
                s_bytes = sum(f.get("bytes", 0) for f in flows)
                per_switch_stats[sw] = {"flows": s_flows, "packets": s_packets, "bytes": s_bytes}

                ctrl_ip = SWITCH_TO_CONTROLLER.get(sw)
                ctrl_id = next((c["id"] for c in CONTROLLERS if c["ip"] == ctrl_ip), None)
                if ctrl_id:
                    agg = per_ctrl_stats.setdefault(ctrl_id, {"flows": 0, "packets": 0, "bytes": 0})
                    agg["flows"] += s_flows
                    agg["packets"] += s_packets
                    agg["bytes"] += s_bytes

            per_ctrl_metrics = {}
            per_ctrl_protos = {}
            for ctrl in CONTROLLERS:
                try:
                    per_ctrl_metrics[ctrl["id"]] = get_onos_metrics_best_effort(ctrl)
                    per_ctrl_protos[ctrl["id"]] = get_onos_flows_with_protocol(ctrl)
                except Exception as e:
                    print(f"[monitor] Error querying {ctrl['id']}: {e}")
                    per_ctrl_metrics[ctrl["id"]] = {"pktin_total": 0.0, "pktout_total": 0.0}
                    per_ctrl_protos[ctrl["id"]] = {"tcp": 0, "udp": 0, "icmp": 0, "other": 0, "syn_flows": 0}

            features = compute_features(all_switch_flows, per_ctrl_stats, per_switch_stats, docker, per_ctrl_metrics, per_ctrl_protos)

            with data_lock:
                collected_samples.append(features)
                parquet_pending_counter += 1
                if len(collected_samples) > 10000:
                    collected_samples.pop(0)

            training_queue.put(features)

            try:
                df = pd.DataFrame([features])
                header = not os.path.exists(CSV_PATH)
                df.to_csv(CSV_PATH, mode='a', header=header, index=False)
            except Exception as e:
                print("[monitor] CSV save failed:", e)

            if parquet_pending_counter >= PARQUET_SYNC_EVERY:
                try:
                    import pyarrow
                    with data_lock:
                        dfp = pd.DataFrame(collected_samples)
                        parquet_pending_counter = 0
                    dfp.to_parquet(PARQUET_PATH, index=False)
                except Exception:
                    pass

        except Exception as e:
            print("[monitor] loop error:", e)

        elapsed = time.time() - t_start_loop
        sleep_time = max(0, INTERVAL - elapsed)
        if monitoring_active and sleep_time > 0:
            time.sleep(sleep_time)

def start_monitoring(label=0):
    global monitoring_active, monitoring_thread, current_label
    if monitoring_active:
        print("Monitoring already running")
        return
    current_label = int(label)
    monitoring_active = True
    reset_prev_state(reason="start_monitoring")
    monitoring_thread = threading.Thread(target=monitoring_loop, daemon=True)
    monitoring_thread.start()
    print(f"✓ Monitoring started (label={current_label}). Sampling every {INTERVAL}s")

def stop_monitoring():
    global monitoring_active, monitoring_thread, parquet_pending_counter
    if not monitoring_active:
        print("Monitoring is not running")
        return
    monitoring_active = False
    if monitoring_thread is not None:
        monitoring_thread.join(timeout=10)
    with data_lock:
        df_all = pd.DataFrame(collected_samples)

    # NEW (appends):
    try:
        header = not os.path.exists(CSV_PATH) or os.path.getsize(CSV_PATH) == 0
        df_all.to_csv(CSV_PATH, mode='a', header=header, index=False)
    except Exception:
        pass

    try:
        import pyarrow
        df_all.to_parquet(PARQUET_PATH, index=False)
        parquet_pending_counter = 0
    except Exception:
        pass
    print(f"✓ Monitoring stopped. {len(collected_samples)} samples saved to: {CSV_PATH}")

def set_label(label):
    global current_label
    old = current_label
    current_label = int(label)
    print(f"✓ Label set to {current_label}")
    if current_label != old:
        reset_prev_state(reason=f"manual_label {old}->{current_label}")

def show_status(verbose=True, top_n=10):
    print("=" * 60)
    print(f"MONITORING STATUS (schema v{SCHEMA_VERSION})")
    print("=" * 60)
    print(f"Active: {monitoring_active}")
    with data_lock:
        print(f"Collected samples (in memory): {len(collected_samples)}")
        last = collected_samples[-1] if collected_samples else None
    print(f"Training queue size: {training_queue.qsize()}")

    if not last:
        print("No samples recorded yet.")
        return

    print("\n" + "-" * 60)
    print("LAST SAMPLE")
    print("-" * 60)
    print(f"Timestamp: {last.get('timestamp_utc')}")
    print(f"Sample ID: {last.get('sample_id')} | Label: {last.get('label')} | Warmup: {last.get('warmup')}")
    print(f"Total flows: {last.get('total_flows', 0)} | Total packets: {last.get('total_packets', 0)}")
    # Show per-source summary
    print(f"Active sources: {last.get('active_sources', 0)} | Source entropy: {last.get('source_entropy', 0):.2f}")
    for i in range(1, min(4, TOP_K_SOURCES + 1)):
        ip = last.get(f'src{i}_ip', 'none')
        pkts = last.get(f'src{i}_packets', 0)
        share = last.get(f'src{i}_pkt_share', 0)
        syn = last.get(f'src{i}_syn_ratio', 0)
        dst_div = last.get(f'src{i}_dst_diversity', 0)
        if ip != 'none':
            print(f"  src{i}: {ip} | pkts={pkts} share={share:.2f} syn_ratio={syn:.2f} dst_div={dst_div}")
    print("=" * 60)

def list_active_flows(limit=20):
    items = sorted(flow_database.items(), key=lambda kv: kv[1].get("total_packets", 0), reverse=True)
    for fid, info in items[:limit]:
        print(fid, info.get("switch"), info.get("src_ip"), "->", info.get("dst_ip"), "pkts=", info.get("total_packets",0))

def get_flow_info(flow_id):
    return flow_database.get(flow_id)

print("=" * 70)
print("✓ SDN Controller Monitoring API v1.4.0 - READY (Per-Source-IP Enriched)")
print("=" * 70)
print("Functions: start_monitoring(label=0), stop_monitoring(), set_label(label)")
print("           show_status(verbose=True), list_active_flows(limit=20)")
print("           get_flow_info(flow_id)")
print("=" * 70)
print(f"REST_LAT_WINDOW={REST_LAT_WINDOW} | WARMUP_SAMPLES={WARMUP_SAMPLES}")
print(f"TOP_K_SOURCES={TOP_K_SOURCES} (63 new per-source columns)")
print("NEW: Per-source-IP features for DRL attacker identification (Approach 1)")
print("     Existing 162 columns UNCHANGED — LSTM works without modification")
print("=" * 70)


✓ SDN Controller Monitoring API v1.4.0 - READY (Per-Source-IP Enriched)
Functions: start_monitoring(label=0), stop_monitoring(), set_label(label)
           show_status(verbose=True), list_active_flows(limit=20)
           get_flow_info(flow_id)
REST_LAT_WINDOW=2 | WARMUP_SAMPLES=0
TOP_K_SOURCES=6 (63 new per-source columns)
NEW: Per-source-IP features for DRL attacker identification (Approach 1)
     Existing 162 columns UNCHANGED — LSTM works without modification


In [3]:
start_monitoring(label=0)

[RESET] prev_* cleared. reason=start_monitoring
✓ Monitoring started (label=0). Sampling every 2s


[Label Sync] Changed to 1
[Label Sync] Attacker: h2 (10.10.0.2)
[RESET] prev_* cleared. reason=label_flip 0->1
[Label Sync] Changed to 0
[RESET] prev_* cleared. reason=label_flip 1->0
[Label Sync] Changed to 1
[Label Sync] Attacker: h4 (10.10.0.4)
[RESET] prev_* cleared. reason=label_flip 0->1
[Label Sync] Changed to 0
[RESET] prev_* cleared. reason=label_flip 1->0
[Label Sync] Changed to 1
[Label Sync] Attacker: iot2 (10.10.0.102)
[RESET] prev_* cleared. reason=label_flip 0->1
[Label Sync] Changed to 0
[RESET] prev_* cleared. reason=label_flip 1->0


In [18]:
# ══════════════════════════════════════════════════════════════════
#  LIVE DRL AGENT v6 — CNN+LSTM → DQN → ONOS Mitigation
#  Paste AFTER the monitoring cell. Requires start_monitoring()
#
#  v6 — CRITICAL FIX:
#   ✅ ONOS treatment: empty instructions = DROP (not {"type":"DROP"})
#   ✅ Sends DROP to master controller per device (like baseline)
#   ✅ Exact preprocess_live_sample from working LSTM cell
#   ✅ /tmp/label_state direct read + auto-cleanup
# ══════════════════════════════════════════════════════════════════
import pickle, json, threading, time, requests, os, subprocess, re
import numpy as np
import pandas as pd
from collections import deque
from requests.auth import HTTPBasicAuth
from stable_baselines3 import DQN

# ── Paths ────────────────────────────────────────────────────────
DETECTOR_DIR = "outputs_detector"
DRL_DIR      = "outputs_ppo_dqn_compare"
SAVE_DIR     = "outputs_drl_live"
os.makedirs(SAVE_DIR, exist_ok=True)
DRL_DURATION = 600

# ── Load Detector ────────────────────────────────────────────────
with open(f"{DETECTOR_DIR}/config.json") as f:
    det_cfg = json.load(f)
LSTM_FEATURES  = det_cfg["feature_cols"]
N_LSTM_FEATS   = det_cfg["n_features"]
WINDOW_LEN     = det_cfg["BEST_WINDOW_LEN"]
BEST_THRESHOLD = det_cfg["BEST_THRESHOLD"]

with open(f"{DETECTOR_DIR}/calibration.json") as f:
    TEMPERATURE = json.load(f)["temperature"]
with open(f"{DETECTOR_DIR}/imputer.pkl", "rb") as f:
    lstm_imputer = pickle.load(f)
with open(f"{DETECTOR_DIR}/scaler.pkl", "rb") as f:
    lstm_scaler = pickle.load(f)

import tensorflow as tf
tf.get_logger().setLevel("ERROR")
lstm_model = tf.keras.models.load_model(f"{DETECTOR_DIR}/model_final.keras")
print(f"✅ LSTM loaded — window={WINDOW_LEN}, threshold={BEST_THRESHOLD:.4f}, T={TEMPERATURE:.4f}")

# ── Load DQN ─────────────────────────────────────────────────────
with open(f"{DRL_DIR}/deployment_recipe.json") as f:
    recipe = json.load(f)
with open(f"{DRL_DIR}/obs_scaler.pkl", "rb") as f:
    dqn_scaler = pickle.load(f)

dqn_agent = DQN.load(f"{DRL_DIR}/dqn_model.zip")
TOP_K = recipe.get("top_k", 6)
SRC_NUM_FEATS = ["packets","bytes","flows","pps","syn_ratio",
                 "dst_diversity","top_dst_share","avg_duration","pkt_share"]
GLOBAL_FEATS  = ["active_sources","source_entropy","top_source_dominance"]
KNOWN_LEGIT   = {"10.10.0.1","10.10.0.3","10.10.0.200","10.10.0.201","none",""}
print(f"✅ DQN loaded — obs_dim={recipe.get('obs_dim')}, actions={TOP_K+1}")

# ── ONOS / OVS Config ───────────────────────────────────────────
ONOS_NODES   = [{"ip":"175.24.1.5","port":8181},
                {"ip":"175.24.1.6","port":8181},
                {"ip":"175.24.1.7","port":8181}]
AUTH         = HTTPBasicAuth("onos","rocks")
ONOS_AUTH    = ("onos","rocks")
SWITCH_DPIDS = [f"of:{i:016x}" for i in range(1,5)]
SWITCHES     = ["s1","s2","s3","s4"]
DROP_PRIO    = 65000   # same as baseline — higher than any ONOS fwd rule

mit_lock    = threading.Lock()
active_mit  = {}
blocked_ips = set()


# ═════════════════════════════════════════════════════════════════
#  FEATURE PREPROCESSING — EXACT COPY from working LSTM cell
# ═════════════════════════════════════════════════════════════════
def parse_memory_string(val):
    """Convert memory strings like '1.638GiB' to numeric bytes."""
    if isinstance(val, (int, float)):
        return float(val)
    if not isinstance(val, str):
        return np.nan
    val = val.strip()
    multipliers = {
        'gib': 1024**3, 'gb': 1e9,
        'mib': 1024**2, 'mb': 1e6,
        'kib': 1024,    'kb': 1e3,
        'b': 1
    }
    for suffix, mult in multipliers.items():
        if val.lower().endswith(suffix):
            try:
                return float(val[:len(val)-len(suffix)]) * mult
            except ValueError:
                return np.nan
    try:
        return float(val)
    except ValueError:
        return np.nan


def apply_feature_mapping(df):
    """Replicate EXACTLY the same feature engineering from training."""
    if 'cluster_syn_flows' in df.columns and 'syn_flows' in df.columns:
        df.drop(columns=['syn_flows'], inplace=True)
        df.rename(columns={'cluster_syn_flows': 'syn_flows'}, inplace=True)
    elif 'cluster_syn_flows' in df.columns:
        df.rename(columns={'cluster_syn_flows': 'syn_flows'}, inplace=True)

    if 'cluster_syn_ratio' in df.columns and 'syn_ratio' in df.columns:
        df.drop(columns=['syn_ratio'], inplace=True)
        df.rename(columns={'cluster_syn_ratio': 'syn_ratio'}, inplace=True)
    elif 'cluster_syn_ratio' in df.columns:
        df.rename(columns={'cluster_syn_ratio': 'syn_ratio'}, inplace=True)

    if 'unique_dst_ports' in df.columns:
        udp = df['unique_dst_ports'].fillna(0).clip(lower=0)
        if 'well_known_port_flows' not in df.columns or df['well_known_port_flows'].isna().all():
            df['well_known_port_flows'] = np.where(
                udp > 0,
                np.minimum(udp, 1024) / np.maximum(udp, 1),
                0.0
            )

    if 'unique_srcs' in df.columns and 'unique_dsts' in df.columns:
        if 'five_unique' not in df.columns or df['five_unique'].isna().all():
            df['five_unique'] = df['unique_srcs'].fillna(0) + df['unique_dsts'].fillna(0)

    return df


def preprocess_live_sample(raw_df):
    """Full preprocessing — exact replica of working LSTM cell."""
    df = raw_df.copy()
    df = apply_feature_mapping(df)
    mem_cols = [c for c in df.columns if '_mem' in c]
    for col in mem_cols:
        df[col] = df[col].apply(parse_memory_string)
    df.replace([np.inf, -np.inf], np.nan, inplace=True)
    for col in LSTM_FEATURES:
        if col not in df.columns:
            df[col] = np.nan
    X = df[LSTM_FEATURES].values.astype(np.float64)
    X = lstm_imputer.transform(X)
    X = lstm_scaler.transform(X)
    return X


# ═════════════════════════════════════════════════════════════════
#  ONOS HELPERS (from baseline — these WORK)
# ═════════════════════════════════════════════════════════════════
def get_master_controller(device_id):
    """Find the ONOS master for a device."""
    for node in ONOS_NODES:
        try:
            url = f"http://{node['ip']}:{node['port']}/onos/v1/mastership/{device_id}/master"
            r = requests.get(url, auth=ONOS_AUTH, timeout=2)
            if r.ok:
                master_id = r.json().get('nodeId', '')
                for n2 in ONOS_NODES:
                    if master_id and (master_id in n2['ip'] or master_id == n2.get('id','')):
                        return n2
        except: continue
    return ONOS_NODES[0]


# ═════════════════════════════════════════════════════════════════
#  MITIGATION — ONOS REST (like baseline, NOT ovs-ofctl)
# ═════════════════════════════════════════════════════════════════
def install_drop(attacker_ip, timeout=0):
    """
    Install DROP flow via ONOS REST on ALL switches.
    Key: treatment.instructions = [] (EMPTY = DROP in ONOS).
    This is what the baseline uses and it WORKS.
    """
    with mit_lock:
        if attacker_ip in active_mit and active_mit[attacker_ip] > time.time():
            return
        active_mit[attacker_ip] = time.time() + 600
        blocked_ips.add(attacker_ip)

    ok = 0
    for dpid in SWITCH_DPIDS:
        master = get_master_controller(dpid)
        flow = {
            "priority": DROP_PRIO,
            "isPermanent": (timeout == 0),
            "deviceId": dpid,
            "treatment": {"instructions": []},     # ← EMPTY = DROP (CRITICAL FIX)
            "selector": {"criteria": [
                {"type": "ETH_TYPE", "ethType": "0x0800"},
                {"type": "IPV4_SRC", "ip": f"{attacker_ip}/32"},
            ]},
        }
        if timeout > 0:
            flow["timeout"] = timeout
            flow["isPermanent"] = False
        try:
            url = f"http://{master['ip']}:{master['port']}/onos/v1/flows"
            r = requests.post(url, json={"flows": [flow]}, auth=ONOS_AUTH, timeout=3)
            if r.ok:
                ok += 1
        except: pass

    ttl = f"TTL={timeout}s" if timeout > 0 else "PERMANENT"
    print(f"[DRL] 🛡️  DROP installed: {attacker_ip} on {ok}/4 switches (prio={DROP_PRIO}, {ttl})")


def clear_all_mitigations():
    """Remove ALL our DROP rules via ONOS REST."""
    with mit_lock:
        ips = list(blocked_ips)
        active_mit.clear()
        blocked_ips.clear()

    # Remove high-priority flows from ONOS
    for node in ONOS_NODES:
        try:
            r = requests.get(f"http://{node['ip']}:{node['port']}/onos/v1/flows",
                             auth=ONOS_AUTH, timeout=5)
            if r.ok:
                for flow in r.json().get("flows", []):
                    if flow.get("priority", 0) >= DROP_PRIO:
                        dev = flow.get("deviceId", "")
                        fid = flow.get("id", "")
                        if dev and fid:
                            try:
                                requests.delete(
                                    f"http://{node['ip']}:{node['port']}/onos/v1/flows/{dev}/{fid}",
                                    auth=ONOS_AUTH, timeout=2)
                            except: pass
            break
        except: pass

    # Also try ovs-ofctl cleanup as backup
    for sw in SWITCHES:
        try:
            subprocess.run(
                f"sudo ovs-ofctl -O OpenFlow13 del-flows {sw} priority={DROP_PRIO}",
                shell=True, timeout=3, capture_output=True)
        except: pass

    if ips:
        print(f"[DRL] 🧹 Cleared DROP rules: {ips}")
    else:
        print(f"[DRL] 🧹 Mitigations cleared")


def get_dropped_packet_count():
    """Count packets dropped by our rules via ONOS REST API."""
    total_dropped = 0
    for node in ONOS_NODES:
        try:
            r = requests.get(f"http://{node['ip']}:{node['port']}/onos/v1/flows",
                             auth=ONOS_AUTH, timeout=3)
            if r.ok:
                for flow in r.json().get("flows", []):
                    if flow.get("priority", 0) >= DROP_PRIO:
                        total_dropped += flow.get("packets", 0)
            break
        except: pass
    return total_dropped


# ═════════════════════════════════════════════════════════════════
#  LABEL + DQN
# ═════════════════════════════════════════════════════════════════
def read_label_state():
    try:
        with open("/tmp/label_state", "r") as f:
            lines = f.readlines()
            if lines:
                parts = lines[-1].strip().split()
                label = int(parts[0])
                if label == 1:
                    host = parts[1].split("_")[0] if len(parts) >= 2 else "unknown"
                    ip   = parts[2].strip() if len(parts) >= 3 else "none"
                    return label, host, ip
                else:
                    return 0, "none", "none"
    except: pass
    return 0, "none", "none"


def build_dqn_obs(attack_prob, sample):
    src_raw = np.array(
        [float(sample.get(f"src{k}_{f}", 0) or 0)
         for k in range(1, TOP_K+1) for f in SRC_NUM_FEATS], dtype=np.float32)
    glob_raw = np.array(
        [float(sample.get(g, 0) or 0) for g in GLOBAL_FEATS], dtype=np.float32)
    extra = np.concatenate([src_raw, glob_raw]).reshape(1, -1)
    extra = np.nan_to_num(extra, nan=0.0, posinf=1e6, neginf=-1e6)
    extra_scaled = np.clip(dqn_scaler.transform(extra), -10, 10).flatten()
    severity   = min(1.0, attack_prob * 1.2)
    confidence = abs(attack_prob - 0.5) * 2
    return np.concatenate([[attack_prob, severity, confidence], extra_scaled]).astype(np.float32)


def calibrate_probability(raw_prob, temperature):
    eps = 1e-7
    logits = np.log(np.clip(raw_prob, eps, 1-eps) /
                    np.clip(1-raw_prob, eps, 1-eps))
    return float(1.0 / (1.0 + np.exp(-logits / temperature)))


# ════════════════════════════════════════════════════════════════
#  MAIN LIVE LOOP
# ════════════════════════════════════════════════════════════════
feature_window       = deque(maxlen=WINDOW_LEN)
step_log             = []
prev_samples         = 0
step                 = 0
t_start              = time.time()
prev_gt_label        = 0
mitigation_on        = False
first_block_step     = None
first_block_elapsed  = None
attack_start_step    = None
attack_start_elapsed = None

clear_all_mitigations()

print("╔══════════════════════════════════════════════════════════╗")
print("║   CNN+LSTM → DQN Live Mitigation Agent  v6              ║")
print(f"║   Duration={DRL_DURATION}s  Window={WINDOW_LEN}  Threshold={BEST_THRESHOLD:.4f}          ║")
print("║   ✅ ONOS REST DROP (empty instructions = drop)         ║")
print("║   ✅ Exact preprocess_live_sample from LSTM cell         ║")
print("║   ✅ Auto-cleanup on attack end                          ║")
print("╚══════════════════════════════════════════════════════════╝")

try:
    while time.time() - t_start < DRL_DURATION:
        time.sleep(INTERVAL)

        with data_lock:
            snap = list(collected_samples)
        if len(snap) <= prev_samples:
            continue
        new = snap[prev_samples:]
        prev_samples = len(snap)

        for sample in new:
            elapsed = round(time.time() - t_start, 1)

            # ── 1. GROUND TRUTH ──────────────────────────────────
            gt_label, gt_host, gt_ip = read_label_state()

            # ── 2. TRANSITIONS ───────────────────────────────────
            if prev_gt_label == 0 and gt_label == 1:
                attack_start_step = step
                attack_start_elapsed = elapsed
                first_block_step = None
                first_block_elapsed = None
                print(f"\n[DRL] ⬆️  ATTACK STARTED step {step} ({elapsed}s) — {gt_host}({gt_ip})")

            if prev_gt_label == 1 and gt_label == 0:
                print(f"\n[DRL] ⬇️  ATTACK ENDED step {step} ({elapsed}s)")
                if first_block_elapsed and attack_start_elapsed is not None:
                    print(f"[DRL]    Reaction: {first_block_elapsed - attack_start_elapsed:.1f}s")
                clear_all_mitigations()
                mitigation_on = False
                first_block_step = None
                attack_start_step = None

            prev_gt_label = gt_label

            # ── 3. PREPROCESS (exact same as working LSTM cell) ──
            sample_df = pd.DataFrame([sample])
            X_row = preprocess_live_sample(sample_df)
            feature_window.append(X_row[0])

            if len(feature_window) < WINDOW_LEN:
                print(f"  [Warmup] {len(feature_window)}/{WINDOW_LEN}")
                continue

            window_arr = np.array(list(feature_window), dtype=np.float32).reshape(
                1, WINDOW_LEN, N_LSTM_FEATS)

            # ── 4. LSTM INFERENCE ────────────────────────────────
            raw_prob = float(lstm_model.predict(window_arr, verbose=0).ravel()[0])
            cal_prob = calibrate_probability(raw_prob, TEMPERATURE)
            binary   = int(cal_prob >= BEST_THRESHOLD)

            # ── 5. DQN DECISION ──────────────────────────────────
            obs       = build_dqn_obs(cal_prob, sample)
            action, _ = dqn_agent.predict(obs, deterministic=True)
            action    = int(action)

            # ── 6. MITIGATION ────────────────────────────────────
            mitigated_ip = None
            action_str   = "ALLOW"

            if gt_label == 1:
                if mitigation_on:
                    action_str = f"HOLD_BLOCK ({','.join(blocked_ips)})"
                elif action > 0:
                    src_ip = str(sample.get(f"src{action}_ip", "none")).strip()
                    if src_ip not in KNOWN_LEGIT and src_ip != "none":
                        mitigated_ip = src_ip
                        install_drop(src_ip, timeout=0)  # PERMANENT until we clear
                        mitigation_on = True
                        action_str = f"🎯 BLOCK src{action}({src_ip})"
                        if first_block_step is None:
                            first_block_step = step
                            first_block_elapsed = elapsed
                            reaction = elapsed - (attack_start_elapsed or elapsed)
                            print(f"[DRL] 🎯 First block step {step} ({elapsed}s) — reaction: {reaction:.1f}s")
                    else:
                        action_str = f"SKIP_LEGIT src{action}({src_ip})"
                elif not mitigation_on:
                    action_str = "SEARCHING..."
            else:
                if action > 0:
                    action_str = "FA_BLOCKED (ignored)"
                else:
                    action_str = "ALLOW ✅"

            # ── 7. METRICS ───────────────────────────────────────
            docker    = get_docker_stats()
            onos1_cpu = (docker.get("onos1") or {}).get("cpu", 0.0)
            onos2_cpu = (docker.get("onos2") or {}).get("cpu", 0.0)
            onos3_cpu = (docker.get("onos3") or {}).get("cpu", 0.0)
            ctrl_cpu  = (onos1_cpu + onos2_cpu + onos3_cpu) / 3

            total_packets   = int(sample.get("total_packets", 0))
            dropped_packets = get_dropped_packet_count() if mitigation_on else 0
            forwarded_pkts  = max(0, total_packets - dropped_packets)
            total_flows     = int(sample.get("total_flows", 0))
            total_bytes     = int(sample.get("total_bytes", 0))
            src_entropy     = float(sample.get("src_entropy",
                              sample.get("source_entropy", 0)) or 0)
            top_src_dom     = float(sample.get("top_source_dominance", 0) or 0)
            active_sources  = int(sample.get("active_sources", 0) or 0)
            flow_arr_rate   = float(sample.get("flow_arrival_rate", 0) or 0)

            # ── 8. PRINT ─────────────────────────────────────────
            correct  = "✅" if binary == gt_label else "❌"
            mit_icon = "🛡️" if mitigation_on else "  "
            prob_bar = "█" * int(cal_prob * 20) + "░" * (20 - int(cal_prob * 20))

            if mitigation_on:
                print(f"[{step:>4}] {elapsed:>6.1f}s {mit_icon} |{prob_bar}| "
                      f"P={cal_prob:.4f} {correct} gt={gt_label}({gt_host}) | "
                      f"{action_str} | fwd={forwarded_pkts} drop={dropped_packets} cpu={ctrl_cpu:.1f}%")
            else:
                print(f"[{step:>4}] {elapsed:>6.1f}s {mit_icon} |{prob_bar}| "
                      f"P={cal_prob:.4f} {correct} gt={gt_label}({gt_host}) | "
                      f"{action_str} | pkts={total_packets} cpu={ctrl_cpu:.1f}%")

            # ── 9. LOG ───────────────────────────────────────────
            step_log.append({
                "step": step, "elapsed_s": elapsed,
                "attack_prob": cal_prob, "raw_prob": raw_prob,
                "binary": binary, "gt_label": gt_label,
                "gt_host": gt_host, "gt_ip": gt_ip,
                "correct": int(binary == gt_label),
                "dqn_action": action, "action_taken": action_str,
                "mitigated_ip": mitigated_ip,
                "mitigation_active": int(mitigation_on),
                "blocked_ips": ",".join(blocked_ips) if blocked_ips else "",
                "total_packets": total_packets,
                "forwarded_packets": forwarded_pkts,
                "dropped_packets": dropped_packets,
                "total_flows": total_flows,
                "total_bytes": total_bytes,
                "s1_packets": int(sample.get("s1_packets", 0) or 0),
                "s2_packets": int(sample.get("s2_packets", 0) or 0),
                "s3_packets": int(sample.get("s3_packets", 0) or 0),
                "s4_packets": int(sample.get("s4_packets", 0) or 0),
                "flow_arrival_rate": flow_arr_rate,
                "src_entropy": src_entropy,
                "top_source_dominance": top_src_dom,
                "active_sources": active_sources,
                "unique_srcs": int(sample.get("unique_srcs", 0) or 0),
                "onos1_cpu": onos1_cpu, "onos2_cpu": onos2_cpu,
                "onos3_cpu": onos3_cpu, "ctrl_cpu_avg": ctrl_cpu,
                "attack_start_elapsed": attack_start_elapsed,
                "first_block_elapsed": first_block_elapsed,
            })
            step += 1

except KeyboardInterrupt:
    print(f"\n🛑 Stopped at step {step}")

# ── FINAL CLEANUP ────────────────────────────────────────────────
clear_all_mitigations()
elapsed_total = time.time() - t_start
print(f"\n⏱️  Total: {step} steps in {elapsed_total:.0f}s")

if step_log:
    df_log = pd.DataFrame(step_log)
    df_log.to_csv(f"{SAVE_DIR}/live_drl_log_v1.csv", index=False)
    print(f"💾 Saved: {SAVE_DIR}/live_drl_log_v1.csv ({len(df_log)} rows)")

    from sklearn.metrics import f1_score, precision_score, recall_score
    f1  = f1_score(df_log["gt_label"], df_log["binary"], zero_division=0)
    pre = precision_score(df_log["gt_label"], df_log["binary"], zero_division=0)
    rec = recall_score(df_log["gt_label"], df_log["binary"], zero_division=0)

    atk = df_log[df_log["gt_label"]==1]
    ben = df_log[df_log["gt_label"]==0]

    print("\n══════════════════════════════════════════════════")
    print("  LIVE DRL v6 — RESULTS SUMMARY")
    print("══════════════════════════════════════════════════")
    print(f"  LSTM Detection F1:  {f1:.4f}")
    print(f"  Precision:          {pre:.4f}")
    print(f"  Recall:             {rec:.4f}")
    print(f"  Total mitigations:  {df_log['mitigated_ip'].notna().sum()} "
          f"({df_log['mitigated_ip'].nunique()} unique IPs)")

    if first_block_elapsed is not None and attack_start_elapsed is not None:
        print(f"  Reaction time:      {first_block_elapsed - attack_start_elapsed:.1f}s")

    print(f"\n  ─── Traffic Analysis ───")
    if len(atk):
        pre_mit  = atk[atk['mitigation_active']==0]
        post_mit = atk[atk['mitigation_active']==1]
        if len(pre_mit):
            print(f"  Pkts (pre-mitigation):   {pre_mit['total_packets'].mean():.0f}")
        if len(post_mit):
            print(f"  Pkts (post-mitigation):  {post_mit['total_packets'].mean():.0f}")
            print(f"  Forwarded (post-mit):    {post_mit['forwarded_packets'].mean():.0f}")
            print(f"  Dropped (post-mit):      {post_mit['dropped_packets'].mean():.0f}")
            if len(pre_mit) and pre_mit['total_packets'].mean() > 0:
                fwd_red = 1 - post_mit['forwarded_packets'].mean()/pre_mit['total_packets'].mean()
                print(f"  ⚡ Forwarded reduction:   {fwd_red*100:.1f}%")
    if len(ben):
        print(f"  Pkts (benign):           {ben['total_packets'].mean():.0f}")

    print(f"\n  ─── Controller Load ───")
    if len(atk): print(f"  CPU (attack):   {atk['ctrl_cpu_avg'].mean():.1f}%")
    if len(ben): print(f"  CPU (benign):   {ben['ctrl_cpu_avg'].mean():.1f}%")
    mit_rows = df_log[df_log['mitigation_active']==1]
    if len(mit_rows): print(f"  CPU (w/ mit):   {mit_rows['ctrl_cpu_avg'].mean():.1f}%")

    ben_fa = df_log[(df_log['gt_label']==0) & (df_log['dqn_action']>0)]
    print(f"\n  ─── Benign Safety ───")
    print(f"  False alarms: {len(ben_fa)} (all ignored)")
    print(f"  Benign:       untouched ✅")

    # Show LSTM behavior during mitigation
    if len(mit_rows):
        print(f"\n  ─── LSTM During Mitigation ───")
        print(f"  Avg prob:  {mit_rows['attack_prob'].mean():.4f}")
        prob_below = len(mit_rows[mit_rows['attack_prob'] < BEST_THRESHOLD])
        print(f"  Steps below threshold: {prob_below}/{len(mit_rows)}")
        print(f"  ↳ Prob should DROP after block (attack traffic stops)")

    print("══════════════════════════════════════════════════")


✅ LSTM loaded — window=5, threshold=0.8472, T=0.9568
✅ DQN loaded — obs_dim=60, actions=7
[DRL] 🧹 Mitigations cleared
╔══════════════════════════════════════════════════════════╗
║   CNN+LSTM → DQN Live Mitigation Agent  v6              ║
║   Duration=600s  Window=5  Threshold=0.8472          ║
║   ✅ ONOS REST DROP (empty instructions = drop)         ║
║   ✅ Exact preprocess_live_sample from LSTM cell         ║
║   ✅ Auto-cleanup on attack end                          ║
╚══════════════════════════════════════════════════════════╝
  [Warmup] 1/5
  [Warmup] 2/5
  [Warmup] 3/5
  [Warmup] 4/5
[   0]    6.2s    |░░░░░░░░░░░░░░░░░░░░| P=0.0041 ✅ gt=0(none) | ALLOW ✅ | pkts=22 cpu=2.4%
[   1]   10.3s    |░░░░░░░░░░░░░░░░░░░░| P=0.0050 ✅ gt=0(none) | ALLOW ✅ | pkts=68 cpu=4.4%
[   2]   12.3s    |░░░░░░░░░░░░░░░░░░░░| P=0.0072 ✅ gt=0(none) | ALLOW ✅ | pkts=117 cpu=5.7%
[   3]   16.4s    |░░░░░░░░░░░░░░░░░░░░| P=0.0043 ✅ gt=0(none) | ALLOW ✅ | pkts=152 cpu=2.7%
[   4]   18.4s    |░░░░░░░░░░░░░░░

In [3]:
# ══════════════════════════════════════════════════════════════════
#  LIVE DRL AGENT v10 — CNN+LSTM → DQN → ONOS Mitigation
#  v10 Fixes:
#   ✅ preprocess = EXACT working cell (5b)
#   ✅ gt_label from SAMPLE (synced with traffic, like 5b)
#   ✅ ONOS REST DROP (empty instructions)
#   ✅ Auto-cleanup on attack end
# ══════════════════════════════════════════════════════════════════
import pickle, json, threading, time, requests, os, subprocess, re
import numpy as np
import pandas as pd
from collections import deque
from requests.auth import HTTPBasicAuth
from stable_baselines3 import DQN

# ── Paths ────────────────────────────────────────────────────────
DETECTOR_DIR = "outputs_detector"
DRL_DIR      = "outputs_ppo_dqn_compare"
SAVE_DIR     = "outputs_drl_live"
os.makedirs(SAVE_DIR, exist_ok=True)
DRL_DURATION = 600

# ── Load Detector ────────────────────────────────────────────────
with open(f"{DETECTOR_DIR}/config.json") as f:
    det_cfg = json.load(f)
LSTM_FEATURES  = det_cfg["feature_cols"]
N_LSTM_FEATS   = det_cfg["n_features"]
WINDOW_LEN     = det_cfg["BEST_WINDOW_LEN"]
BEST_THRESHOLD = det_cfg["BEST_THRESHOLD"]

with open(f"{DETECTOR_DIR}/calibration.json") as f:
    TEMPERATURE = json.load(f)["temperature"]
with open(f"{DETECTOR_DIR}/imputer.pkl", "rb") as f:
    lstm_imputer = pickle.load(f)
with open(f"{DETECTOR_DIR}/scaler.pkl", "rb") as f:
    lstm_scaler = pickle.load(f)

import tensorflow as tf
tf.get_logger().setLevel("ERROR")
lstm_model = tf.keras.models.load_model(f"{DETECTOR_DIR}/model_final.keras")
print(f"✅ LSTM loaded — window={WINDOW_LEN}, threshold={BEST_THRESHOLD:.4f}, T={TEMPERATURE:.4f}")

# ── Load DQN ─────────────────────────────────────────────────────
with open(f"{DRL_DIR}/deployment_recipe.json") as f:
    recipe = json.load(f)
with open(f"{DRL_DIR}/obs_scaler.pkl", "rb") as f:
    dqn_scaler = pickle.load(f)

dqn_agent = DQN.load(f"{DRL_DIR}/dqn_model.zip")
TOP_K = recipe.get("top_k", 6)
SRC_NUM_FEATS = ["packets","bytes","flows","pps","syn_ratio",
                 "dst_diversity","top_dst_share","avg_duration","pkt_share"]
GLOBAL_FEATS  = ["active_sources","source_entropy","top_source_dominance"]
KNOWN_LEGIT   = {"10.10.0.1","10.10.0.3","10.10.0.200","10.10.0.201","none",""}
print(f"✅ DQN loaded — obs_dim={recipe.get('obs_dim')}, actions={TOP_K+1}")

# ── ONOS / OVS Config ───────────────────────────────────────────
ONOS_NODES   = [{"ip":"175.24.1.5","port":8181},
                {"ip":"175.24.1.6","port":8181},
                {"ip":"175.24.1.7","port":8181}]
ONOS_AUTH    = ("onos","rocks")
SWITCH_DPIDS = [f"of:{i:016x}" for i in range(1,5)]
SWITCHES     = ["s1","s2","s3","s4"]
DROP_PRIO    = 65000

mit_lock    = threading.Lock()
active_mit  = {}
blocked_ips = set()


# ═════════════════════════════════════════════════════════════════
#  PREPROCESSING — EXACT COPY of WORKING cell 5b
# ═════════════════════════════════════════════════════════════════
def preprocess_live_sample(raw_df):
    df = raw_df.copy()
    for c in LSTM_FEATURES:
        if c not in df.columns:
            df[c] = 0.0
    for c in LSTM_FEATURES:
        if df[c].dtype == "object":
            ex = df[c].astype(str).str.extract(r"([-+]?\d*\.?\d+)")[0]
            df[c] = pd.to_numeric(ex, errors="coerce")
        else:
            df[c] = pd.to_numeric(df[c], errors="coerce")
        df[c] = df[c].replace([np.inf, -np.inf], np.nan).fillna(0.0)
    X = df[LSTM_FEATURES].values.astype(np.float64)
    if lstm_imputer is not None:
        X = lstm_imputer.transform(X)
    if lstm_scaler is not None:
        X = lstm_scaler.transform(X)
    X = np.nan_to_num(X, nan=0.0, posinf=0.0, neginf=0.0)
    return X.astype(np.float32)


# ═════════════════════════════════════════════════════════════════
#  ONOS HELPERS
# ═════════════════════════════════════════════════════════════════
def get_master_controller(device_id):
    for node in ONOS_NODES:
        try:
            r = requests.get(
                f"http://{node['ip']}:{node['port']}/onos/v1/mastership/{device_id}/master",
                auth=ONOS_AUTH, timeout=2)
            if r.ok:
                mid = r.json().get('nodeId','')
                for n2 in ONOS_NODES:
                    if mid and (mid in n2['ip'] or mid == n2.get('id','')):
                        return n2
        except: continue
    return ONOS_NODES[0]


# ═════════════════════════════════════════════════════════════════
#  MITIGATION — ONOS REST
# ═════════════════════════════════════════════════════════════════
def install_drop(attacker_ip, timeout=0):
    with mit_lock:
        if attacker_ip in active_mit and active_mit[attacker_ip] > time.time():
            return
        active_mit[attacker_ip] = time.time() + 600
        blocked_ips.add(attacker_ip)
    ok = 0
    for dpid in SWITCH_DPIDS:
        master = get_master_controller(dpid)
        flow = {
            "priority": DROP_PRIO,
            "isPermanent": (timeout == 0),
            "deviceId": dpid,
            "treatment": {"instructions": []},
            "selector": {"criteria": [
                {"type":"ETH_TYPE","ethType":"0x0800"},
                {"type":"IPV4_SRC","ip":f"{attacker_ip}/32"},
            ]},
        }
        if timeout > 0:
            flow["timeout"] = timeout
            flow["isPermanent"] = False
        try:
            r = requests.post(
                f"http://{master['ip']}:{master['port']}/onos/v1/flows",
                json={"flows":[flow]}, auth=ONOS_AUTH, timeout=3)
            if r.ok: ok += 1
        except: pass
    ttl = f"TTL={timeout}s" if timeout > 0 else "PERMANENT"
    print(f"[DRL] 🛡️  DROP: {attacker_ip} on {ok}/4 switches (prio={DROP_PRIO}, {ttl})")


def clear_all_mitigations():
    with mit_lock:
        ips = list(blocked_ips)
        active_mit.clear()
        blocked_ips.clear()
    for node in ONOS_NODES:
        try:
            r = requests.get(f"http://{node['ip']}:{node['port']}/onos/v1/flows",
                             auth=ONOS_AUTH, timeout=5)
            if r.ok:
                for flow in r.json().get("flows",[]):
                    if flow.get("priority",0) >= DROP_PRIO:
                        dev = flow.get("deviceId","")
                        fid = flow.get("id","")
                        if dev and fid:
                            try:
                                requests.delete(
                                    f"http://{node['ip']}:{node['port']}/onos/v1/flows/{dev}/{fid}",
                                    auth=ONOS_AUTH, timeout=2)
                            except: pass
            break
        except: pass
    for sw in SWITCHES:
        try:
            subprocess.run(f"sudo ovs-ofctl -O OpenFlow13 del-flows {sw} priority={DROP_PRIO}",
                           shell=True, timeout=3, capture_output=True)
        except: pass
    if ips: print(f"[DRL] 🧹 Cleared DROP: {ips}")
    else:   print(f"[DRL] 🧹 Mitigations cleared")


def get_dropped_packet_count():
    total = 0
    for node in ONOS_NODES:
        try:
            r = requests.get(f"http://{node['ip']}:{node['port']}/onos/v1/flows",
                             auth=ONOS_AUTH, timeout=3)
            if r.ok:
                for flow in r.json().get("flows",[]):
                    if flow.get("priority",0) >= DROP_PRIO:
                        total += flow.get("packets",0)
            break
        except: pass
    return total


# ═════════════════════════════════════════════════════════════════
#  HELPERS
# ═════════════════════════════════════════════════════════════════
def read_label_state():
    """Fallback label reader for attacker identity."""
    try:
        with open("/tmp/label_state","r") as f:
            lines = f.readlines()
            if lines:
                parts = lines[-1].strip().split()
                label = int(parts[0])
                if label == 1:
                    host = parts[1].split("_")[0] if len(parts)>=2 else "unknown"
                    ip   = parts[2].strip() if len(parts)>=3 else "none"
                    return label, host, ip
                else:
                    return 0, "none", "none"
    except: pass
    return 0, "none", "none"


def build_dqn_obs(attack_prob, sample):
    src_raw = np.array(
        [float(sample.get(f"src{k}_{f}",0) or 0)
         for k in range(1,TOP_K+1) for f in SRC_NUM_FEATS], dtype=np.float32)
    glob_raw = np.array(
        [float(sample.get(g,0) or 0) for g in GLOBAL_FEATS], dtype=np.float32)
    extra = np.concatenate([src_raw, glob_raw]).reshape(1,-1)
    extra = np.nan_to_num(extra, nan=0.0, posinf=1e6, neginf=-1e6)
    extra_scaled = np.clip(dqn_scaler.transform(extra), -10, 10).flatten()
    severity   = min(1.0, attack_prob * 1.2)
    confidence = abs(attack_prob - 0.5) * 2
    return np.concatenate([[attack_prob, severity, confidence], extra_scaled]).astype(np.float32)


def calibrate_probability(raw_prob, temperature):
    eps = 1e-7
    logits = np.log(np.clip(raw_prob,eps,1-eps)/np.clip(1-raw_prob,eps,1-eps))
    return float(1.0/(1.0+np.exp(-logits/temperature)))


# ════════════════════════════════════════════════════════════════
#  MAIN LIVE LOOP
# ════════════════════════════════════════════════════════════════
feature_window       = deque(maxlen=WINDOW_LEN)
step_log             = []
prev_samples         = 0
step                 = 0
t_start              = time.time()
prev_gt_label        = 0
mitigation_on        = False
first_block_step     = None
first_block_elapsed  = None
attack_start_step    = None
attack_start_elapsed = None

clear_all_mitigations()

print("╔══════════════════════════════════════════════════════════╗")
print("║   CNN+LSTM → DQN Live Mitigation Agent  v10             ║")
print(f"║   Duration={DRL_DURATION}s  Window={WINDOW_LEN}  Threshold={BEST_THRESHOLD:.4f}          ║")
print("║   ✅ Exact preprocess from 5b (fillna+regex+nan_to_num) ║")
print("║   ✅ gt_label from SAMPLE (synced with traffic)          ║")
print("║   ✅ ONOS REST DROP + auto-cleanup                       ║")
print("╚══════════════════════════════════════════════════════════╝")

try:
    while time.time() - t_start < DRL_DURATION:
        time.sleep(INTERVAL)

        with data_lock:
            snap = list(collected_samples)
        if len(snap) <= prev_samples:
            continue
        new = snap[prev_samples:]
        prev_samples = len(snap)

        for sample in new:
            elapsed = round(time.time() - t_start, 1)

            # ── 1. GROUND TRUTH — from SAMPLE (like 5b) ─────────
            #    The label in the sample was set by compute_features()
            #    at the SAME time traffic was collected → synced.
            #    We also read label_state for attacker identity.
            gt_label = int(sample.get('label', sample.get('attack_in_progress', 0)) or 0)
            _, gt_host_ls, gt_ip = read_label_state()

            # Use label_state only for host/IP identity, not the label itself
            if gt_label == 1:
                gt_host = gt_host_ls if gt_host_ls != "none" else "unknown"
            else:
                gt_host = "none"
                gt_ip   = "none"

            # ── 2. TRANSITIONS ───────────────────────────────────
            if prev_gt_label == 0 and gt_label == 1:
                attack_start_step = step
                attack_start_elapsed = elapsed
                first_block_step = None
                first_block_elapsed = None
                print(f"\n[DRL] ⬆️  ATTACK STARTED step {step} ({elapsed}s) — {gt_host}({gt_ip})")

            if prev_gt_label == 1 and gt_label == 0:
                print(f"\n[DRL] ⬇️  ATTACK ENDED step {step} ({elapsed}s)")
                if first_block_elapsed and attack_start_elapsed is not None:
                    print(f"[DRL]    Reaction: {first_block_elapsed - attack_start_elapsed:.1f}s")
                clear_all_mitigations()
                mitigation_on = False
                first_block_step = None
                attack_start_step = None

            prev_gt_label = gt_label

            # ── 3. PREPROCESS (EXACT 5b logic) ───────────────────
            sample_df = pd.DataFrame([sample])
            X_row = preprocess_live_sample(sample_df)
            feature_window.append(X_row[0])

            if len(feature_window) < WINDOW_LEN:
                print(f"  [Warmup] {len(feature_window)}/{WINDOW_LEN}")
                continue

            window_arr = np.array(list(feature_window), dtype=np.float32).reshape(
                1, WINDOW_LEN, N_LSTM_FEATS)

            # ── 4. LSTM INFERENCE ────────────────────────────────
            raw_prob = float(lstm_model.predict(window_arr, verbose=0).ravel()[0])
            cal_prob = calibrate_probability(raw_prob, TEMPERATURE)
            binary   = int(cal_prob >= BEST_THRESHOLD)

            # ── 5. DQN DECISION ──────────────────────────────────
            obs       = build_dqn_obs(cal_prob, sample)
            action, _ = dqn_agent.predict(obs, deterministic=True)
            action    = int(action)

            # ── 6. MITIGATION ────────────────────────────────────
            mitigated_ip = None
            action_str   = "ALLOW"

            if gt_label == 1:
                if mitigation_on:
                    action_str = f"HOLD_BLOCK ({','.join(blocked_ips)})"
                elif action > 0:
                    src_ip = str(sample.get(f"src{action}_ip","none")).strip()
                    if src_ip not in KNOWN_LEGIT and src_ip != "none":
                        mitigated_ip = src_ip
                        install_drop(src_ip, timeout=0)
                        mitigation_on = True
                        action_str = f"🎯 BLOCK src{action}({src_ip})"
                        if first_block_step is None:
                            first_block_step = step
                            first_block_elapsed = elapsed
                            reaction = elapsed - (attack_start_elapsed or elapsed)
                            print(f"[DRL] 🎯 First block step {step} ({elapsed}s) — reaction: {reaction:.1f}s")
                    else:
                        action_str = f"SKIP_LEGIT src{action}({src_ip})"
                elif not mitigation_on:
                    action_str = "SEARCHING..."
            else:
                if action > 0:
                    action_str = "FA_BLOCKED (ignored)"
                else:
                    action_str = "ALLOW ✅"

            # ── 7. METRICS ───────────────────────────────────────
            docker    = get_docker_stats()
            onos1_cpu = (docker.get("onos1") or {}).get("cpu", 0.0)
            onos2_cpu = (docker.get("onos2") or {}).get("cpu", 0.0)
            onos3_cpu = (docker.get("onos3") or {}).get("cpu", 0.0)
            ctrl_cpu  = (onos1_cpu + onos2_cpu + onos3_cpu) / 3

            total_packets   = int(sample.get("total_packets", 0))
            dropped_packets = get_dropped_packet_count() if mitigation_on else 0
            forwarded_pkts  = max(0, total_packets - dropped_packets)
            total_flows     = int(sample.get("total_flows", 0))
            total_bytes     = int(sample.get("total_bytes", 0))
            src_entropy     = float(sample.get("src_entropy",
                              sample.get("source_entropy", 0)) or 0)
            top_src_dom     = float(sample.get("top_source_dominance", 0) or 0)
            active_sources  = int(sample.get("active_sources", 0) or 0)
            flow_arr_rate   = float(sample.get("flow_arrival_rate", 0) or 0)

            # ── 8. PRINT ─────────────────────────────────────────
            correct  = "✅" if binary == gt_label else "❌"
            mit_icon = "🛡️" if mitigation_on else "  "
            prob_bar = "█" * int(cal_prob * 20) + "░" * (20 - int(cal_prob * 20))

            if mitigation_on:
                print(f"[{step:>4}] {elapsed:>6.1f}s {mit_icon} |{prob_bar}| "
                      f"P={cal_prob:.4f} {correct} gt={gt_label}({gt_host}) | "
                      f"{action_str} | fwd={forwarded_pkts} drop={dropped_packets} cpu={ctrl_cpu:.1f}%")
            else:
                print(f"[{step:>4}] {elapsed:>6.1f}s {mit_icon} |{prob_bar}| "
                      f"P={cal_prob:.4f} {correct} gt={gt_label}({gt_host}) | "
                      f"{action_str} | pkts={total_packets} cpu={ctrl_cpu:.1f}%")

            # ── 9. LOG ───────────────────────────────────────────
            step_log.append({
                "step": step, "elapsed_s": elapsed,
                "attack_prob": cal_prob, "raw_prob": raw_prob,
                "binary": binary, "gt_label": gt_label,
                "gt_host": gt_host, "gt_ip": gt_ip,
                "correct": int(binary == gt_label),
                "dqn_action": action, "action_taken": action_str,
                "mitigated_ip": mitigated_ip,
                "mitigation_active": int(mitigation_on),
                "blocked_ips": ",".join(blocked_ips) if blocked_ips else "",
                "total_packets": total_packets,
                "forwarded_packets": forwarded_pkts,
                "dropped_packets": dropped_packets,
                "total_flows": total_flows,
                "total_bytes": total_bytes,
                "s1_packets": int(sample.get("s1_packets",0) or 0),
                "s2_packets": int(sample.get("s2_packets",0) or 0),
                "s3_packets": int(sample.get("s3_packets",0) or 0),
                "s4_packets": int(sample.get("s4_packets",0) or 0),
                "flow_arrival_rate": flow_arr_rate,
                "src_entropy": src_entropy,
                "top_source_dominance": top_src_dom,
                "active_sources": active_sources,
                "unique_srcs": int(sample.get("unique_srcs",0) or 0),
                "onos1_cpu": onos1_cpu, "onos2_cpu": onos2_cpu,
                "onos3_cpu": onos3_cpu, "ctrl_cpu_avg": ctrl_cpu,
                "attack_start_elapsed": attack_start_elapsed,
                "first_block_elapsed": first_block_elapsed,
            })
            step += 1

except KeyboardInterrupt:
    print(f"\n🛑 Stopped at step {step}")

# ── FINAL CLEANUP ────────────────────────────────────────────────
clear_all_mitigations()
elapsed_total = time.time() - t_start
print(f"\n⏱️  Total: {step} steps in {elapsed_total:.0f}s")

if step_log:
    df_log = pd.DataFrame(step_log)
    df_log.to_csv(f"{SAVE_DIR}/live_drl_log.csv", index=False)
    print(f"💾 Saved: {SAVE_DIR}/live_drl_log.csv ({len(df_log)} rows)")

    from sklearn.metrics import f1_score, precision_score, recall_score
    f1  = f1_score(df_log["gt_label"], df_log["binary"], zero_division=0)
    pre = precision_score(df_log["gt_label"], df_log["binary"], zero_division=0)
    rec = recall_score(df_log["gt_label"], df_log["binary"], zero_division=0)

    atk = df_log[df_log["gt_label"]==1]
    ben = df_log[df_log["gt_label"]==0]

    print("\n══════════════════════════════════════════════════")
    print("  LIVE DRL v10 — RESULTS SUMMARY")
    print("══════════════════════════════════════════════════")
    print(f"  LSTM Detection F1:  {f1:.4f}")
    print(f"  Precision:          {pre:.4f}")
    print(f"  Recall:             {rec:.4f}")
    print(f"  Total mitigations:  {df_log['mitigated_ip'].notna().sum()} "
          f"({df_log['mitigated_ip'].nunique()} unique IPs)")

    if first_block_elapsed is not None and attack_start_elapsed is not None:
        print(f"  Reaction time:      {first_block_elapsed - attack_start_elapsed:.1f}s")

    print(f"\n  ─── Traffic Analysis ───")
    if len(atk):
        pre_mit  = atk[atk['mitigation_active']==0]
        post_mit = atk[atk['mitigation_active']==1]
        if len(pre_mit):
            print(f"  Pkts (pre-mitigation):   {pre_mit['total_packets'].mean():.0f}")
        if len(post_mit):
            print(f"  Pkts (post-mitigation):  {post_mit['total_packets'].mean():.0f}")
            print(f"  Forwarded (post-mit):    {post_mit['forwarded_packets'].mean():.0f}")
            print(f"  Dropped (post-mit):      {post_mit['dropped_packets'].mean():.0f}")
    if len(ben):
        print(f"  Pkts (benign):           {ben['total_packets'].mean():.0f}")

    print(f"\n  ─── Controller Load ───")
    if len(atk): print(f"  CPU (attack):   {atk['ctrl_cpu_avg'].mean():.1f}%")
    if len(ben): print(f"  CPU (benign):   {ben['ctrl_cpu_avg'].mean():.1f}%")
    mit_rows = df_log[df_log['mitigation_active']==1]
    if len(mit_rows): print(f"  CPU (w/ mit):   {mit_rows['ctrl_cpu_avg'].mean():.1f}%")

    ben_fa = df_log[(df_log['gt_label']==0) & (df_log['dqn_action']>0)]
    print(f"\n  ─── Benign Safety ───")
    print(f"  False alarms: {len(ben_fa)} (all ignored)")
    print(f"  Benign:       untouched ✅")

    if len(mit_rows):
        print(f"\n  ─── LSTM During Mitigation ───")
        print(f"  Avg prob:  {mit_rows['attack_prob'].mean():.4f}")
        below = len(mit_rows[mit_rows['attack_prob'] < BEST_THRESHOLD])
        print(f"  Steps below threshold: {below}/{len(mit_rows)}")

    print("══════════════════════════════════════════════════")


2026-03-04 12:25:38.679788: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
Gym has been unmaintained since 2022 and does not support NumPy 2.0 amongst other critical functionality.
Please upgrade to Gymnasium, the maintained drop-in replacement of Gym, or contact the authors of your software and request that they upgrade.
Users of this version of Gym should be able to simply replace 'import gym' with 'import gymnasium as gym' in the vast majority of cases.
See the migration guide at https://gymnasium.farama.org/introduction/migration_guide/ for additional information.
I0000 00:00:1772609143.026712    8431 gpu_device.cc:2020] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 2158 MB memory:  -> device: 0, name: NVIDIA GeForce GTX 1650, pci

✅ LSTM loaded — window=5, threshold=0.8472, T=0.9568
✅ DQN loaded — obs_dim=60, actions=7
[DRL] 🧹 Mitigations cleared
╔══════════════════════════════════════════════════════════╗
║   CNN+LSTM → DQN Live Mitigation Agent  v10             ║
║   Duration=600s  Window=5  Threshold=0.8472          ║
║   ✅ Exact preprocess from 5b (fillna+regex+nan_to_num) ║
║   ✅ gt_label from SAMPLE (synced with traffic)          ║
║   ✅ ONOS REST DROP + auto-cleanup                       ║
╚══════════════════════════════════════════════════════════╝
  [Warmup] 1/5
  [Warmup] 2/5
  [Warmup] 3/5
  [Warmup] 4/5


2026-03-04 12:25:49.162279: I external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:473] Loaded cuDNN version 91002


[   0]    2.3s    |░░░░░░░░░░░░░░░░░░░░| P=0.0034 ✅ gt=0(none) | ALLOW ✅ | pkts=1668 cpu=5.4%
[   1]    5.2s    |░░░░░░░░░░░░░░░░░░░░| P=0.0075 ✅ gt=0(none) | ALLOW ✅ | pkts=1760 cpu=4.3%
[   2]    7.2s    |░░░░░░░░░░░░░░░░░░░░| P=0.0063 ✅ gt=0(none) | ALLOW ✅ | pkts=1833 cpu=9.7%
[   3]    9.3s    |░░░░░░░░░░░░░░░░░░░░| P=0.0026 ✅ gt=0(none) | ALLOW ✅ | pkts=1957 cpu=5.6%
[   4]   11.3s    |░░░░░░░░░░░░░░░░░░░░| P=0.0020 ✅ gt=0(none) | ALLOW ✅ | pkts=2006 cpu=2.8%
[   5]   13.3s    |░░░░░░░░░░░░░░░░░░░░| P=0.0012 ✅ gt=0(none) | ALLOW ✅ | pkts=2091 cpu=6.1%
[   6]   17.4s    |░░░░░░░░░░░░░░░░░░░░| P=0.0018 ✅ gt=0(none) | ALLOW ✅ | pkts=2159 cpu=7.7%
[   7]   19.4s    |░░░░░░░░░░░░░░░░░░░░| P=0.0016 ✅ gt=0(none) | ALLOW ✅ | pkts=2232 cpu=2.0%
[   8]   21.5s    |░░░░░░░░░░░░░░░░░░░░| P=0.0014 ✅ gt=0(none) | ALLOW ✅ | pkts=1135 cpu=4.2%
[   9]   23.5s    |░░░░░░░░░░░░░░░░░░░░| P=0.0011 ✅ gt=0(none) | ALLOW ✅ | pkts=1201 cpu=4.9%
[  10]   25.5s    |░░░░░░░░░░░░░░░░░░░░| P=0.0009 ✅ gt=0(non

In [20]:
stop_monitoring()

✓ Monitoring stopped. 269 samples saved to: DRL_test.csv


In [19]:
# ============================================================================
# LIVE DRL AGENT v14 — aligned + delta metrics + latency hardening + CI gates
# ============================================================================
import json, os, threading, time
import numpy as np
import pandas as pd
import psutil
import tensorflow as tf
from collections import deque

# ----------------------------- Runtime knobs ---------------------------------
GT_MODE = "sample_or_file"   # "file_only" | "sample_or_file"
INTERVAL = float(globals().get("INTERVAL", 2))
DROP_QUERY_EVERY = int(globals().get("DROP_QUERY_EVERY", 3))   # query ONOS drop counter every N steps
DOCKER_CACHE_SEC = float(globals().get("DOCKER_CACHE_SEC", 3))

# Tune these to your lab SLOs
CI_LIMITS = {
    "p95_step_total_ms": 8000.0,
    "p99_step_total_ms": 30000.0,
    "p95_lstm_inference_ms": 5000.0,
    "max_backlog_samples": 6,
    "gt_mismatch_rate": 0.10,
}

# ------------------------ Optional TF latency stability -----------------------
try:
    tf.config.threading.set_intra_op_parallelism_threads(2)
    tf.config.threading.set_inter_op_parallelism_threads(2)
except Exception:
    pass

@tf.function(reduce_retracing=True)
def _lstm_forward(x):
    return lstm_model(x, training=False)

# Warmup compile
_ = _lstm_forward(tf.zeros((1, WINDOW_LEN, N_LSTM_FEATS), dtype=tf.float32))

# ------------------------ Non-blocking flow flushing --------------------------
_flush_threads = {}
_flush_threads_lock = threading.Lock()

def _flush_worker(attacker_ip):
    try:
        flush_attacker_flows(attacker_ip)
    except Exception as e:
        print(f"[DRL][WARN] async flush failed for {attacker_ip}: {e}")
    finally:
        with _flush_threads_lock:
            _flush_threads.pop(attacker_ip, None)

def install_drop(attacker_ip, timeout=0, async_flush=True):
    """Override: install DROP fast, flush old flows asynchronously."""
    with mit_lock:
        if attacker_ip in active_mit and active_mit[attacker_ip] > time.time():
            return 0.0
        active_mit[attacker_ip] = time.time() + 600
        blocked_ips.add(attacker_ip)

    t0 = time.perf_counter()
    ok = 0
    for dpid in SWITCH_DPIDS:
        master = get_master_controller(dpid)
        flow = {
            "priority": DROP_PRIO,
            "isPermanent": (timeout == 0),
            "deviceId": dpid,
            "treatment": {"instructions": []},
            "selector": {"criteria": [
                {"type": "ETH_TYPE", "ethType": "0x0800"},
                {"type": "IPV4_SRC", "ip": f"{attacker_ip}/32"},
            ]},
        }
        if timeout > 0:
            flow["timeout"] = timeout
            flow["isPermanent"] = False

        try:
            r = requests.post(
                f"http://{master['ip']}:{master['port']}/onos/v1/flows",
                json={"flows": [flow]}, auth=ONOS_AUTH, timeout=3
            )
            if r.ok:
                ok += 1
        except Exception:
            pass

    api_ms = (time.perf_counter() - t0) * 1000.0
    ttl = f"TTL={timeout}s" if timeout > 0 else "PERMANENT"
    print(f"[DRL] DROP: {attacker_ip} on {ok}/4 switches (prio={DROP_PRIO}, {ttl}) [{api_ms:.0f}ms]")

    if async_flush:
        with _flush_threads_lock:
            th = _flush_threads.get(attacker_ip)
            if th is None or not th.is_alive():
                th = threading.Thread(target=_flush_worker, args=(attacker_ip,), daemon=True)
                _flush_threads[attacker_ip] = th
                th.start()
    else:
        flush_attacker_flows(attacker_ip)

    return api_ms

# ------------------------------- Helpers --------------------------------------
def _safe_int(v, default=None):
    try:
        return int(v)
    except Exception:
        return default

def _counter_delta(curr, prev):
    # Handle monotonic counter reset gracefully
    if prev is None:
        return max(0, curr)
    if curr >= prev:
        return curr - prev
    return max(0, curr)

def safe_get_docker_stats():
    fn = globals().get("get_docker_stats", None)
    if callable(fn):
        try:
            return fn()
        except Exception:
            return {}
    return {}

def resolve_ground_truth(sample):
    file_label, file_host, file_ip = read_label_state()

    sample_label = _safe_int(sample.get("label", None), default=None)
    sample_host = str(sample.get("label_host", sample.get("host", "")) or "")
    sample_ip = str(sample.get("label_ip", "") or "")

    if GT_MODE == "file_only" or sample_label not in (0, 1):
        return file_label, file_host, file_ip, file_label, sample_label, "file"

    if sample_label == 0:
        return 0, "none", "none", file_label, sample_label, "sample"

    # sample_label == 1
    if sample_host.lower() in ("", "none", "unknown"):
        sample_host = file_host if file_label == 1 else "unknown"
    if sample_ip.lower() in ("", "none", "unknown"):
        sample_ip = file_ip if file_label == 1 else "none"

    return 1, sample_host, sample_ip, file_label, sample_label, "sample"

# ------------------------------- Main loop ------------------------------------
feature_window = deque(maxlen=WINDOW_LEN)
step_log = []
prev_samples = 0
step = 0
t_start = time.time()

prev_gt_label = 0
mitigation_on = False
first_block_step = None
first_block_elapsed = None
attack_start_step = None
attack_start_elapsed = None
reaction_times = []

# Counter state (absolute + delta)
prev_total_packets_abs = None
prev_dropped_packets_abs = 0
last_dropped_packets_abs = 0
drop_counter_stale_steps = 0

# Telemetry cache
docker_cache = {}
docker_cache_ts = 0.0

process = psutil.Process(os.getpid())
_ = process.cpu_percent(interval=None)
_ = psutil.cpu_percent(interval=None)

clear_all_mitigations()

print("╔══════════════════════════════════════════════════════════╗")
print("║   CNN+LSTM -> DQN Live Mitigation Agent v14             ║")
print(f"║   Duration={DRL_DURATION}s  Window={WINDOW_LEN}  Threshold={BEST_THRESHOLD:.4f}          ║")
print("║   + Delta traffic metrics (interval-true)               ║")
print("║   + Non-blocking flush after DROP                        ║")
print("║   + Cached telemetry + TF compiled inference             ║")
print("║   + CI-style runtime gates                               ║")
print("╚══════════════════════════════════════════════════════════╝")

try:
    while time.time() - t_start < DRL_DURATION:
        time.sleep(INTERVAL)

        with data_lock:
            snap = list(collected_samples)

        n_total = len(snap)
        backlog = n_total - prev_samples
        if backlog <= 0:
            continue

        new_samples = snap[prev_samples:n_total]
        prev_samples = n_total

        step_t0 = time.perf_counter()
        elapsed = round(time.time() - t_start, 1)

        # 1) Batch preprocess all new samples (5b style)
        t_pre = time.perf_counter()
        df_new = pd.DataFrame(new_samples)
        X_new = preprocess_live_sample(df_new)
        preprocess_ms = (time.perf_counter() - t_pre) * 1000.0

        for row in X_new:
            feature_window.append(row)

        if len(feature_window) < WINDOW_LEN:
            print(f"  [Warmup] {len(feature_window)}/{WINDOW_LEN} (backlog={backlog})")
            continue

        # Latest sample only for decision
        sample = new_samples[-1]
        sample_id = _safe_int(sample.get("sample_id", step), default=step)

        # 2) Ground truth
        gt_label, gt_host, gt_ip, gt_file_label, gt_sample_label, gt_source = resolve_ground_truth(sample)
        gt_mismatch = int((gt_sample_label in (0, 1)) and (gt_sample_label != gt_file_label))
        if gt_mismatch:
            print(f"[WARN] GT mismatch sample={gt_sample_label} file={gt_file_label} (using {gt_source})")

        # 3) Label transitions
        if prev_gt_label == 0 and gt_label == 1:
            attack_start_step = step
            attack_start_elapsed = elapsed
            first_block_step = None
            first_block_elapsed = None
            print(f"\n[DRL] ATTACK STARTED step {step} ({elapsed}s) — {gt_host}({gt_ip})")

        if prev_gt_label == 1 and gt_label == 0:
            print(f"\n[DRL] ATTACK ENDED step {step} ({elapsed}s)")
            if first_block_elapsed is not None and attack_start_elapsed is not None:
                print(f"[DRL] Reaction: {first_block_elapsed - attack_start_elapsed:.1f}s")
            clear_all_mitigations()
            mitigation_on = False
            first_block_step = None
            attack_start_step = None
            prev_dropped_packets_abs = 0
            last_dropped_packets_abs = 0
            drop_counter_stale_steps = 0

        prev_gt_label = gt_label

        # 4) LSTM inference (compiled)
        window_arr = np.array(feature_window, dtype=np.float32).reshape(1, WINDOW_LEN, N_LSTM_FEATS)
        t_lstm = time.perf_counter()
        pred_tensor = _lstm_forward(tf.convert_to_tensor(window_arr, dtype=tf.float32))
        raw_prob = float(tf.reshape(pred_tensor, [-1])[0].numpy())
        cal_prob = calibrate_probability(raw_prob, TEMPERATURE)
        binary = int(cal_prob >= BEST_THRESHOLD)
        lstm_ms = (time.perf_counter() - t_lstm) * 1000.0

        # 5) DQN action
        t_dqn = time.perf_counter()
        obs = build_dqn_obs(cal_prob, sample)
        action, _ = dqn_agent.predict(obs, deterministic=True)
        action = int(action)
        dqn_ms = (time.perf_counter() - t_dqn) * 1000.0

        # 6) Mitigation action
        mitigated_ip = None
        action_str = "ALLOW"
        api_ms = 0.0

        if gt_label == 1:
            if mitigation_on:
                action_str = f"HOLD_BLOCK ({','.join(sorted(blocked_ips))})"
            elif action > 0:
                src_ip = str(sample.get(f"src{action}_ip", "none")).strip()
                if src_ip not in KNOWN_LEGIT and src_ip != "none":
                    mitigated_ip = src_ip
                    api_ms = install_drop(src_ip, timeout=0, async_flush=True)
                    mitigation_on = True
                    action_str = f"BLOCK src{action}({src_ip})"
                    if first_block_step is None:
                        first_block_step = step
                        first_block_elapsed = elapsed
                        reaction = elapsed - (attack_start_elapsed or elapsed)
                        reaction_times.append(reaction)
                        print(f"[DRL] First block step {step} ({elapsed}s) — reaction: {reaction:.1f}s")
                else:
                    action_str = f"SKIP_LEGIT src{action}({src_ip})"
            else:
                action_str = "SEARCHING..."
        else:
            action_str = "FA_BLOCKED (ignored)" if action > 0 else "ALLOW"

        # 7) Telemetry
        now = time.time()
        if now - docker_cache_ts >= DOCKER_CACHE_SEC:
            docker_cache = safe_get_docker_stats()
            docker_cache_ts = now
        docker = docker_cache or {}

        onos1_cpu = float((docker.get("onos1") or {}).get("cpu", 0.0))
        onos2_cpu = float((docker.get("onos2") or {}).get("cpu", 0.0))
        onos3_cpu = float((docker.get("onos3") or {}).get("cpu", 0.0))
        ctrl_cpu = (onos1_cpu + onos2_cpu + onos3_cpu) / 3.0

        py_cpu = float(process.cpu_percent(interval=None))
        sys_cpu = float(psutil.cpu_percent(interval=None))
        mem_mb = process.memory_info().rss / (1024 * 1024)

        # 8) Traffic counters (absolute + delta)
        total_packets_abs = int(sample.get("total_packets", 0) or 0)

        drop_query_ms = 0.0
        if mitigation_on:
            need_drop_query = (
                (step % DROP_QUERY_EVERY == 0)
                or (first_block_step == step)
                or (last_dropped_packets_abs == 0)
            )
            if need_drop_query:
                dropped_packets_abs, drop_query_ms = get_dropped_packet_count()
                last_dropped_packets_abs = int(dropped_packets_abs)
                drop_counter_stale_steps = 0
            else:
                dropped_packets_abs = int(last_dropped_packets_abs)
                drop_counter_stale_steps += 1
        else:
            dropped_packets_abs = 0
            last_dropped_packets_abs = 0
            drop_counter_stale_steps = 0

        total_counter_reset = int(prev_total_packets_abs is not None and total_packets_abs < prev_total_packets_abs)
        drop_counter_reset = int(prev_dropped_packets_abs is not None and dropped_packets_abs < prev_dropped_packets_abs)

        total_packets_delta = _counter_delta(total_packets_abs, prev_total_packets_abs)
        dropped_packets_delta = _counter_delta(dropped_packets_abs, prev_dropped_packets_abs) if mitigation_on else 0
        forwarded_packets_delta = max(0, total_packets_delta - dropped_packets_delta)

        forwarded_packets_abs = max(0, total_packets_abs - dropped_packets_abs)

        prev_total_packets_abs = total_packets_abs
        prev_dropped_packets_abs = dropped_packets_abs

        total_flows = int(sample.get("total_flows", 0) or 0)
        total_bytes = int(sample.get("total_bytes", 0) or 0)
        src_entropy = float(sample.get("src_entropy", sample.get("source_entropy", 0)) or 0.0)
        top_src_dom = float(sample.get("top_source_dominance", 0) or 0.0)
        active_sources = int(sample.get("active_sources", 0) or 0)
        flow_arr_rate = float(sample.get("flow_arrival_rate", 0) or 0.0)

        step_total_ms = (time.perf_counter() - step_t0) * 1000.0

        # 9) Print
        correct = "OK" if binary == gt_label else "ERR"
        mit = "ON " if mitigation_on else "OFF"
        lag_tag = f" backlog={backlog}" if backlog > 1 else ""

        if mitigation_on:
            print(
                f"[{step:>4}] {elapsed:>6.1f}s mit={mit} P={cal_prob:.4f} {correct} gt={gt_label}({gt_host}) | "
                f"{action_str} | dFWD={forwarded_packets_delta} dDROP={dropped_packets_delta} "
                f"absFWD={forwarded_packets_abs} absDROP={dropped_packets_abs} "
                f"cpu_ctrl={ctrl_cpu:.1f}% cpu_py={py_cpu:.1f}% [{step_total_ms:.0f}ms]{lag_tag}"
            )
        else:
            print(
                f"[{step:>4}] {elapsed:>6.1f}s mit={mit} P={cal_prob:.4f} {correct} gt={gt_label}({gt_host}) | "
                f"{action_str} | dPKT={total_packets_delta} absPKT={total_packets_abs} "
                f"cpu_ctrl={ctrl_cpu:.1f}% cpu_py={py_cpu:.1f}% [{step_total_ms:.0f}ms]{lag_tag}"
            )

        # 10) Log
        step_log.append({
            "step": step,
            "sample_id": sample_id,
            "elapsed_s": elapsed,

            "backlog_samples": backlog,
            "gt_source": gt_source,
            "gt_file_label": gt_file_label,
            "gt_sample_label": gt_sample_label,
            "gt_mismatch": gt_mismatch,

            "attack_prob": cal_prob,
            "raw_prob": raw_prob,
            "binary": binary,
            "gt_label": gt_label,
            "gt_host": gt_host,
            "gt_ip": gt_ip,
            "correct": int(binary == gt_label),

            "dqn_action": action,
            "action_taken": action_str,
            "mitigated_ip": mitigated_ip,
            "mitigation_active": int(mitigation_on),
            "blocked_ips": ",".join(sorted(blocked_ips)) if blocked_ips else "",

            # Absolute counters
            "total_packets": total_packets_abs,
            "forwarded_packets": forwarded_packets_abs,
            "dropped_packets": dropped_packets_abs,

            # Interval-true deltas
            "total_packets_delta": total_packets_delta,
            "forwarded_packets_delta": forwarded_packets_delta,
            "dropped_packets_delta": dropped_packets_delta,
            "drop_counter_stale_steps": drop_counter_stale_steps,
            "total_counter_reset": total_counter_reset,
            "drop_counter_reset": drop_counter_reset,

            "total_flows": total_flows,
            "total_bytes": total_bytes,
            "s1_packets": int(sample.get("s1_packets", 0) or 0),
            "s2_packets": int(sample.get("s2_packets", 0) or 0),
            "s3_packets": int(sample.get("s3_packets", 0) or 0),
            "s4_packets": int(sample.get("s4_packets", 0) or 0),
            "flow_arrival_rate": flow_arr_rate,
            "src_entropy": src_entropy,
            "top_source_dominance": top_src_dom,
            "active_sources": active_sources,
            "unique_srcs": int(sample.get("unique_srcs", 0) or 0),

            "onos1_cpu": onos1_cpu,
            "onos2_cpu": onos2_cpu,
            "onos3_cpu": onos3_cpu,
            "ctrl_cpu_avg": ctrl_cpu,
            "python_cpu_pct": py_cpu,
            "system_cpu_pct": sys_cpu,
            "python_mem_mb": round(mem_mb, 1),

            "preprocess_ms": round(preprocess_ms, 2),
            "lstm_inference_ms": round(lstm_ms, 2),
            "dqn_inference_ms": round(dqn_ms, 2),
            "api_latency_ms": round(api_ms, 2),
            "drop_query_ms": round(drop_query_ms, 2),
            "step_total_ms": round(step_total_ms, 2),

            "attack_start_elapsed": attack_start_elapsed,
            "first_block_elapsed": first_block_elapsed,
        })
        step += 1

except KeyboardInterrupt:
    print(f"\nStopped at step {step}")

# ------------------------------- Final cleanup --------------------------------
clear_all_mitigations()
elapsed_total = time.time() - t_start
print(f"\nTotal: {step} steps in {elapsed_total:.0f}s")

if step_log:
    df_log = pd.DataFrame(step_log)
    csv_path = f"{SAVE_DIR}/live_drl_log_2.csv"
    df_log.to_csv(csv_path, index=False)
    print(f"Saved: {csv_path} ({len(df_log)} rows)")

    from sklearn.metrics import f1_score, precision_score, recall_score
    f1 = f1_score(df_log["gt_label"], df_log["binary"], zero_division=0)
    pre = precision_score(df_log["gt_label"], df_log["binary"], zero_division=0)
    rec = recall_score(df_log["gt_label"], df_log["binary"], zero_division=0)

    atk = df_log[df_log["gt_label"] == 1]
    ben = df_log[df_log["gt_label"] == 0]
    mit_rows = df_log[df_log["mitigation_active"] == 1]

    # GT mismatch rate
    valid_mask = df_log["gt_sample_label"].isin([0, 1]) & df_log["gt_file_label"].isin([0, 1])
    mismatch_total = int(valid_mask.sum())
    mismatch_count = int((df_log.loc[valid_mask, "gt_sample_label"] != df_log.loc[valid_mask, "gt_file_label"]).sum()) if mismatch_total else 0
    mismatch_rate = (mismatch_count / mismatch_total) if mismatch_total else 0.0

    print("\n" + "=" * 58)
    print(" LIVE DRL v14 — RESULTS SUMMARY")
    print("=" * 58)
    print(f" LSTM Detection F1:   {f1:.4f}")
    print(f" Precision:           {pre:.4f}")
    print(f" Recall:              {rec:.4f}")
    print(f" Total mitigations:   {df_log['mitigated_ip'].notna().sum()} ({df_log['mitigated_ip'].nunique()} unique IPs)")
    if reaction_times:
        print(f" Reaction times (s):  min={np.min(reaction_times):.1f}  mean={np.mean(reaction_times):.1f}  max={np.max(reaction_times):.1f}")

    print("\n Traffic (interval deltas)")
    if len(atk):
        print(f" Attack dPKT avg:     {atk['total_packets_delta'].mean():.1f}")
        print(f" Attack dFWD avg:     {atk['forwarded_packets_delta'].mean():.1f}")
        print(f" Attack dDROP avg:    {atk['dropped_packets_delta'].mean():.1f}")
    if len(ben):
        print(f" Benign dPKT avg:     {ben['total_packets_delta'].mean():.1f}")
    if len(mit_rows):
        drop_ratio = mit_rows["dropped_packets_delta"].sum() / max(1.0, mit_rows["total_packets_delta"].sum())
        print(f" Mitigation drop ratio: {drop_ratio:.3f}")

    print("\n CPU")
    print(f" Controller CPU avg:  {df_log['ctrl_cpu_avg'].mean():.1f}%")
    print(f" Python CPU avg:      {df_log['python_cpu_pct'].mean():.1f}%")
    print(f" Python mem (last):   {df_log['python_mem_mb'].iloc[-1]:.0f} MB")

    print("\n Latency")
    print(f" Preprocess avg/max:  {df_log['preprocess_ms'].mean():.1f} / {df_log['preprocess_ms'].max():.1f} ms")
    print(f" LSTM avg/max:        {df_log['lstm_inference_ms'].mean():.1f} / {df_log['lstm_inference_ms'].max():.1f} ms")
    print(f" DQN avg/max:         {df_log['dqn_inference_ms'].mean():.1f} / {df_log['dqn_inference_ms'].max():.1f} ms")
    api_calls = df_log[df_log["api_latency_ms"] > 0]
    if len(api_calls):
        print(f" ONOS API avg/max:    {api_calls['api_latency_ms'].mean():.1f} / {api_calls['api_latency_ms'].max():.1f} ms")
    dq = df_log[df_log["drop_query_ms"] > 0]
    if len(dq):
        print(f" Drop query avg/max:  {dq['drop_query_ms'].mean():.1f} / {dq['drop_query_ms'].max():.1f} ms")
    print(f" Step avg/max:        {df_log['step_total_ms'].mean():.1f} / {df_log['step_total_ms'].max():.1f} ms")

    print("\n Alignment / backlog")
    print(f" GT mismatches:       {mismatch_count}/{mismatch_total} ({mismatch_rate:.2%})")
    print(f" Avg backlog:         {df_log['backlog_samples'].mean():.2f}")
    print(f" Max backlog:         {int(df_log['backlog_samples'].max())}")

    # ----------------------------- CI gate report -----------------------------
    ci_metrics = {
        "p95_step_total_ms": float(df_log["step_total_ms"].quantile(0.95)),
        "p99_step_total_ms": float(df_log["step_total_ms"].quantile(0.99)),
        "p95_lstm_inference_ms": float(df_log["lstm_inference_ms"].quantile(0.95)),
        "max_backlog_samples": int(df_log["backlog_samples"].max()),
        "gt_mismatch_rate": float(mismatch_rate),
    }

    checks = []
    for k, lim in CI_LIMITS.items():
        val = ci_metrics[k]
        ok = val <= lim
        checks.append({"metric": k, "value": float(val), "limit": float(lim), "ok": bool(ok)})

    ci_pass = all(c["ok"] for c in checks)

    print("\n CI Gates")
    for c in checks:
        status = "PASS" if c["ok"] else "FAIL"
        print(f" {status:4s} {c['metric']}: {c['value']:.4f} (limit {c['limit']})")
    print(f" CI overall: {'PASS' if ci_pass else 'FAIL'}")

    ci_report = {
        "ci_pass": ci_pass,
        "limits": CI_LIMITS,
        "metrics": ci_metrics,
        "checks": checks,
        "rows": int(len(df_log)),
        "generated_at_epoch": time.time(),
    }
    ci_path = f"{SAVE_DIR}/live_drl_ci_report.json"
    with open(ci_path, "w") as f:
        json.dump(ci_report, f, indent=2)
    print(f" Saved CI report: {ci_path}")
    print("=" * 58)


[DRL] 🧹 Mitigations cleared
╔══════════════════════════════════════════════════════════╗
║   CNN+LSTM -> DQN Live Mitigation Agent v14             ║
║   Duration=600s  Window=5  Threshold=0.8472          ║
║   + Delta traffic metrics (interval-true)               ║
║   + Non-blocking flush after DROP                        ║
║   + Cached telemetry + TF compiled inference             ║
║   + CI-style runtime gates                               ║
╚══════════════════════════════════════════════════════════╝
[   0]    2.1s mit=OFF P=0.0008 OK gt=0(none) | ALLOW | dPKT=1854 absPKT=1854 cpu_ctrl=4.7% cpu_py=6.7% [1919ms] backlog=8
[   1]    6.0s mit=OFF P=0.0008 OK gt=0(none) | ALLOW | dPKT=310 absPKT=2164 cpu_ctrl=2.5% cpu_py=7.4% [2065ms] backlog=2
[   2]   10.1s mit=OFF P=0.0008 OK gt=0(none) | ALLOW | dPKT=1885 absPKT=1885 cpu_ctrl=4.7% cpu_py=6.6% [2060ms] backlog=2
[   3]   14.2s mit=OFF P=0.0009 OK gt=0(none) | ALLOW | dPKT=326 absPKT=2211 cpu_ctrl=7.3% cpu_py=6.4% [2068ms] backlog=2


In [7]:
# =====================================================================
# LIVE DRL NO-MITIGATION (self-contained)
# - Loads detector + DQN artifacts from disk
# - Uses collected_samples/data_lock from monitoring cell
# - NEVER applies mitigation; logs WOULD_BLOCK only
# =====================================================================
import os
import time
import json
import pickle
from pathlib import Path
from collections import deque

import numpy as np
import pandas as pd
import psutil
import tensorflow as tf
from stable_baselines3 import DQN
from sklearn.metrics import f1_score, precision_score, recall_score

# ---------------------------- Resolve paths ----------------------------
CANDIDATE_BASES = [
    Path.cwd(),
    Path.cwd() / "Models/newTrail",
    Path("/home/shokran/PycharmProjects/JupyterProject/Models/newTrail"),
]
BASE = None
for b in CANDIDATE_BASES:
    if (b / "outputs_detector" / "config.json").exists():
        BASE = b
        break
if BASE is None:
    raise RuntimeError("Could not find outputs_detector/config.json. Set BASE manually.")

DETECTOR_DIR = BASE / "outputs_detector"
DRL_DIR = BASE / "outputs_ppo_dqn_compare"
SAVE_DIR = BASE / "outputs_drl_live_nomit"
SAVE_DIR.mkdir(parents=True, exist_ok=True)

# ---------------------------- Runtime config ---------------------------
DRL_DURATION = int(globals().get("DRL_DURATION", 600))
INTERVAL = float(globals().get("INTERVAL", 2))
GT_MODE = "sample_or_file"  # "file_only" | "sample_or_file"

# ---------------------------- Required live feed -----------------------
if "data_lock" not in globals() or "collected_samples" not in globals():
    raise RuntimeError("Run your monitoring/collector cell first (data_lock + collected_samples required).")

# ---------------------------- Load detector ----------------------------
with open(DETECTOR_DIR / "config.json", "r") as f:
    det_cfg = json.load(f)

LSTM_FEATURES = det_cfg["feature_cols"]
N_LSTM_FEATS = int(det_cfg.get("n_features", len(LSTM_FEATURES)))
WINDOW_LEN = int(det_cfg["BEST_WINDOW_LEN"])
BEST_THRESHOLD = float(det_cfg["BEST_THRESHOLD"])

with open(DETECTOR_DIR / "calibration.json", "r") as f:
    TEMPERATURE = float(json.load(f)["temperature"])
with open(DETECTOR_DIR / "imputer.pkl", "rb") as f:
    lstm_imputer = pickle.load(f)
with open(DETECTOR_DIR / "scaler.pkl", "rb") as f:
    lstm_scaler = pickle.load(f)

tf.get_logger().setLevel("ERROR")
lstm_model = tf.keras.models.load_model(DETECTOR_DIR / "model_final.keras", compile=False)

# ---------------------------- Load DQN ---------------------------------
with open(DRL_DIR / "deployment_recipe.json", "r") as f:
    recipe = json.load(f)
with open(DRL_DIR / "obs_scaler.pkl", "rb") as f:
    dqn_scaler = pickle.load(f)

dqn_agent = DQN.load(str(DRL_DIR / "dqn_model.zip"))
TOP_K = int(recipe.get("top_k", 6))

SRC_NUM_FEATS = [
    "packets", "bytes", "flows", "pps", "syn_ratio",
    "dst_diversity", "top_dst_share", "avg_duration", "pkt_share"
]
GLOBAL_FEATS = ["active_sources", "source_entropy", "top_source_dominance"]
KNOWN_LEGIT = {"10.10.0.1", "10.10.0.3", "10.10.0.200", "10.10.0.201", "none", ""}

print(f"[NO-MIT] Loaded LSTM window={WINDOW_LEN}, threshold={BEST_THRESHOLD:.4f}, T={TEMPERATURE:.4f}")
print(f"[NO-MIT] Loaded DQN actions={TOP_K + 1}")

# ---------------------------- Helpers ----------------------------------
def preprocess_live_sample(raw_df):
    df = raw_df.copy()
    for c in LSTM_FEATURES:
        if c not in df.columns:
            df[c] = 0.0
    for c in LSTM_FEATURES:
        if df[c].dtype == "object":
            ex = df[c].astype(str).str.extract(r"([-+]?\d*\.?\d+)")[0]
            df[c] = pd.to_numeric(ex, errors="coerce")
        else:
            df[c] = pd.to_numeric(df[c], errors="coerce")
        df[c] = df[c].replace([np.inf, -np.inf], np.nan).fillna(0.0)

    X = df[LSTM_FEATURES].values.astype(np.float64)
    if lstm_imputer is not None:
        X = lstm_imputer.transform(X)
    if lstm_scaler is not None:
        X = lstm_scaler.transform(X)
    X = np.nan_to_num(X, nan=0.0, posinf=0.0, neginf=0.0)
    return X.astype(np.float32)

def calibrate_probability(raw_prob, temperature):
    eps = 1e-7
    logits = np.log(np.clip(raw_prob, eps, 1 - eps) / np.clip(1 - raw_prob, eps, 1 - eps))
    return float(1.0 / (1.0 + np.exp(-logits / temperature)))

def read_label_state():
    try:
        with open("/tmp/label_state", "r") as f:
            lines = f.readlines()
            if lines:
                parts = lines[-1].strip().split()
                label = int(parts[0])
                if label == 1:
                    host = parts[1].split("_")[0] if len(parts) >= 2 else "unknown"
                    ip = parts[2].strip() if len(parts) >= 3 else "none"
                    return label, host, ip
                return 0, "none", "none"
    except Exception:
        pass
    return 0, "none", "none"

def build_dqn_obs(attack_prob, sample):
    src_raw = np.array(
        [float(sample.get(f"src{k}_{f}", 0) or 0) for k in range(1, TOP_K + 1) for f in SRC_NUM_FEATS],
        dtype=np.float32
    )
    glob_raw = np.array([float(sample.get(g, 0) or 0) for g in GLOBAL_FEATS], dtype=np.float32)
    extra = np.concatenate([src_raw, glob_raw]).reshape(1, -1)
    extra = np.nan_to_num(extra, nan=0.0, posinf=1e6, neginf=-1e6)
    extra_scaled = np.clip(dqn_scaler.transform(extra), -10, 10).flatten()

    severity = min(1.0, attack_prob * 1.2)
    confidence = abs(attack_prob - 0.5) * 2
    return np.concatenate([[attack_prob, severity, confidence], extra_scaled]).astype(np.float32)

def resolve_ground_truth(sample):
    file_label, file_host, file_ip = read_label_state()
    try:
        sample_label = int(sample.get("label"))
    except Exception:
        sample_label = None

    if GT_MODE == "file_only" or sample_label not in (0, 1):
        return file_label, file_host, file_ip, file_label, sample_label, "file"

    if sample_label == 0:
        return 0, "none", "none", file_label, sample_label, "sample"

    sample_host = str(sample.get("label_host", sample.get("host", file_host)) or file_host)
    sample_ip = str(sample.get("label_ip", file_ip) or file_ip)
    return 1, sample_host, sample_ip, file_label, sample_label, "sample"

def safe_get_docker_stats():
    fn = globals().get("get_docker_stats", None)
    if callable(fn):
        try:
            return fn()
        except Exception:
            return {}
    return {}

# ---------------------------- Optional cleanup --------------------------
if "clear_all_mitigations" in globals():
    try:
        clear_all_mitigations()
        print("[NO-MIT] Cleared existing mitigations")
    except Exception:
        pass

# ---------------------------- Main loop --------------------------------
feature_window = deque(maxlen=WINDOW_LEN)
step_log = []
prev_samples = 0
step = 0
t_start = time.time()
prev_gt_label = 0

attack_start_elapsed = None
first_policy_block_elapsed = None
policy_reactions = []

docker_cache = {}
docker_cache_ts = 0.0
DOCKER_CACHE_SEC = 2.0

process = psutil.Process(os.getpid())
_ = process.cpu_percent(interval=None)

print("\n====================================================")
print("LIVE DRL NO-MITIGATION MODE")
print(f"Duration={DRL_DURATION}s, Interval={INTERVAL}s")
print("Mitigation is DISABLED (WOULD_BLOCK only).")
print("====================================================\n")

try:
    while time.time() - t_start < DRL_DURATION:
        time.sleep(INTERVAL)

        with data_lock:
            snap = list(collected_samples)

        n_total = len(snap)
        backlog = n_total - prev_samples
        if backlog <= 0:
            continue

        new_samples = snap[prev_samples:n_total]
        prev_samples = n_total

        step_t0 = time.perf_counter()
        elapsed = round(time.time() - t_start, 1)

        # Batch preprocess
        t_pre = time.perf_counter()
        df_new = pd.DataFrame(new_samples)
        X_new = preprocess_live_sample(df_new)
        preprocess_ms = (time.perf_counter() - t_pre) * 1000.0

        for row in X_new:
            feature_window.append(row)

        if len(feature_window) < WINDOW_LEN:
            print(f"[Warmup] {len(feature_window)}/{WINDOW_LEN} (backlog={backlog})")
            continue

        sample = new_samples[-1]
        sample_id = int(sample.get("sample_id", step)) if str(sample.get("sample_id", "")).isdigit() else step

        # Ground truth
        gt_label, gt_host, gt_ip, gt_file_label, gt_sample_label, gt_source = resolve_ground_truth(sample)
        gt_mismatch = int((gt_sample_label in (0, 1)) and (gt_sample_label != gt_file_label))

        # Attack transitions
        if prev_gt_label == 0 and gt_label == 1:
            attack_start_elapsed = elapsed
            first_policy_block_elapsed = None
            print(f"[NO-MIT] ATTACK START step={step} t={elapsed}s host={gt_host} ip={gt_ip}")

        if prev_gt_label == 1 and gt_label == 0:
            if first_policy_block_elapsed is not None and attack_start_elapsed is not None:
                reaction = first_policy_block_elapsed - attack_start_elapsed
                policy_reactions.append(reaction)
                print(f"[NO-MIT] ATTACK END step={step} reaction={reaction:.1f}s")
            else:
                print(f"[NO-MIT] ATTACK END step={step} reaction=none")
            attack_start_elapsed = None
            first_policy_block_elapsed = None

        prev_gt_label = gt_label

        # LSTM inference
        window_arr = np.array(feature_window, dtype=np.float32).reshape(1, WINDOW_LEN, N_LSTM_FEATS)
        t_lstm = time.perf_counter()
        raw_prob = float(lstm_model.predict(window_arr, verbose=0).ravel()[0])
        cal_prob = calibrate_probability(raw_prob, TEMPERATURE)
        binary = int(cal_prob >= BEST_THRESHOLD)
        lstm_ms = (time.perf_counter() - t_lstm) * 1000.0

        # DQN decision
        t_dqn = time.perf_counter()
        obs = build_dqn_obs(cal_prob, sample)
        action, _ = dqn_agent.predict(obs, deterministic=True)
        action = int(action)
        dqn_ms = (time.perf_counter() - t_dqn) * 1000.0

        # NO mitigation branch
        would_mitigate = 0
        would_ip = ""
        action_str = "ALLOW"

        if gt_label == 1:
            if action > 0:
                src_ip = str(sample.get(f"src{action}_ip", "none")).strip()
                if src_ip not in KNOWN_LEGIT and src_ip != "none":
                    would_mitigate = 1
                    would_ip = src_ip
                    action_str = f"WOULD_BLOCK src{action}({src_ip})"
                    if first_policy_block_elapsed is None:
                        first_policy_block_elapsed = elapsed
                else:
                    action_str = f"WOULD_SKIP_LEGIT src{action}({src_ip})"
            else:
                action_str = "SEARCHING..."
        else:
            action_str = "FA_BLOCKED (ignored)" if action > 0 else "ALLOW"

        # Metrics
        now = time.time()
        if now - docker_cache_ts >= DOCKER_CACHE_SEC:
            docker_cache = safe_get_docker_stats()
            docker_cache_ts = now

        onos1_cpu = float((docker_cache.get("onos1") or {}).get("cpu", 0.0))
        onos2_cpu = float((docker_cache.get("onos2") or {}).get("cpu", 0.0))
        onos3_cpu = float((docker_cache.get("onos3") or {}).get("cpu", 0.0))
        ctrl_cpu = (onos1_cpu + onos2_cpu + onos3_cpu) / 3.0

        py_cpu = float(process.cpu_percent(interval=None))
        mem_mb = process.memory_info().rss / (1024 * 1024)

        total_packets = int(sample.get("total_packets", 0) or 0)
        total_packets_delta = float(sample.get("total_packets_delta", total_packets) or 0.0)
        forwarded_packets_delta = float(sample.get("forwarded_packets_delta", total_packets_delta) or 0.0)
        dropped_packets_delta = 0.0  # forced off

        step_total_ms = (time.perf_counter() - step_t0) * 1000.0
        correct = "OK" if binary == gt_label else "ERR"

        lag_tag = f" backlog={backlog}" if backlog > 1 else ""
        print(
            f"[{step:>4}] {elapsed:>6.1f}s P={cal_prob:.4f} {correct} gt={gt_label}({gt_host}) "
            f"| {action_str} | dTot={total_packets_delta:.0f} dFwd={forwarded_packets_delta:.0f} "
            f"dDrop={dropped_packets_delta:.0f} ctrlCPU={ctrl_cpu:.1f}% pyCPU={py_cpu:.1f}% "
            f"[{step_total_ms:.0f}ms]{lag_tag}"
        )

        # Log row
        step_log.append({
            "step": step,
            "sample_id": sample_id,
            "elapsed_s": elapsed,
            "backlog_samples": backlog,
            "gt_source": gt_source,
            "gt_file_label": gt_file_label,
            "gt_sample_label": gt_sample_label,
            "gt_mismatch": gt_mismatch,

            "attack_prob": cal_prob,
            "raw_prob": raw_prob,
            "binary": binary,
            "gt_label": gt_label,
            "gt_host": gt_host,
            "gt_ip": gt_ip,
            "correct": int(binary == gt_label),

            "dqn_action": action,
            "action_taken": action_str,
            "would_mitigate": would_mitigate,
            "would_mitigated_ip": would_ip,
            "mitigation_active": 0,

            "total_packets": total_packets,
            "total_packets_delta": total_packets_delta,
            "forwarded_packets_delta": forwarded_packets_delta,
            "dropped_packets_delta": dropped_packets_delta,

            "total_flows": int(sample.get("total_flows", 0) or 0),
            "total_bytes": int(sample.get("total_bytes", 0) or 0),
            "flow_arrival_rate": float(sample.get("flow_arrival_rate", 0) or 0),
            "src_entropy": float(sample.get("src_entropy", sample.get("source_entropy", 0)) or 0),
            "top_source_dominance": float(sample.get("top_source_dominance", 0) or 0),
            "active_sources": int(sample.get("active_sources", 0) or 0),

            "onos1_cpu": onos1_cpu,
            "onos2_cpu": onos2_cpu,
            "onos3_cpu": onos3_cpu,
            "ctrl_cpu_avg": ctrl_cpu,
            "python_cpu_pct": py_cpu,
            "python_mem_mb": round(mem_mb, 1),

            "preprocess_ms": round(preprocess_ms, 2),
            "lstm_inference_ms": round(lstm_ms, 2),
            "dqn_inference_ms": round(dqn_ms, 2),
            "api_latency_ms": 0.0,
            "drop_query_ms": 0.0,
            "step_total_ms": round(step_total_ms, 2),

            "attack_start_elapsed": attack_start_elapsed,
            "first_policy_block_elapsed": first_policy_block_elapsed,
        })

        step += 1

except KeyboardInterrupt:
    print(f"\n[NO-MIT] Stopped at step {step}")

# ---------------------------- Save + summary ---------------------------
elapsed_total = time.time() - t_start
print(f"\n[NO-MIT] Total: {step} steps in {elapsed_total:.0f}s")

if step_log:
    df_log = pd.DataFrame(step_log)
    out_csv = SAVE_DIR / "live_drl_log_nomit.csv"
    df_log.to_csv(out_csv, index=False)
    print(f"[NO-MIT] Saved: {out_csv} ({len(df_log)} rows)")

    f1 = f1_score(df_log["gt_label"], df_log["binary"], zero_division=0)
    pre = precision_score(df_log["gt_label"], df_log["binary"], zero_division=0)
    rec = recall_score(df_log["gt_label"], df_log["binary"], zero_division=0)

    would_blocks = int(df_log["would_mitigate"].sum())
    benign_would = int(((df_log["gt_label"] == 0) & (df_log["would_mitigate"] == 1)).sum())

    print("\n================ NO-MIT SUMMARY ================")
    print(f"Detection F1:        {f1:.4f}")
    print(f"Precision:           {pre:.4f}")
    print(f"Recall:              {rec:.4f}")
    print(f"Would mitigations:   {would_blocks}")
    print(f"Would on benign:     {benign_would}")
    if policy_reactions:
        print(f"Policy reaction mean:{np.mean(policy_reactions):.2f}s")
    print(f"Ctrl CPU mean:       {df_log['ctrl_cpu_avg'].mean():.2f}%")
    print(f"Step total mean:     {df_log['step_total_ms'].mean():.2f}ms")
    print("===============================================")


Gym has been unmaintained since 2022 and does not support NumPy 2.0 amongst other critical functionality.
Please upgrade to Gymnasium, the maintained drop-in replacement of Gym, or contact the authors of your software and request that they upgrade.
Users of this version of Gym should be able to simply replace 'import gym' with 'import gymnasium as gym' in the vast majority of cases.
See the migration guide at https://gymnasium.farama.org/introduction/migration_guide/ for additional information.
I0000 00:00:1772618561.662872    8101 gpu_device.cc:2020] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 2147 MB memory:  -> device: 0, name: NVIDIA GeForce GTX 1650, pci bus id: 0000:01:00.0, compute capability: 7.5


[NO-MIT] Loaded LSTM window=5, threshold=0.8472, T=0.9568
[NO-MIT] Loaded DQN actions=7

LIVE DRL NO-MITIGATION MODE
Duration=600s, Interval=2.0s
Mitigation is DISABLED (WOULD_BLOCK only).



2026-03-04 15:02:46.827928: I external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:473] Loaded cuDNN version 91002


[   0]    2.0s P=0.0007 OK gt=0(none) | ALLOW | dTot=596 dFwd=596 dDrop=0 ctrlCPU=5.1% pyCPU=34.5% [2869ms] backlog=71
[   1]    6.9s P=0.0006 OK gt=0(none) | ALLOW | dTot=768 dFwd=768 dDrop=0 ctrlCPU=5.5% pyCPU=9.1% [2065ms] backlog=2
[   2]   10.9s P=0.0007 OK gt=0(none) | ALLOW | dTot=736 dFwd=736 dDrop=0 ctrlCPU=2.8% pyCPU=6.6% [2060ms] backlog=2
[   3]   15.0s P=0.0008 OK gt=0(none) | ALLOW | dTot=889 dFwd=889 dDrop=0 ctrlCPU=4.9% pyCPU=7.4% [2074ms] backlog=2
[   4]   19.1s P=0.0009 OK gt=0(none) | ALLOW | dTot=1003 dFwd=1003 dDrop=0 ctrlCPU=3.1% pyCPU=7.9% [2062ms] backlog=2
[   5]   23.1s P=0.0014 OK gt=0(none) | ALLOW | dTot=814 dFwd=814 dDrop=0 ctrlCPU=3.7% pyCPU=8.1% [2065ms] backlog=2
[   6]   27.2s P=0.0009 OK gt=0(none) | ALLOW | dTot=894 dFwd=894 dDrop=0 ctrlCPU=6.4% pyCPU=7.4% [2056ms] backlog=2
[   7]   31.3s P=0.0008 OK gt=0(none) | ALLOW | dTot=995 dFwd=995 dDrop=0 ctrlCPU=4.2% pyCPU=7.6% [2062ms] backlog=2
[   8]   35.3s P=0.0006 OK gt=0(none) | ALLOW | dTot=810 dFw